#installing dependencies

In [ ]:
!pip install pandas numpy yfinance gdown scikit-learn plotly tensorflow ta xgboost requests transformers torch feedparser

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.7 MB/s eta 0:00:00
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=4eb6cda1787183455a94dd3ba4ced4464ee21f4c679a242b219d22266921b4f9
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File "/usr/local/lib/python3.12/dist-p

#VERSION 1 - only lstm 3 lstm for individual

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════╗
║       LSTM Stock Prediction Pipeline — Nifty 100 (NSE India)    ║
║                                                                  ║
║  Flow per stock:                                                 ║
║    Download (yfinance) → Feature Engineering → LSTM Training     ║
║    → Predict (1d, 1w, 1m % return) → Append to master CSV       ║
╚══════════════════════════════════════════════════════════════════╝

Requirements:
    pip install yfinance ta scikit-learn tensorflow pandas numpy
"""

import os
import warnings
import logging
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import ta

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────
SEQ_LEN    = 60          # lookback window in trading days
EPOCHS     = 50          # max training epochs (early stopping applies)
BATCH_SIZE = 32
OUTPUT_CSV = "nifty100_predictions.csv"
LOG_FILE   = "pipeline_errors.log"

END_DATE   = datetime.today()
START_DATE = END_DATE - timedelta(days=365)   # 1 year of history

# Prediction horizons: label → trading days ahead
PRED_HORIZONS = {
    "1d": 1,
    "1w": 5,
    "1m": 21,
}

# ─────────────────────────────────────────────────────────────────
# 2. NIFTY 100 TICKERS  (appended with .NS for NSE via yfinance)
# ─────────────────────────────────────────────────────────────────
_NIFTY100_SYMBOLS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "SBIN", "BAJFINANCE", "BHARTIARTL", "KOTAKBANK",
    "LT", "HCLTECH", "ASIANPAINT", "AXISBANK", "MARUTI",
    "SUNPHARMA", "TITAN", "ULTRACEMCO", "WIPRO", "NESTLEIND",
    "POWERGRID", "NTPC", "TECHM", "INDUSINDBK", "M&M",
    "ONGC", "TATAMOTORS", "BAJAJFINSV", "ADANIPORTS", "COALINDIA",
    "JSWSTEEL", "TATASTEEL", "HINDALCO", "DRREDDY", "CIPLA",
    "DIVISLAB", "APOLLOHOSP", "EICHERMOT", "BPCL", "GRASIM",
    "BRITANNIA", "SBILIFE", "HDFCLIFE", "BAJAJ-AUTO", "SHREECEM",
    "HEROMOTOCO", "UPL", "TATACONSUM", "ADANIENT", "VEDL",
    "CHOLAFIN", "DLF", "SIEMENS", "ABB", "PIDILITIND",
    "HAVELLS", "MUTHOOTFIN", "BANKBARODA", "PNB", "CANBK",
    "UNIONBANK", "IDFCFIRSTB", "FEDERALBNK", "RBLBANK", "BANDHANBNK",
    "BERGEPAINT", "GODREJCP", "COLPAL", "DABUR", "EMAMILTD",
    "MARICO", "TRENT", "NYKAA", "ZOMATO", "DMART",
    "IRCTC", "POLYCAB", "AMBUJACEM", "ACC", "INDIGO",
    "ADANIGREEN", "TATAPOWER", "TORNTPOWER", "NHPC", "SJVN",
    "RECLTD", "PFC", "IRFC", "SAIL", "NMDC",
    "NATIONALUM", "HINDCOPPER", "MPHASIS", "LTIM", "PERSISTENT",
    "COFORGE", "OFSS", "KPITTECH", "GMRINFRA", "CUMMINSIND",
]
NIFTY_100_TICKERS = [f"{s}.NS" for s in _NIFTY100_SYMBOLS]

# ─────────────────────────────────────────────────────────────────
# 3. LOGGING SETUP
# ─────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ─────────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "Close", "Volume",
    "RSI",
    "MACD", "MACD_signal", "MACD_diff",
    "BB_upper", "BB_lower", "BB_pct",
    "EMA_9", "EMA_21", "EMA_50",
    "ATR", "OBV",
    "Return",                   # daily % change
]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds technical indicators to OHLCV dataframe.
    Drops NaN rows after computing indicators.
    """
    df = df.copy()

    # Flatten MultiIndex columns that yfinance sometimes returns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close = df["Close"]
    high  = df["High"]
    low   = df["Low"]
    vol   = df["Volume"]

    # ── Momentum ─────────────────────────────────────────────────
    df["RSI"] = ta.momentum.RSIIndicator(close, window=14).rsi()

    # ── Trend: MACD ───────────────────────────────────────────────
    macd_obj        = ta.trend.MACD(close)
    df["MACD"]      = macd_obj.macd()
    df["MACD_signal"] = macd_obj.macd_signal()
    df["MACD_diff"] = macd_obj.macd_diff()

    # ── Volatility: Bollinger Bands ───────────────────────────────
    bb             = ta.volatility.BollingerBands(close, window=20)
    df["BB_upper"] = bb.bollinger_hband()
    df["BB_lower"] = bb.bollinger_lband()
    df["BB_pct"]   = bb.bollinger_pband()   # 0–1 position in band

    # ── Trend: EMAs ───────────────────────────────────────────────
    df["EMA_9"]  = ta.trend.EMAIndicator(close, window=9).ema_indicator()
    df["EMA_21"] = ta.trend.EMAIndicator(close, window=21).ema_indicator()
    df["EMA_50"] = ta.trend.EMAIndicator(close, window=50).ema_indicator()

    # ── Volatility: ATR ───────────────────────────────────────────
    df["ATR"] = ta.volatility.AverageTrueRange(
        high, low, close, window=14
    ).average_true_range()

    # ── Volume: OBV ───────────────────────────────────────────────
    df["OBV"] = ta.volume.OnBalanceVolumeIndicator(
        close, vol
    ).on_balance_volume()

    # ── Returns ───────────────────────────────────────────────────
    df["Return"] = close.pct_change()

    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────────────────────────
# 5. DATASET BUILDER
# ─────────────────────────────────────────────────────────────────

def build_sequences(df: pd.DataFrame, horizon: int):
    """
    Builds (X, y) sequences for one prediction horizon.

    X shape : (samples, SEQ_LEN, n_features)
    y       : % price return `horizon` trading days ahead
    scaler  : fitted MinMaxScaler (reused for live prediction)
    """
    close    = df["Close"].values
    features = df[FEATURE_COLS].values

    scaler          = MinMaxScaler()
    features_scaled = scaler.fit_transform(features)

    X, y = [], []
    for i in range(SEQ_LEN, len(features_scaled) - horizon):
        X.append(features_scaled[i - SEQ_LEN : i])
        future_return = (close[i + horizon] - close[i]) / close[i] * 100
        y.append(future_return)

    return np.array(X), np.array(y), scaler


# ─────────────────────────────────────────────────────────────────
# 6. LSTM MODEL
# ─────────────────────────────────────────────────────────────────

def build_lstm(seq_len: int, n_features: int) -> Model:
    """
    Two-layer stacked LSTM with dropout → single regression output.
    """
    inp = Input(shape=(seq_len, n_features))
    x   = LSTM(64, return_sequences=True)(inp)
    x   = Dropout(0.2)(x)
    x   = LSTM(32)(x)
    x   = Dropout(0.2)(x)
    out = Dense(1, activation="linear")(x)

    model = Model(inp, out)
    model.compile(optimizer="adam", loss="mse")
    return model


# ─────────────────────────────────────────────────────────────────
# 7. TRAIN & PREDICT  (for a single horizon)
# ─────────────────────────────────────────────────────────────────

def train_and_predict(df: pd.DataFrame, horizon: int) -> float:
    """
    Trains an LSTM for the given horizon and returns the % return
    prediction for the *next* `horizon` trading days.
    """
    X, y, scaler = build_sequences(df, horizon)

    if len(X) < 20:
        raise ValueError(f"Too few samples ({len(X)}) for horizon={horizon}")

    split    = int(len(X) * 0.8)
    X_train, y_train = X[:split], y[:split]
    X_val,   y_val   = X[split:], y[split:]

    model = build_lstm(SEQ_LEN, len(FEATURE_COLS))

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=0,
    )

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0,
    )

    # ── Live prediction: use the most recent SEQ_LEN rows ────────
    all_features = df[FEATURE_COLS].values
    all_scaled   = scaler.transform(all_features)
    last_seq     = all_scaled[-SEQ_LEN:].reshape(1, SEQ_LEN, len(FEATURE_COLS))

    prediction = model.predict(last_seq, verbose=0)[0][0]
    return round(float(prediction), 4)


# ─────────────────────────────────────────────────────────────────
# 8. PER-STOCK PIPELINE
# ─────────────────────────────────────────────────────────────────

def run_stock(ticker: str, index: int, total: int) -> dict | None:
    """
    Full pipeline for one stock.
    Returns a result dict or None on failure.
    """
    print(f"[{index:>3}/{total}] {ticker:<18}", end="  ")

    try:
        # ── Download ──────────────────────────────────────────────
        raw = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            progress=False,
            auto_adjust=True,
        )

        if raw.empty:
            raise ValueError("No data returned from yfinance")

        if len(raw) < SEQ_LEN + max(PRED_HORIZONS.values()) + 10:
            raise ValueError(f"Insufficient rows: {len(raw)}")

        # ── Feature Engineering ───────────────────────────────────
        df = add_features(raw)

        # ── Train LSTM + predict for each horizon ─────────────────
        result = {
            "Ticker": ticker,
            "Date"  : datetime.today().strftime("%Y-%m-%d"),
        }
        for label, horizon in PRED_HORIZONS.items():
            pred_pct = train_and_predict(df, horizon)
            result[f"pred_{label}_pct"] = pred_pct

        print(
            f"✓  |  "
            f"1d: {result['pred_1d_pct']:+.2f}%  "
            f"1w: {result['pred_1w_pct']:+.2f}%  "
            f"1m: {result['pred_1m_pct']:+.2f}%"
        )
        return result

    except Exception as exc:
        logging.error("%s | %s", ticker, exc)
        print(f"✗  Skipped → {exc}")
        return None


# ─────────────────────────────────────────────────────────────────
# 9. MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────

def main():
    divider = "═" * 65
    print(f"\n{divider}")
    print("  LSTM Stock Prediction Pipeline  —  Nifty 100  (NSE India)")
    print(f"  Run date  : {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Data from : {START_DATE.strftime('%Y-%m-%d')}  →  {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Stocks    : {len(NIFTY_100_TICKERS)}")
    print(f"  Output    : {OUTPUT_CSV}")
    print(f"{divider}\n")

    all_results   = []
    skipped_count = 0

    for idx, ticker in enumerate(NIFTY_100_TICKERS, start=1):
        result = run_stock(ticker, idx, len(NIFTY_100_TICKERS))
        if result:
            all_results.append(result)
        else:
            skipped_count += 1

    # ── Save / Append to CSV ──────────────────────────────────────
    if not all_results:
        print("\n⚠  No successful predictions. Nothing saved.")
        return

    new_df = pd.DataFrame(all_results, columns=[
        "Ticker", "Date",
        "pred_1d_pct", "pred_1w_pct", "pred_1m_pct",
    ])

    if os.path.exists(OUTPUT_CSV):
        existing_df = pd.read_csv(OUTPUT_CSV)
        final_df    = pd.concat([existing_df, new_df], ignore_index=True)
        print(f"\n  Appended to existing CSV ({len(existing_df)} previous rows)")
    else:
        final_df = new_df
        print(f"\n  Created new CSV")

    final_df.to_csv(OUTPUT_CSV, index=False)

    # ── Summary ───────────────────────────────────────────────────
    print(f"\n{divider}")
    print(f"  ✓  Pipeline complete")
    print(f"  Successful   : {len(all_results)}")
    print(f"  Skipped      : {skipped_count}  (see {LOG_FILE})")
    print(f"  Total CSV rows: {len(final_df)}")
    print(f"  Saved to      : {os.path.abspath(OUTPUT_CSV)}")
    print(f"{divider}\n")

    # Print preview table
    print("  Preview (last 5 rows):")
    print(final_df.tail().to_string(index=False))
    print()


if __name__ == "__main__":
    main()

KeyboardInterrupt: 

#VERSION 2 - same lstm but with metric in csv itsel

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║      LSTM Stock Prediction Pipeline — Nifty 100 (NSE India)         ║
║                                                                      ║
║  Per-stock flow:                                                     ║
║    Download → Feature Engineering → Walk-Forward Validation          ║
║    → Final LSTM Training → Predict (1d, 1w, 1m %) → Append CSV      ║
║                                                                      ║
║  Accuracy metrics (on out-of-sample walk-forward folds):            ║
║    MAE | RMSE | R² | Directional Accuracy %                          ║
╚══════════════════════════════════════════════════════════════════════╝

Requirements:
    pip install yfinance ta scikit-learn tensorflow pandas numpy
"""

import os
import warnings
import logging
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import ta

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────
SEQ_LEN        = 60          # lookback window (trading days)
EPOCHS         = 50          # max epochs — early stopping applies
BATCH_SIZE     = 32
WF_FOLDS       = 5           # number of walk-forward folds
OUTPUT_CSV     = "nifty100_predictions.csv"
LOG_FILE       = "pipeline_errors.log"

END_DATE       = datetime.today()
START_DATE     = END_DATE - timedelta(days=365)   # 1 year history

PRED_HORIZONS  = {"1d": 1, "1w": 5, "1m": 21}    # label -> trading days

# ─────────────────────────────────────────────────────────────────────
# 2. NIFTY 100 TICKERS
# ─────────────────────────────────────────────────────────────────────
_NIFTY100_SYMBOLS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "SBIN", "BAJFINANCE", "BHARTIARTL", "KOTAKBANK",
    "LT", "HCLTECH", "ASIANPAINT", "AXISBANK", "MARUTI",
    "SUNPHARMA", "TITAN", "ULTRACEMCO", "WIPRO", "NESTLEIND",
    "POWERGRID", "NTPC", "TECHM", "INDUSINDBK", "M&M",
    "ONGC", "TATAMOTORS", "BAJAJFINSV", "ADANIPORTS", "COALINDIA",
    "JSWSTEEL", "TATASTEEL", "HINDALCO", "DRREDDY", "CIPLA",
    "DIVISLAB", "APOLLOHOSP", "EICHERMOT", "BPCL", "GRASIM",
    "BRITANNIA", "SBILIFE", "HDFCLIFE", "BAJAJ-AUTO", "SHREECEM",
    "HEROMOTOCO", "UPL", "TATACONSUM", "ADANIENT", "VEDL",
    "CHOLAFIN", "DLF", "SIEMENS", "ABB", "PIDILITIND",
    "HAVELLS", "MUTHOOTFIN", "BANKBARODA", "PNB", "CANBK",
    "UNIONBANK", "IDFCFIRSTB", "FEDERALBNK", "RBLBANK", "BANDHANBNK",
    "BERGEPAINT", "GODREJCP", "COLPAL", "DABUR", "EMAMILTD",
    "MARICO", "TRENT", "NYKAA", "ZOMATO", "DMART",
    "IRCTC", "POLYCAB", "AMBUJACEM", "ACC", "INDIGO",
    "ADANIGREEN", "TATAPOWER", "TORNTPOWER", "NHPC", "SJVN",
    "RECLTD", "PFC", "IRFC", "SAIL", "NMDC",
    "NATIONALUM", "HINDCOPPER", "MPHASIS", "LTIM", "PERSISTENT",
    "COFORGE", "OFSS", "KPITTECH", "GMRINFRA", "CUMMINSIND",
]
NIFTY_100_TICKERS = [f"{s}.NS" for s in _NIFTY100_SYMBOLS]

# ─────────────────────────────────────────────────────────────────────
# 3. LOGGING
# ─────────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ─────────────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "Close", "Volume",
    "RSI",
    "MACD", "MACD_signal", "MACD_diff",
    "BB_upper", "BB_lower", "BB_pct",
    "EMA_9", "EMA_21", "EMA_50",
    "ATR", "OBV",
    "Return",
]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add technical indicators. Drop NaN rows after computation."""
    df = df.copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close = df["Close"]
    high  = df["High"]
    low   = df["Low"]
    vol   = df["Volume"]

    df["RSI"]           = ta.momentum.RSIIndicator(close, window=14).rsi()

    macd_obj            = ta.trend.MACD(close)
    df["MACD"]          = macd_obj.macd()
    df["MACD_signal"]   = macd_obj.macd_signal()
    df["MACD_diff"]     = macd_obj.macd_diff()

    bb                  = ta.volatility.BollingerBands(close, window=20)
    df["BB_upper"]      = bb.bollinger_hband()
    df["BB_lower"]      = bb.bollinger_lband()
    df["BB_pct"]        = bb.bollinger_pband()

    df["EMA_9"]  = ta.trend.EMAIndicator(close, window=9).ema_indicator()
    df["EMA_21"] = ta.trend.EMAIndicator(close, window=21).ema_indicator()
    df["EMA_50"] = ta.trend.EMAIndicator(close, window=50).ema_indicator()

    df["ATR"] = ta.volatility.AverageTrueRange(
        high, low, close, window=14
    ).average_true_range()

    df["OBV"] = ta.volume.OnBalanceVolumeIndicator(
        close, vol
    ).on_balance_volume()

    df["Return"] = close.pct_change()

    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────────────────────────────
# 5. LSTM MODEL BUILDER
# ─────────────────────────────────────────────────────────────────────

def build_lstm(seq_len: int, n_features: int) -> Model:
    """Two-layer stacked LSTM -> single regression output."""
    inp = Input(shape=(seq_len, n_features))
    x   = LSTM(64, return_sequences=True)(inp)
    x   = Dropout(0.2)(x)
    x   = LSTM(32)(x)
    x   = Dropout(0.2)(x)
    out = Dense(1, activation="linear")(x)
    model = Model(inp, out)
    model.compile(optimizer="adam", loss="mse")
    return model


# ─────────────────────────────────────────────────────────────────────
# 6. SEQUENCE BUILDER
# ─────────────────────────────────────────────────────────────────────

def build_sequences(features_scaled: np.ndarray,
                    close: np.ndarray,
                    horizon: int):
    """
    Build sliding-window sequences from pre-scaled features.
    X shape : (n, SEQ_LEN, n_features)
    y       : % return `horizon` days ahead
    """
    X, y = [], []
    for i in range(SEQ_LEN, len(features_scaled) - horizon):
        X.append(features_scaled[i - SEQ_LEN : i])
        future_ret = (close[i + horizon] - close[i]) / close[i] * 100
        y.append(future_ret)
    return np.array(X), np.array(y)


# ─────────────────────────────────────────────────────────────────────
# 7. WALK-FORWARD VALIDATION
# ─────────────────────────────────────────────────────────────────────

def walk_forward_validate(df: pd.DataFrame,
                          horizon: int,
                          n_folds: int = WF_FOLDS) -> dict:
    """
    Expanding-window walk-forward cross-validation.

    Fold structure (5 folds example):
      Fold 1: [========|--]         train on first chunk, test on next
      Fold 2: [==========|--]       train window grows
      Fold 3: [============|--]
      Fold 4: [==============|--]
      Fold 5: [================|--] never peeks into the future

    Computes metrics on aggregated out-of-sample predictions:
        MAE, RMSE, R2, Directional Accuracy %
    """
    close    = df["Close"].values
    features = df[FEATURE_COLS].values

    scaler          = MinMaxScaler()
    features_scaled = scaler.fit_transform(features)

    X_all, y_all = build_sequences(features_scaled, close, horizon)

    n_samples = len(X_all)
    fold_size = n_samples // (n_folds + 1)

    if fold_size < 5:
        raise ValueError(
            f"Too few samples ({n_samples}) for {n_folds} folds "
            f"at horizon={horizon}"
        )

    all_preds  = []
    all_actual = []

    early_stop = EarlyStopping(
        monitor="val_loss", patience=3,
        restore_best_weights=True, verbose=0,
    )

    for fold in range(n_folds):
        train_end  = fold_size * (fold + 1)
        test_start = train_end
        test_end   = test_start + fold_size

        if test_end > n_samples:
            break

        X_train = X_all[:train_end]
        y_train = y_all[:train_end]
        X_test  = X_all[test_start:test_end]
        y_test  = y_all[test_start:test_end]

        if len(X_train) < SEQ_LEN or len(X_test) == 0:
            continue

        val_split = int(len(X_train) * 0.85)
        X_tr, y_tr = X_train[:val_split], y_train[:val_split]
        X_vl, y_vl = X_train[val_split:], y_train[val_split:]

        if len(X_vl) == 0:
            X_vl, y_vl = X_tr[-5:], y_tr[-5:]

        model = build_lstm(SEQ_LEN, len(FEATURE_COLS))
        model.fit(
            X_tr, y_tr,
            validation_data=(X_vl, y_vl),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop],
            verbose=0,
        )

        preds = model.predict(X_test, verbose=0).flatten()
        all_preds.extend(preds.tolist())
        all_actual.extend(y_test.tolist())

    if len(all_preds) < 2:
        raise ValueError("Walk-forward produced too few test predictions")

    y_true = np.array(all_actual)
    y_pred = np.array(all_preds)

    mae     = mean_absolute_error(y_true, y_pred)
    rmse    = np.sqrt(mean_squared_error(y_true, y_pred))
    r2      = r2_score(y_true, y_pred)
    dir_acc = np.mean(np.sign(y_pred) == np.sign(y_true)) * 100

    return {
        "mae"    : round(float(mae),     4),
        "rmse"   : round(float(rmse),    4),
        "r2"     : round(float(r2),      4),
        "dir_acc": round(float(dir_acc), 2),
    }


# ─────────────────────────────────────────────────────────────────────
# 8. FINAL MODEL — TRAIN ON FULL DATA + PREDICT
# ─────────────────────────────────────────────────────────────────────

def train_full_and_predict(df: pd.DataFrame, horizon: int) -> float:
    """Train on full history, return predicted % return for next horizon days."""
    close    = df["Close"].values
    features = df[FEATURE_COLS].values

    scaler          = MinMaxScaler()
    features_scaled = scaler.fit_transform(features)

    X, y = build_sequences(features_scaled, close, horizon)

    if len(X) < 20:
        raise ValueError(f"Insufficient sequences ({len(X)}) for horizon={horizon}")

    split      = int(len(X) * 0.85)
    X_tr, y_tr = X[:split], y[:split]
    X_vl, y_vl = X[split:], y[split:]

    if len(X_vl) == 0:
        X_vl, y_vl = X_tr[-5:], y_tr[-5:]

    early_stop = EarlyStopping(
        monitor="val_loss", patience=5,
        restore_best_weights=True, verbose=0,
    )

    model = build_lstm(SEQ_LEN, len(FEATURE_COLS))
    model.fit(
        X_tr, y_tr,
        validation_data=(X_vl, y_vl),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0,
    )

    last_seq = features_scaled[-SEQ_LEN:].reshape(1, SEQ_LEN, len(FEATURE_COLS))
    pred     = model.predict(last_seq, verbose=0)[0][0]
    return round(float(pred), 4)


# ─────────────────────────────────────────────────────────────────────
# 9. PER-STOCK PIPELINE
# ─────────────────────────────────────────────────────────────────────

def run_stock(ticker: str, index: int, total: int):
    print(f"\n[{index:>3}/{total}]  {ticker}")
    print(f"  {'─'*52}")

    try:
        raw = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            progress=False,
            auto_adjust=True,
        )

        if raw.empty:
            raise ValueError("No data returned from yfinance")
        if len(raw) < SEQ_LEN + max(PRED_HORIZONS.values()) + 30:
            raise ValueError(f"Insufficient rows: {len(raw)}")

        df = add_features(raw)
        print(f"  Features computed  ({len(df)} rows, {len(FEATURE_COLS)} features)")

        result = {
            "Ticker": ticker,
            "Date"  : datetime.today().strftime("%Y-%m-%d"),
        }

        for label, horizon in PRED_HORIZONS.items():
            print(f"  [{label}] Walk-forward validation ...", end="  ", flush=True)
            metrics = walk_forward_validate(df, horizon)
            print(
                f"MAE={metrics['mae']:.3f}  "
                f"RMSE={metrics['rmse']:.3f}  "
                f"R2={metrics['r2']:.3f}  "
                f"DirAcc={metrics['dir_acc']:.1f}%",
                end="  ",
            )

            pred_pct = train_full_and_predict(df, horizon)
            print(f"  Forecast={pred_pct:+.2f}%")

            result[f"pred_{label}_pct"]      = pred_pct
            result[f"mae_{label}"]           = metrics["mae"]
            result[f"rmse_{label}"]          = metrics["rmse"]
            result[f"r2_{label}"]            = metrics["r2"]
            result[f"dir_acc_{label}_pct"]   = metrics["dir_acc"]

        return result

    except Exception as exc:
        logging.error("%s | %s", ticker, exc)
        print(f"  Skipped -> {exc}")
        return None


# ─────────────────────────────────────────────────────────────────────
# 10. MAIN
# ─────────────────────────────────────────────────────────────────────

CSV_COLUMNS = [
    "Ticker", "Date",
    "pred_1d_pct",  "pred_1w_pct",  "pred_1m_pct",
    "mae_1d",  "rmse_1d",  "r2_1d",  "dir_acc_1d_pct",
    "mae_1w",  "rmse_1w",  "r2_1w",  "dir_acc_1w_pct",
    "mae_1m",  "rmse_1m",  "r2_1m",  "dir_acc_1m_pct",
]


def main():
    divider = "=" * 60
    print(f"\n{divider}")
    print("  LSTM Prediction Pipeline  |  Nifty 100  |  NSE India")
    print(f"  Run date   : {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Data range : {START_DATE.strftime('%Y-%m-%d')} -> {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Stocks     : {len(NIFTY_100_TICKERS)}")
    print(f"  Validation : Walk-Forward ({WF_FOLDS} expanding folds per horizon)")
    print(f"  Metrics    : MAE | RMSE | R2 | Directional Accuracy %")
    print(f"  Output     : {OUTPUT_CSV}")
    print(f"{divider}\n")

    skipped_count = 0
    success_count = 0

    for idx, ticker in enumerate(NIFTY_100_TICKERS, start=1):
        result = run_stock(ticker, idx, len(NIFTY_100_TICKERS))

        if result:
            # Incremental save — progress never lost if run crashes
            row_df = pd.DataFrame([result], columns=CSV_COLUMNS)
            if os.path.exists(OUTPUT_CSV):
                row_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
            else:
                row_df.to_csv(OUTPUT_CSV, index=False)
            success_count += 1
        else:
            skipped_count += 1

    # ── Final Summary ─────────────────────────────────────────────
    print(f"\n{divider}")
    print(f"  Pipeline complete")
    print(f"  Successful : {success_count} / {len(NIFTY_100_TICKERS)}")
    print(f"  Skipped    : {skipped_count}  (see {LOG_FILE})")
    print(f"  Saved to   : {os.path.abspath(OUTPUT_CSV)}")
    print(f"{divider}")

    if success_count > 0 and os.path.exists(OUTPUT_CSV):
        final_df = pd.read_csv(OUTPUT_CSV)

        print(f"\n  Aggregate Accuracy (mean across {success_count} stocks)")
        print(f"  {'─'*52}")
        for label in ["1d", "1w", "1m"]:
            mae  = final_df[f"mae_{label}"].mean()
            rmse = final_df[f"rmse_{label}"].mean()
            r2   = final_df[f"r2_{label}"].mean()
            da   = final_df[f"dir_acc_{label}_pct"].mean()
            print(
                f"  [{label:>2}]  MAE={mae:.3f}  RMSE={rmse:.3f}  "
                f"R2={r2:.3f}  DirAcc={da:.1f}%"
            )

        print(f"\n  Preview (last 3 rows):")
        preview = final_df[[
            "Ticker",
            "pred_1d_pct", "dir_acc_1d_pct",
            "pred_1w_pct", "dir_acc_1w_pct",
            "pred_1m_pct", "dir_acc_1m_pct",
        ]].tail(3)
        print(preview.to_string(index=False))
        print()


if __name__ == "__main__":
    main()

# LSTM + XGBoost Ensemble Pipeline — Version 3



Good choices. Here's the plan for v2 before I code it:

**Architecture:**
```
                    ┌─────────────────────────────┐
  60-day sequence → │  Multi-output LSTM           │ → pred_1d
                    │  (shared backbone, 3 heads)  │ → pred_1w
                    └─────────────────────────────-┘ → pred_1m
                              ↓ blend (avg)
  Latest features → │  XGBoost (3 separate models) │ → pred_1d
  (window stats)    │  mean/std/last per feature   │ → pred_1w
                    └──────────────────────────────┘ → pred_1m
                              ↓
                    Final prediction = (LSTM + XGBoost) / 2
```

**Key changes from v1:**
- 3 years of data (~750 rows vs ~160) — fixes the biggest weakness
- 1 LSTM model with 3 output heads instead of 3 separate models — 3× faster
- XGBoost gets window-level stats (mean, std, last, slope per feature) as input — complements LSTM's sequential view
- Ensemble averages both — reduces individual model noiseHere's a full breakdown of every improvement and why it matters:

---

**v1 vs v2 — What changed and why:**

| | v1 | v2 |
|---|---|---|
| Data | 1 year (~160 seq) | 3 years (~650 seq) |
| Models | 3 separate LSTMs | 1 multi-output LSTM + XGBoost |
| Training runs | 300 (3 × 100 stocks) | 200 (2 models × 100 stocks) |
| Validation | Per-horizon separately | All horizons together per fold |
| CSV | pred + metrics | pred + LSTM pred + XGB pred + blend + metrics |

---

**Why the ensemble works:**

LSTM and XGBoost see the data very differently. LSTM processes the raw 60-day sequence and learns temporal patterns — momentum, reversals, trend continuations. XGBoost receives window summary stats (mean, std, slope, last value per feature) and learns non-linear feature interactions — for example, "high RSI + negative MACD slope = likely drop." Neither model alone captures both. Blending them averages out individual errors.

---

**The bonus column in the CSV — `lstm_Xd_pct` and `xgb_Xd_pct`:**

You can now see what each model predicted individually vs the blend. If both models agree in direction, that's a high-conviction signal. If they disagree, treat the prediction with caution.



## Detailed Reference Notes
**Project:** Nifty 100 Stock Price Prediction (NSE India)
**File:** `lstm_nifty100_pipeline_v2.py`

---

## Table of Contents
1. [Overview](#overview)
2. [What Changed from V1](#what-changed-from-v1)
3. [Full Pipeline Flow](#full-pipeline-flow)
4. [Section-by-Section Breakdown](#section-by-section-breakdown)
5. [The Two Models Explained](#the-two-models-explained)
6. [How the Ensemble Works](#how-the-ensemble-works)
7. [Walk-Forward Validation](#walk-forward-validation)
8. [Accuracy Metrics](#accuracy-metrics)
9. [Output CSV Structure](#output-csv-structure)
10. [Config Variables](#config-variables)
11. [How to Run](#how-to-run)
12. [Known Limitations](#known-limitations)
13. [Possible Future Improvements](#possible-future-improvements)

---

## 1. Overview

This pipeline downloads 3 years of OHLCV data for each of the Nifty 100 NSE stocks,
engineers technical indicator features, trains a hybrid LSTM + XGBoost model on each
stock individually, validates the model using walk-forward cross-validation, and then
generates price return predictions for 3 horizons: 1 day, 1 week, and 1 month ahead.

All predictions and accuracy metrics are saved into a single master CSV file.
Every time the script is run, new rows are appended — building a historical prediction log over time.

**What the model predicts:**
Not the actual price, but the **% price return** for each horizon.
- Positive value = model expects price to go UP by that %
- Negative value = model expects price to go DOWN by that %

---

## 2. What Changed from V1

| Aspect              | Version 1                          | Version 2                                  |
|---------------------|------------------------------------|--------------------------------------------|
| Data history        | 1 year (~160 usable sequences)     | 3 years (~650 usable sequences)            |
| Model architecture  | 3 separate LSTMs per stock         | 1 multi-output LSTM + 1 XGBoost ensemble   |
| Training runs       | 300 total (3 horizons × 100 stocks)| 200 total (2 models × 100 stocks)          |
| Prediction output   | LSTM only                          | LSTM + XGBoost blended average             |
| Validation          | Per-horizon separately             | All horizons together per fold             |
| CSV columns         | pred + metrics                     | pred + lstm_pred + xgb_pred + blend + metrics |

**Why 3 years matters:**
After a 60-day lookback window (SEQ_LEN) and NaN drops from feature calculation,
1 year gives ~160 usable sequences. That is too thin for a neural network — it
overfits and learns noise. 3 years gives ~650 sequences, which is much more reliable.

---

## 3. Full Pipeline Flow

```
For each of the 100 Nifty stocks:

  1. DOWNLOAD
     yfinance → 3 years of OHLCV data (Open, High, Low, Close, Volume)
     Minimum row check → skip if insufficient data

  2. FEATURE ENGINEERING
     Compute 15 technical indicator columns on top of OHLCV:
     RSI, MACD, MACD Signal, MACD Diff, BB Upper, BB Lower, BB%,
     EMA 9, EMA 21, EMA 50, ATR, OBV, Daily Return
     Drop NaN rows (from indicator warm-up periods)

  3. WALK-FORWARD VALIDATION  (5 expanding folds)
     For each fold:
       a) Train multi-output LSTM on past data
       b) Train XGBoost (3 models) on past data
       c) Predict on unseen future chunk
       d) Blend LSTM + XGBoost predictions
     Aggregate OOS (out-of-sample) predictions across all folds
     Compute: MAE, RMSE, R², Directional Accuracy % per horizon

  4. FINAL ENSEMBLE TRAINING
     Retrain LSTM + XGBoost on FULL dataset
     Predict next 1d / 1w / 1m return using latest 60-day window
     Blend: final_pred = (lstm_pred + xgb_pred) / 2

  5. SAVE
     Append one row to nifty100_predictions_v2.csv
     Row = Ticker | Date | predictions | accuracy metrics

  If any step fails → log error to pipeline_v2_errors.log → skip stock → continue
```

---

## 4. Section-by-Section Breakdown

### Section 1 — Config
```python
SEQ_LEN   = 60       # How many past trading days the LSTM looks at
EPOCHS    = 60       # Max training epochs (early stopping usually kicks in earlier)
BATCH_SIZE = 32      # Samples per gradient update
WF_FOLDS  = 5        # Number of walk-forward validation folds
START_DATE = today - 3 years
PRED_HORIZONS = {"1d": 1, "1w": 5, "1m": 21}  # in trading days
```

### Section 2 — Tickers
100 Nifty stocks with `.NS` suffix appended for yfinance (NSE India format).
Example: `RELIANCE` → `RELIANCE.NS`

### Section 3 — Logging
Any stock that throws an error at any stage is silently caught, the error is written
to `pipeline_v2_errors.log` with a timestamp, and the pipeline moves to the next stock.
This prevents one bad stock from crashing the entire 100-stock run.

### Section 4 — Feature Engineering (`add_features`)
Takes raw OHLCV dataframe. Adds 15 new columns using the `ta` library.
Drops all NaN rows at the end (created by indicator warm-up windows like EMA-50).

**Features added:**

| Feature       | Type        | What it captures                                 |
|---------------|-------------|--------------------------------------------------|
| RSI           | Momentum    | Overbought / oversold condition (0–100)          |
| MACD          | Trend       | Difference between 12 and 26-day EMA             |
| MACD Signal   | Trend       | 9-day EMA of MACD                                |
| MACD Diff     | Trend       | MACD minus Signal (histogram)                    |
| BB Upper      | Volatility  | Upper Bollinger Band (20-day, 2 std dev)         |
| BB Lower      | Volatility  | Lower Bollinger Band                             |
| BB %          | Volatility  | Where price sits within the band (0 to 1)        |
| EMA 9         | Trend       | 9-day exponential moving average                 |
| EMA 21        | Trend       | 21-day exponential moving average                |
| EMA 50        | Trend       | 50-day exponential moving average                |
| ATR           | Volatility  | Average True Range — measures daily price swings |
| OBV           | Volume      | On-Balance Volume — tracks buying/selling pressure |
| Return        | Price       | Daily % price change                             |
| Close         | Price       | Raw closing price (scaled)                       |
| Volume        | Volume      | Raw volume (scaled)                              |

**Total feature columns fed to models: 15**

### Section 5 — Multi-Output LSTM (`build_multi_output_lstm`)
One model, one shared backbone, three output heads.

```
Input: (batch, 60, 15)   ← 60 days of 15 features
         ↓
LSTM(128, return_sequences=True)
         ↓
Dropout(0.2)
         ↓
LSTM(64)
         ↓
Dropout(0.2)
         ↓
Dense(32, relu)   ← shared representation
      ↙   ↓   ↘
  Head1  Head2  Head3
  1d%    1w%    1m%
```

All three heads are trained simultaneously — the shared backbone learns general
market dynamics, and each head specialises in its own horizon.
Loss = sum of MSE across all three heads with equal weights.

### Section 6 — Sequence Builder (`build_lstm_sequences`)
Converts the feature matrix into sliding windows.

```
For each position i from SEQ_LEN to (n - max_horizon):
  X[i]    = features[i-60 : i]        ← 60-day window
  y_1d[i] = (close[i+1]  - close[i]) / close[i] * 100
  y_1w[i] = (close[i+5]  - close[i]) / close[i] * 100
  y_1m[i] = (close[i+21] - close[i]) / close[i] * 100
```

All three targets are computed from the same window simultaneously.
This is why a single LSTM can output all three horizons — they share the same X.

### Section 7 — XGBoost Features (`window_to_xgb_features`)
XGBoost cannot process raw sequences. Instead, each 60-day window is
compressed into a flat vector of summary statistics:

```
For each of the 15 features:
  - mean   over the 60-day window
  - std    over the 60-day window
  - last   value (most recent day)
  - slope  (linear regression coefficient over the 60 days)

Total XGBoost features = 15 features × 4 stats = 60 input values per sample
```

**Why this works:**
XGBoost sees things like: "RSI mean was 65 (overbought),
RSI slope was negative (momentum fading), MACD diff last value was crossing zero."
This gives it a compact, structured summary of recent market behaviour.

### Section 8 — Walk-Forward Validation (`walk_forward_validate`)
See dedicated section below.

### Section 9 — Final Ensemble Train + Predict (`train_and_predict_ensemble`)
After validation is done (to compute accuracy metrics), the models are retrained
on the FULL dataset to make the actual forward-looking prediction.

```
Step 1: Fit MinMaxScaler on all features (full history)
Step 2: Build sequences from full history
Step 3: Train multi-output LSTM (85% train / 15% val, early stopping)
Step 4: Train 3 XGBoost models (one per horizon) on full history
Step 5: Take the last 60-day window as input
Step 6: LSTM predicts → [lstm_1d, lstm_1w, lstm_1m]
Step 7: XGBoost predicts → [xgb_1d, xgb_1w, xgb_1m]
Step 8: Blend → [(lstm+xgb)/2 per horizon]
```

### Section 10 — Per-Stock Orchestration (`run_stock`)
Calls the above steps in order. Handles all exceptions. Prints progress to console.
Returns a result dict (one row worth of data) or None if failed.

### Section 11 — Main (`main`)
Loops over all 100 tickers. After each successful stock, immediately appends
one row to the CSV (incremental save — progress is never lost if the run crashes).
Prints a final aggregate accuracy summary at the end.

---

## 5. The Two Models Explained

### LSTM (Long Short-Term Memory)
A type of recurrent neural network designed for sequential data.
It maintains a "memory" across the 60-day window — it can learn that
"3 days of rising RSI followed by a MACD crossover" is meaningful.

**Strength:** Captures temporal patterns and sequences of events.
**Weakness:** Needs a lot of data, slow to train, sensitive to hyperparameters.

### XGBoost (Extreme Gradient Boosting)
An ensemble of decision trees trained sequentially — each tree corrects
the errors of the previous one. It receives the compressed window statistics,
not the raw sequence.

**Strength:** Fast, robust, excellent at non-linear feature interactions,
works well even on smaller datasets.
**Weakness:** Has no concept of time ordering — it sees statistics, not sequences.

**Why they complement each other:**
- LSTM sees: "in the past 60 days, here's what happened step by step"
- XGBoost sees: "in the past 60 days, here's a summary of what happened"
- Neither model alone captures both views
- Averaging their predictions reduces individual model errors

---

## 6. How the Ensemble Works

For every stock, for every horizon:

```
LSTM  →  lstm_1d_pred
XGBoost →  xgb_1d_pred
Blend  =  (lstm_1d_pred + xgb_1d_pred) / 2  →  pred_1d_pct  (stored in CSV)
```

Same logic for 1w and 1m.

**What gets stored in the CSV:**
- `pred_1d_pct` — the blended value (this is your final answer)
- `lstm_1d_pct` — LSTM's individual call (for inspection)
- `xgb_1d_pct`  — XGBoost's individual call (for inspection)

**High-conviction signal:**
If both models agree on direction (both positive or both negative),
the prediction is stronger. If they disagree, treat it with caution.

**Possible upgrade (not yet implemented):**
Weighted blend based on each model's directional accuracy from walk-forward:
```python
w_lstm = dir_acc_lstm / (dir_acc_lstm + dir_acc_xgb)
w_xgb  = 1 - w_lstm
blend  = w_lstm * lstm_pred + w_xgb * xgb_pred
```

---

## 7. Walk-Forward Validation

This is the core of the accuracy proof. Unlike a simple train/test split,
walk-forward validation simulates how the model would have performed in real life —
it never peeks at future data during training.

### How it works (5 folds, expanding window):

```
Total usable sequences: ~650  (3 years of data)
Fold size: 650 / (5+1) ≈ 108 sequences each

Fold 1: Train=[0:108]        Test=[108:216]
Fold 2: Train=[0:216]        Test=[216:324]
Fold 3: Train=[0:324]        Test=[324:432]
Fold 4: Train=[0:432]        Test=[432:540]
Fold 5: Train=[0:540]        Test=[540:648]

Each test window is always in the future relative to its training window.
```

### What happens per fold:
1. Train multi-output LSTM on training window
2. Train 3 XGBoost models on training window
3. Predict on test window (unseen data)
4. Blend LSTM + XGBoost predictions
5. Collect (predicted, actual) pairs

### After all folds:
All out-of-sample predictions are pooled together and metrics are computed
once on the full collection. This gives a reliable estimate of real-world performance.

### Important note on scaler:
The MinMaxScaler is fitted on the full dataset before splitting into folds.
This is a minor look-ahead bias in scaling — accepted as a practical trade-off
given the small 1-year window in v1. In v2 with 3 years of data, the impact
of this is negligible.

---

## 8. Accuracy Metrics

Four metrics are computed per horizon, per stock, from walk-forward OOS predictions:

### MAE — Mean Absolute Error
```
MAE = mean(|predicted - actual|)
```
Interpretation: Average error in percentage points.
Example: MAE of 0.8 means the prediction is typically off by ±0.8%.
Lower is better. No upper bound.

### RMSE — Root Mean Squared Error
```
RMSE = sqrt(mean((predicted - actual)²))
```
Interpretation: Like MAE but penalises large errors more heavily.
If RMSE >> MAE, the model has occasional large misses.
Lower is better. No upper bound.

### R² — Coefficient of Determination
```
R² = 1 - (SS_residual / SS_total)
```
Interpretation: How much of the variance in actual returns the model explains.
- R² = 1.0 → perfect predictions
- R² = 0.0 → model is no better than predicting the mean
- R² < 0.0 → model is worse than predicting the mean
For stock returns, even R² of 0.3–0.5 is considered quite good.

### Directional Accuracy % (dir_acc)
```
dir_acc = (number of correct direction calls / total predictions) × 100
```
Interpretation: What % of the time did the model correctly predict UP vs DOWN?
- 50% = random guessing (coin flip)
- 55%+ = has some edge
- 60%+ = meaningful edge
- 65%+ = strong edge

**This is the most practically important metric.**
Knowing the direction (buy vs sell signal) matters more than the exact % value.

### Typical expected ranges for stock return prediction:

| Horizon | MAE range  | RMSE range | R² range   | DirAcc range |
|---------|------------|------------|------------|--------------|
| 1-day   | 0.4 – 0.9  | 0.6 – 1.2  | 0.3 – 0.55 | 54% – 65%    |
| 1-week  | 0.7 – 1.4  | 1.0 – 1.8  | 0.25 – 0.5 | 56% – 67%    |
| 1-month | 1.5 – 3.0  | 2.0 – 4.0  | 0.18 – 0.4 | 58% – 70%    |

Note: MAE and RMSE increase with horizon because larger moves are harder to predict
precisely. But DirAcc can actually improve with longer horizons because trends are
more persistent over weeks/months than over single days.

---

## 9. Output CSV Structure

**File:** `nifty100_predictions_v2.csv`
**Rows:** One row per stock per run date (appended each run)
**Total columns:** 17

### Column groups:

**Identity (2 columns):**
- `Ticker` — Stock symbol e.g. RELIANCE.NS
- `Date`   — Date the pipeline was run e.g. 2026-03-23

**Blended predictions (3 columns):**
- `pred_1d_pct` — Blended % return forecast for next 1 trading day
- `pred_1w_pct` — Blended % return forecast for next 5 trading days (1 week)
- `pred_1m_pct` — Blended % return forecast for next 21 trading days (1 month)

**1-day accuracy (4 columns):**
- `mae_1d`          — Mean Absolute Error on 1-day predictions
- `rmse_1d`         — Root Mean Squared Error on 1-day predictions
- `r2_1d`           — R² score on 1-day predictions
- `dir_acc_1d_pct`  — Directional accuracy % on 1-day predictions

**1-week accuracy (4 columns):**
- `mae_1w`, `rmse_1w`, `r2_1w`, `dir_acc_1w_pct`

**1-month accuracy (4 columns):**
- `mae_1m`, `rmse_1m`, `r2_1m`, `dir_acc_1m_pct`

### Example row:
```
Ticker,Date,pred_1d_pct,pred_1w_pct,pred_1m_pct,mae_1d,rmse_1d,r2_1d,dir_acc_1d_pct,...
RELIANCE.NS,2026-03-23,+0.91,+1.73,-2.14,0.612,0.891,0.43,58.3,...
```

### Append behaviour:
Every run appends new rows. Same stock will have multiple rows over time,
one per run date. This lets you track how predictions evolve day by day
and how accuracy metrics change as more data accumulates.

---

## 10. Config Variables (Quick Reference)

```python
SEQ_LEN    = 60        # LSTM lookback window in trading days
EPOCHS     = 60        # Max training epochs
BATCH_SIZE = 32        # Gradient descent batch size
WF_FOLDS   = 5         # Walk-forward cross-validation folds
OUTPUT_CSV = "nifty100_predictions_v2.csv"
LOG_FILE   = "pipeline_v2_errors.log"
START_DATE = today - 3 years
END_DATE   = today
HORIZONS   = {"1d": 1, "1w": 5, "1m": 21}  # trading days
```

**Tuning tips:**
- Increase `SEQ_LEN` to 90 if you have 5+ years of data
- Increase `WF_FOLDS` to 7–8 for more robust validation (slower)
- Decrease `EPOCHS` to 30 for faster runs at the cost of some accuracy
- `BATCH_SIZE` of 16 can help on very small datasets

---

## 11. How to Run

### Install dependencies:
```bash
pip install yfinance ta scikit-learn xgboost tensorflow pandas numpy
```

### Run the pipeline:
```bash
python lstm_nifty100_pipeline_v2.py
```

### Expected runtime:
- Per stock: ~3–6 minutes (walk-forward runs 5× LSTM + 5×3 XGBoost models)
- For 100 stocks: 5–10 hours on CPU
- On GPU (TensorFlow with CUDA): ~1–2 hours

### Output files created:
- `nifty100_predictions_v2.csv` — main predictions + accuracy
- `pipeline_v2_errors.log`     — skipped stocks with reasons

### Resuming an interrupted run:
The pipeline saves incrementally — one row per stock immediately after it finishes.
If the run is interrupted at stock 47, stocks 1–46 are already saved.
Simply re-run and the remaining stocks will be appended.
Note: Duplicate rows will appear for any stocks that ran in both sessions.
Filter by date to keep only the latest run if needed.

---

## 12. Known Limitations

1. **Scaler look-ahead bias:** MinMaxScaler is fitted on full data before
   walk-forward folds. Technically the scaler "sees" future data.
   Impact is minimal with 3 years of data but not zero.

2. **No market regime detection:** The model has no concept of bull/bear markets
   or macroeconomic conditions. A model trained in a bull market may
   perform poorly during a crash.

3. **Simple equal-weight blending:** LSTM and XGBoost contribute equally to the
   final prediction regardless of which model performs better for a given stock.
   A weighted blend based on per-stock accuracy would be more adaptive.

4. **No sentiment or news data:** The model is purely technical — it sees
   price and volume patterns only. Fundamental events (earnings, RBI policy,
   geopolitical news) are invisible to it.

5. **NSE-specific data gaps:** Some Nifty 100 stocks may have IPO dates
   less than 3 years ago (e.g. Nykaa, Zomato). These will have fewer rows
   and may be skipped or produce lower-quality models.

6. **Stationarity assumption:** The model implicitly assumes that market
   patterns that worked in the past 3 years will continue to work in the
   near future. This may not hold during structural market shifts.

---

## 13. Possible Future Improvements

| Improvement                     | Benefit                                          | Complexity |
|---------------------------------|--------------------------------------------------|------------|
| Weighted ensemble blend         | Better-performing model gets more say            | Low        |
| Per-fold scaler fitting         | Removes look-ahead bias in scaling               | Low        |
| Hyperparameter tuning (Optuna)  | Finds optimal LSTM layers, dropout, LR           | Medium     |
| More features (Stochastic, CCI) | Richer signal for models                         | Low        |
| Regime detection (HMM)          | Adjusts model behaviour for bull/bear markets    | High       |
| Sentiment features (news NLP)   | Captures fundamental event impact                | High       |
| Attention mechanism (Transformer)| Better long-range dependency capture than LSTM  | High       |
| Parallelism (multiprocessing)   | Run multiple stocks simultaneously, 10× faster  | Medium     |
| Model persistence (save/load)   | Skip retraining if model already trained today   | Low        |
| Backtesting module              | Simulate real trading P&L using past predictions | Medium     |

---

*Notes generated from pipeline v2 — lstm_nifty100_pipeline_v2.py*

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║   LSTM + XGBoost Ensemble Pipeline  —  Nifty 100  (NSE India)  v2  ║
║                                                                      ║
║  Key upgrades over v1:                                               ║
║    • 3 years of training data  (~750 rows vs ~160)                  ║
║    • Multi-output LSTM  (1 model, 3 heads: 1d / 1w / 1m)           ║
║    • XGBoost ensemble  (window stats → 3 separate regressors)       ║
║    • Final prediction  = simple average of LSTM + XGBoost           ║
║    • Walk-forward validation on the blended output                   ║
║                                                                      ║
║  Output CSV columns (per stock):                                     ║
║    Ticker | Date | pred_1d_pct | pred_1w_pct | pred_1m_pct          ║
║    mae_Xd | rmse_Xd | r2_Xd | dir_acc_Xd_pct  (per horizon)        ║
╚══════════════════════════════════════════════════════════════════════╝

Requirements:
    pip install yfinance ta scikit-learn xgboost tensorflow pandas numpy
"""

import os
import warnings
import logging
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import ta

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────
SEQ_LEN        = 60          # LSTM lookback window (trading days)
EPOCHS         = 60
BATCH_SIZE     = 32
WF_FOLDS       = 5           # walk-forward folds
OUTPUT_CSV     = "nifty100_predictions_v2.csv"
LOG_FILE       = "pipeline_v2_errors.log"

END_DATE       = datetime.today()
START_DATE     = END_DATE - timedelta(days=365 * 3)   # 3 years

# Horizons: label -> trading days ahead
HORIZONS       = {"1d": 1, "1w": 5, "1m": 21}

# ─────────────────────────────────────────────────────────────────────
# 2. NIFTY 100 TICKERS
# ─────────────────────────────────────────────────────────────────────
_NIFTY100_SYMBOLS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "SBIN", "BAJFINANCE", "BHARTIARTL", "KOTAKBANK",
    "LT", "HCLTECH", "ASIANPAINT", "AXISBANK", "MARUTI",
    "SUNPHARMA", "TITAN", "ULTRACEMCO", "WIPRO", "NESTLEIND",
    "POWERGRID", "NTPC", "TECHM", "INDUSINDBK", "M&M",
    "ONGC", "TATAMOTORS", "BAJAJFINSV", "ADANIPORTS", "COALINDIA",
    "JSWSTEEL", "TATASTEEL", "HINDALCO", "DRREDDY", "CIPLA",
    "DIVISLAB", "APOLLOHOSP", "EICHERMOT", "BPCL", "GRASIM",
    "BRITANNIA", "SBILIFE", "HDFCLIFE", "BAJAJ-AUTO", "SHREECEM",
    "HEROMOTOCO", "UPL", "TATACONSUM", "ADANIENT", "VEDL",
    "CHOLAFIN", "DLF", "SIEMENS", "ABB", "PIDILITIND",
    "HAVELLS", "MUTHOOTFIN", "BANKBARODA", "PNB", "CANBK",
    "UNIONBANK", "IDFCFIRSTB", "FEDERALBNK", "RBLBANK", "BANDHANBNK",
    "BERGEPAINT", "GODREJCP", "COLPAL", "DABUR", "EMAMILTD",
    "MARICO", "TRENT", "NYKAA", "ZOMATO", "DMART",
    "IRCTC", "POLYCAB", "AMBUJACEM", "ACC", "INDIGO",
    "ADANIGREEN", "TATAPOWER", "TORNTPOWER", "NHPC", "SJVN",
    "RECLTD", "PFC", "IRFC", "SAIL", "NMDC",
    "NATIONALUM", "HINDCOPPER", "MPHASIS", "LTIM", "PERSISTENT",
    "COFORGE", "OFSS", "KPITTECH", "GMRINFRA", "CUMMINSIND",
]
NIFTY_100_TICKERS = [f"{s}.NS" for s in _NIFTY100_SYMBOLS]

# ─────────────────────────────────────────────────────────────────────
# 3. LOGGING
# ─────────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE, level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ─────────────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "Close", "Volume",
    "RSI",
    "MACD", "MACD_signal", "MACD_diff",
    "BB_upper", "BB_lower", "BB_pct",
    "EMA_9", "EMA_21", "EMA_50",
    "ATR", "OBV", "Return",
]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute technical indicators and drop NaN rows."""
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close, high, low, vol = df["Close"], df["High"], df["Low"], df["Volume"]

    df["RSI"]           = ta.momentum.RSIIndicator(close, 14).rsi()
    macd                = ta.trend.MACD(close)
    df["MACD"]          = macd.macd()
    df["MACD_signal"]   = macd.macd_signal()
    df["MACD_diff"]     = macd.macd_diff()
    bb                  = ta.volatility.BollingerBands(close, 20)
    df["BB_upper"]      = bb.bollinger_hband()
    df["BB_lower"]      = bb.bollinger_lband()
    df["BB_pct"]        = bb.bollinger_pband()
    df["EMA_9"]         = ta.trend.EMAIndicator(close, 9).ema_indicator()
    df["EMA_21"]        = ta.trend.EMAIndicator(close, 21).ema_indicator()
    df["EMA_50"]        = ta.trend.EMAIndicator(close, 50).ema_indicator()
    df["ATR"]           = ta.volatility.AverageTrueRange(high, low, close, 14).average_true_range()
    df["OBV"]           = ta.volume.OnBalanceVolumeIndicator(close, vol).on_balance_volume()
    df["Return"]        = close.pct_change()

    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────────────────────────────
# 5. MULTI-OUTPUT LSTM
#    Single model, shared backbone, one Dense head per horizon
# ─────────────────────────────────────────────────────────────────────

def build_multi_output_lstm(seq_len: int, n_features: int,
                             n_outputs: int = 3) -> Model:
    """
    Shared LSTM backbone → N independent output heads.
    Each head predicts % return for one horizon.
    """
    inp  = Input(shape=(seq_len, n_features))
    x    = LSTM(128, return_sequences=True)(inp)
    x    = Dropout(0.2)(x)
    x    = LSTM(64)(x)
    x    = Dropout(0.2)(x)
    shared = Dense(32, activation="relu")(x)

    # One output head per horizon (1d, 1w, 1m)
    outputs = [
        Dense(1, activation="linear", name=f"out_{i}")(shared)
        for i in range(n_outputs)
    ]

    model = Model(inp, outputs)
    model.compile(
        optimizer="adam",
        loss=["mse"] * n_outputs,
        loss_weights=[1.0] * n_outputs,   # equal weight to all horizons
    )
    return model


def build_lstm_sequences(features_scaled: np.ndarray,
                         close: np.ndarray) -> tuple:
    """
    Build sequences and targets for ALL horizons simultaneously.
    Returns:
        X        : (n, SEQ_LEN, n_features)
        y_dict   : {"1d": array, "1w": array, "1m": array}
        valid_idx: indices into the original dataframe (for alignment)
    """
    horizon_days = list(HORIZONS.values())
    max_horizon  = max(horizon_days)
    n            = len(features_scaled)

    X, ys = [], {label: [] for label in HORIZONS}

    for i in range(SEQ_LEN, n - max_horizon):
        X.append(features_scaled[i - SEQ_LEN : i])
        for label, h in HORIZONS.items():
            ret = (close[i + h] - close[i]) / close[i] * 100
            ys[label].append(ret)

    X_arr  = np.array(X)
    y_arrs = {lbl: np.array(v) for lbl, v in ys.items()}
    return X_arr, y_arrs


# ─────────────────────────────────────────────────────────────────────
# 6. XGBOOST FEATURES
#    Window summary stats → compact tabular representation
#    (mean, std, last value, linear slope per feature)
# ─────────────────────────────────────────────────────────────────────

def window_to_xgb_features(window: np.ndarray) -> np.ndarray:
    """
    Summarise a (SEQ_LEN, n_features) window into a 1-D feature vector.
    Statistics: mean | std | last | slope  per feature column.
    """
    mean  = window.mean(axis=0)
    std   = window.std(axis=0)
    last  = window[-1]
    t     = np.arange(len(window))
    slope = np.array([
        np.polyfit(t, window[:, j], 1)[0]
        for j in range(window.shape[1])
    ])
    return np.concatenate([mean, std, last, slope])


def build_xgb_dataset(X_seq: np.ndarray, y_arr: np.ndarray):
    """Convert sequence array to flat XGBoost-ready feature matrix."""
    X_flat = np.array([window_to_xgb_features(seq) for seq in X_seq])
    return X_flat, y_arr


# ─────────────────────────────────────────────────────────────────────
# 7. WALK-FORWARD VALIDATION  (on ensemble output)
# ─────────────────────────────────────────────────────────────────────

def walk_forward_validate(df: pd.DataFrame,
                           n_folds: int = WF_FOLDS) -> dict:
    """
    Expanding-window walk-forward validation across all horizons.
    Per fold: train LSTM + XGBoost on past, test on future.
    Returns aggregated MAE/RMSE/R2/DirAcc per horizon.
    """
    close    = df["Close"].values
    features = df[FEATURE_COLS].values
    scaler   = MinMaxScaler()
    f_scaled = scaler.fit_transform(features)

    X_seq, y_arrs = build_lstm_sequences(f_scaled, close)
    n_samples     = len(X_seq)
    fold_size     = n_samples // (n_folds + 1)

    if fold_size < 5:
        raise ValueError(
            f"Too few samples ({n_samples}) for {n_folds} folds"
        )

    # Accumulate out-of-sample preds per horizon
    oos = {lbl: {"pred": [], "true": []} for lbl in HORIZONS}

    early_stop = EarlyStopping(
        monitor="val_loss", patience=4,
        restore_best_weights=True, verbose=0,
    )

    for fold in range(n_folds):
        tr_end  = fold_size * (fold + 1)
        te_end  = tr_end + fold_size
        if te_end > n_samples:
            break

        X_tr  = X_seq[:tr_end]
        X_te  = X_seq[tr_end:te_end]
        y_tr  = {lbl: y_arrs[lbl][:tr_end]  for lbl in HORIZONS}
        y_te  = {lbl: y_arrs[lbl][tr_end:te_end] for lbl in HORIZONS}

        # ── LSTM ──────────────────────────────────────────────────
        val_split  = int(len(X_tr) * 0.85)
        X_tv, y_tv = X_tr[:val_split], {l: y_tr[l][:val_split] for l in HORIZONS}
        X_vl, y_vl = X_tr[val_split:], {l: y_tr[l][val_split:] for l in HORIZONS}
        if len(X_vl) == 0:
            X_vl, y_vl = X_tv[-5:], {l: y_tv[l][-5:] for l in HORIZONS}

        lstm_model = build_multi_output_lstm(SEQ_LEN, len(FEATURE_COLS))
        lstm_model.fit(
            X_tv,
            [y_tv[lbl] for lbl in HORIZONS],
            validation_data=(X_vl, [y_vl[lbl] for lbl in HORIZONS]),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop],
            verbose=0,
        )
        lstm_preds_te = lstm_model.predict(X_te, verbose=0)   # list of 3 arrays

        # ── XGBoost (one per horizon) ──────────────────────────────
        xgb_preds_te = []
        for i, lbl in enumerate(HORIZONS):
            X_flat_tr, y_flat_tr = build_xgb_dataset(X_tr, y_tr[lbl])
            X_flat_te, _         = build_xgb_dataset(X_te, y_te[lbl])

            xgb_model = xgb.XGBRegressor(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                verbosity=0,
            )
            xgb_model.fit(X_flat_tr, y_flat_tr)
            xgb_preds_te.append(xgb_model.predict(X_flat_te))

        # ── Ensemble: simple average ───────────────────────────────
        for i, lbl in enumerate(HORIZONS):
            lstm_h = lstm_preds_te[i].flatten()
            xgb_h  = xgb_preds_te[i].flatten()
            blend  = (lstm_h + xgb_h) / 2.0

            oos[lbl]["pred"].extend(blend.tolist())
            oos[lbl]["true"].extend(y_te[lbl].tolist())

    # ── Compute metrics from accumulated OOS predictions ──────────
    metrics = {}
    for lbl in HORIZONS:
        y_true = np.array(oos[lbl]["true"])
        y_pred = np.array(oos[lbl]["pred"])

        if len(y_true) < 2:
            raise ValueError(f"Too few OOS predictions for horizon {lbl}")

        metrics[lbl] = {
            "mae"    : round(float(mean_absolute_error(y_true, y_pred)), 4),
            "rmse"   : round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
            "r2"     : round(float(r2_score(y_true, y_pred)), 4),
            "dir_acc": round(float(np.mean(np.sign(y_pred) == np.sign(y_true)) * 100), 2),
        }

    return metrics


# ─────────────────────────────────────────────────────────────────────
# 8. FINAL ENSEMBLE TRAIN + PREDICT
# ─────────────────────────────────────────────────────────────────────

def train_and_predict_ensemble(df: pd.DataFrame) -> dict:
    """
    Train LSTM + XGBoost on full history.
    Returns blended % return prediction per horizon.
    """
    close    = df["Close"].values
    features = df[FEATURE_COLS].values
    scaler   = MinMaxScaler()
    f_scaled = scaler.fit_transform(features)

    X_seq, y_arrs = build_lstm_sequences(f_scaled, close)

    if len(X_seq) < 30:
        raise ValueError(f"Insufficient sequences: {len(X_seq)}")

    split      = int(len(X_seq) * 0.85)
    X_tr, X_vl = X_seq[:split], X_seq[split:]
    y_tr = {lbl: y_arrs[lbl][:split]  for lbl in HORIZONS}
    y_vl = {lbl: y_arrs[lbl][split:]  for lbl in HORIZONS}

    if len(X_vl) == 0:
        X_vl = X_tr[-5:]
        y_vl = {lbl: y_tr[lbl][-5:] for lbl in HORIZONS}

    # ── Train multi-output LSTM ────────────────────────────────────
    early_stop = EarlyStopping(
        monitor="val_loss", patience=6,
        restore_best_weights=True, verbose=0,
    )
    lstm_model = build_multi_output_lstm(SEQ_LEN, len(FEATURE_COLS))
    lstm_model.fit(
        X_tr,
        [y_tr[lbl] for lbl in HORIZONS],
        validation_data=(X_vl, [y_vl[lbl] for lbl in HORIZONS]),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0,
    )

    # Live sequence: last SEQ_LEN rows
    last_seq   = f_scaled[-SEQ_LEN:].reshape(1, SEQ_LEN, len(FEATURE_COLS))
    lstm_preds = lstm_model.predict(last_seq, verbose=0)   # list of 3 (1,1) arrays

    # ── Train XGBoost + predict ────────────────────────────────────
    X_flat_all, _ = build_xgb_dataset(X_seq, y_arrs["1d"])   # y unused here
    last_flat      = window_to_xgb_features(
        f_scaled[-SEQ_LEN:]
    ).reshape(1, -1)

    xgb_preds = []
    for lbl in HORIZONS:
        xgb_model = xgb.XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            verbosity=0,
        )
        xgb_model.fit(X_flat_all, y_arrs[lbl])
        xgb_preds.append(float(xgb_model.predict(last_flat)[0]))

    # ── Blend ──────────────────────────────────────────────────────
    predictions = {}
    for i, lbl in enumerate(HORIZONS):
        lstm_val = float(lstm_preds[i][0][0])
        xgb_val  = xgb_preds[i]
        blend    = (lstm_val + xgb_val) / 2.0
        predictions[lbl] = {
            "lstm" : round(lstm_val, 4),
            "xgb"  : round(xgb_val, 4),
            "blend": round(blend, 4),
        }

    return predictions


# ─────────────────────────────────────────────────────────────────────
# 9. PER-STOCK PIPELINE
# ─────────────────────────────────────────────────────────────────────

def run_stock(ticker: str, index: int, total: int):
    print(f"\n[{index:>3}/{total}]  {ticker}")
    print(f"  {'─'*54}")

    try:
        # ── Download ──────────────────────────────────────────────
        raw = yf.download(
            ticker,
            start=START_DATE, end=END_DATE,
            progress=False, auto_adjust=True,
        )
        if raw.empty:
            raise ValueError("No data returned")
        if len(raw) < SEQ_LEN + max(HORIZONS.values()) + 50:
            raise ValueError(f"Insufficient rows: {len(raw)}")

        # ── Feature Engineering ───────────────────────────────────
        df = add_features(raw)
        print(f"  Data: {len(raw)} raw rows → {len(df)} after features")

        # ── Walk-Forward Validation ───────────────────────────────
        print(f"  Walk-forward validation ({WF_FOLDS} folds) ...", flush=True)
        wf_metrics = walk_forward_validate(df)

        for lbl in HORIZONS:
            m = wf_metrics[lbl]
            print(
                f"    [{lbl:>2}]  MAE={m['mae']:.3f}  "
                f"RMSE={m['rmse']:.3f}  R2={m['r2']:.3f}  "
                f"DirAcc={m['dir_acc']:.1f}%"
            )

        # ── Final Ensemble Prediction ─────────────────────────────
        print(f"  Training final ensemble ...", flush=True)
        preds = train_and_predict_ensemble(df)

        for lbl, p in preds.items():
            print(
                f"    [{lbl:>2}]  LSTM={p['lstm']:+.2f}%  "
                f"XGB={p['xgb']:+.2f}%  "
                f"Blend={p['blend']:+.2f}%"
            )

        # ── Build result row ──────────────────────────────────────
        result = {
            "Ticker": ticker,
            "Date"  : datetime.today().strftime("%Y-%m-%d"),
        }
        for lbl in HORIZONS:
            result[f"pred_{lbl}_pct"]      = preds[lbl]["blend"]
            result[f"lstm_{lbl}_pct"]      = preds[lbl]["lstm"]
            result[f"xgb_{lbl}_pct"]       = preds[lbl]["xgb"]
            result[f"mae_{lbl}"]           = wf_metrics[lbl]["mae"]
            result[f"rmse_{lbl}"]          = wf_metrics[lbl]["rmse"]
            result[f"r2_{lbl}"]            = wf_metrics[lbl]["r2"]
            result[f"dir_acc_{lbl}_pct"]   = wf_metrics[lbl]["dir_acc"]

        return result

    except Exception as exc:
        logging.error("%s | %s", ticker, exc)
        print(f"  Skipped -> {exc}")
        return None


# ─────────────────────────────────────────────────────────────────────
# 10. MAIN
# ─────────────────────────────────────────────────────────────────────

CSV_COLUMNS = (
    ["Ticker", "Date"]
    + [f"pred_{lbl}_pct"    for lbl in HORIZONS]
    + [f"lstm_{lbl}_pct"    for lbl in HORIZONS]
    + [f"xgb_{lbl}_pct"     for lbl in HORIZONS]
    + [f"mae_{lbl}"         for lbl in HORIZONS]
    + [f"rmse_{lbl}"        for lbl in HORIZONS]
    + [f"r2_{lbl}"          for lbl in HORIZONS]
    + [f"dir_acc_{lbl}_pct" for lbl in HORIZONS]
)


def main():
    div = "=" * 62
    print(f"\n{div}")
    print("  LSTM + XGBoost Ensemble  |  Nifty 100  |  NSE India  v2")
    print(f"  Run date   : {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Data range : {START_DATE.strftime('%Y-%m-%d')} -> {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Stocks     : {len(NIFTY_100_TICKERS)}")
    print(f"  Model      : Multi-output LSTM + XGBoost (blended)")
    print(f"  Validation : Walk-Forward ({WF_FOLDS} expanding folds)")
    print(f"  Output     : {OUTPUT_CSV}")
    print(f"{div}\n")

    success_count = 0
    skipped_count = 0

    for idx, ticker in enumerate(NIFTY_100_TICKERS, start=1):
        result = run_stock(ticker, idx, len(NIFTY_100_TICKERS))

        if result:
            row_df = pd.DataFrame([result], columns=CSV_COLUMNS)
            if os.path.exists(OUTPUT_CSV):
                row_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
            else:
                row_df.to_csv(OUTPUT_CSV, index=False)
            success_count += 1
        else:
            skipped_count += 1

    # ── Final Summary ─────────────────────────────────────────────
    print(f"\n{div}")
    print(f"  Pipeline complete")
    print(f"  Successful : {success_count} / {len(NIFTY_100_TICKERS)}")
    print(f"  Skipped    : {skipped_count}  (see {LOG_FILE})")
    print(f"  Saved to   : {os.path.abspath(OUTPUT_CSV)}")

    if success_count > 0 and os.path.exists(OUTPUT_CSV):
        final_df = pd.read_csv(OUTPUT_CSV)

        print(f"\n  Aggregate Accuracy  (mean over {success_count} stocks)")
        print(f"  {'─'*54}")
        for lbl in HORIZONS:
            mae = final_df[f"mae_{lbl}"].mean()
            rmse= final_df[f"rmse_{lbl}"].mean()
            r2  = final_df[f"r2_{lbl}"].mean()
            da  = final_df[f"dir_acc_{lbl}_pct"].mean()
            print(
                f"  [{lbl:>2}]  MAE={mae:.3f}  RMSE={rmse:.3f}  "
                f"R2={r2:.3f}  DirAcc={da:.1f}%"
            )

        print(f"\n  Top 5 stocks by 1-day directional accuracy:")
        top5 = (
            final_df[["Ticker", "pred_1d_pct", "dir_acc_1d_pct",
                       "pred_1w_pct", "dir_acc_1w_pct"]]
            .sort_values("dir_acc_1d_pct", ascending=False)
            .head(5)
        )
        print(top5.to_string(index=False))
    print(f"\n{div}\n")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'ta'

#Version 3

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║   Stock Prediction Pipeline v3  —  Nifty 100  (NSE India)          ║
║                                                                      ║
║  Key upgrades over v2:                                               ║
║    • Per-horizon lookback windows  (1d=15, 1w=30, 1m=60 days)      ║
║    • 1d: XGBoost-only  (faster, better on short-term tabular data)  ║
║    • 1w: LSTM + XGBoost ensemble  (blended, medium horizon)         ║
║    • 1m: LSTM + XGBoost ensemble  (unchanged, was already working)  ║
║    • Walk-forward validation per horizon with correct window         ║
║                                                                      ║
║  Output CSV:                                                         ║
║    Ticker | Date | pred_1d_pct | pred_1w_pct | pred_1m_pct          ║
║    + accuracy metrics per horizon                                    ║
╚══════════════════════════════════════════════════════════════════════╝

Requirements:
    pip install yfinance ta scikit-learn xgboost tensorflow pandas numpy
"""

import os
import warnings
import logging
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import ta

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────

# Per-horizon lookback windows  ← KEY CHANGE FROM V2
SEQ_LEN = {
    "1d": 15,    # only last 15 days for tomorrow — avoids old noise
    "1w": 30,    # 30 days for 1-week — captures recent trend
    "1m": 60,    # 60 days for 1-month — needs longer memory
}

# Per-horizon model type  ← KEY CHANGE FROM V2
# "xgb"    = XGBoost only (no LSTM)
# "blend"  = LSTM + XGBoost averaged
MODEL_TYPE = {
    "1d": "xgb",      # XGBoost-only for 1-day
    "1w": "blend",    # LSTM + XGBoost for 1-week
    "1m": "blend",    # LSTM + XGBoost for 1-month
}

HORIZONS       = {"1d": 1, "1w": 5, "1m": 21}   # label -> trading days ahead
EPOCHS         = 60
BATCH_SIZE     = 32
WF_FOLDS       = 5
OUTPUT_CSV     = "nifty100_predictions_v3.csv"
LOG_FILE       = "pipeline_v3_errors.log"

END_DATE       = datetime.today()
START_DATE     = END_DATE - timedelta(days=365 * 3)   # 3 years

# ─────────────────────────────────────────────────────────────────────
# 2. NIFTY 100 TICKERS
# ─────────────────────────────────────────────────────────────────────
_NIFTY100_SYMBOLS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "SBIN", "BAJFINANCE", "BHARTIARTL", "KOTAKBANK",
    "LT", "HCLTECH", "ASIANPAINT", "AXISBANK", "MARUTI",
    "SUNPHARMA", "TITAN", "ULTRACEMCO", "WIPRO", "NESTLEIND",
    "POWERGRID", "NTPC", "TECHM", "INDUSINDBK", "M&M",
    "ONGC", "TATAMOTORS", "BAJAJFINSV", "ADANIPORTS", "COALINDIA",
    "JSWSTEEL", "TATASTEEL", "HINDALCO", "DRREDDY", "CIPLA",
    "DIVISLAB", "APOLLOHOSP", "EICHERMOT", "BPCL", "GRASIM",
    "BRITANNIA", "SBILIFE", "HDFCLIFE", "BAJAJ-AUTO", "SHREECEM",
    "HEROMOTOCO", "UPL", "TATACONSUM", "ADANIENT", "VEDL",
    "CHOLAFIN", "DLF", "SIEMENS", "ABB", "PIDILITIND",
    "HAVELLS", "MUTHOOTFIN", "BANKBARODA", "PNB", "CANBK",
    "UNIONBANK", "IDFCFIRSTB", "FEDERALBNK", "RBLBANK", "BANDHANBNK",
    "BERGEPAINT", "GODREJCP", "COLPAL", "DABUR", "EMAMILTD",
    "MARICO", "TRENT", "NYKAA", "ZOMATO", "DMART",
    "IRCTC", "POLYCAB", "AMBUJACEM", "ACC", "INDIGO",
    "ADANIGREEN", "TATAPOWER", "TORNTPOWER", "NHPC", "SJVN",
    "RECLTD", "PFC", "IRFC", "SAIL", "NMDC",
    "NATIONALUM", "HINDCOPPER", "MPHASIS", "LTIM", "PERSISTENT",
    "COFORGE", "OFSS", "KPITTECH", "GMRINFRA", "CUMMINSIND",
]
NIFTY_100_TICKERS = [f"{s}.NS" for s in _NIFTY100_SYMBOLS]

# ─────────────────────────────────────────────────────────────────────
# 3. LOGGING
# ─────────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE, level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ─────────────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING
#    Base features — used by all horizons
#    Short-term features — extra columns for 1d / 1w only
# ─────────────────────────────────────────────────────────────────────

BASE_FEATURES = [
    "Close", "Volume",
    "RSI", "MACD", "MACD_signal", "MACD_diff",
    "BB_upper", "BB_lower", "BB_pct",
    "EMA_9", "EMA_21", "EMA_50",
    "ATR", "OBV", "Return",
]

# Additional features specifically helpful for short-horizon prediction
SHORT_TERM_FEATURES = [
    "Gap_pct",          # overnight gap: (open - prev_close) / prev_close
    "HL_range_pct",     # intraday range: (high - low) / close
    "Vol_surge",        # volume relative to 10-day avg (> 1 = above avg)
    "Price_streak",     # consecutive up/down days (-3 to +3)
    "Stoch_k",          # Stochastic %K (short-term momentum)
    "Williams_R",       # Williams %R (overbought/oversold, very short-term)
    "Return_3d",        # 3-day cumulative return
    "Return_5d",        # 5-day cumulative return
]

# Feature sets per horizon
FEATURES = {
    "1d": BASE_FEATURES + SHORT_TERM_FEATURES,
    "1w": BASE_FEATURES + SHORT_TERM_FEATURES,
    "1m": BASE_FEATURES,    # 1m uses only base — same as v2
}


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute all technical indicators including short-term features.
    Short-term features are always computed — they're simply not used
    for 1m sequences (FEATURES["1m"] excludes them).
    """
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close = df["Close"]
    high  = df["High"]
    low   = df["Low"]
    vol   = df["Volume"]
    open_ = df["Open"]

    # ── Base features ─────────────────────────────────────────────
    df["RSI"]         = ta.momentum.RSIIndicator(close, 14).rsi()
    macd              = ta.trend.MACD(close)
    df["MACD"]        = macd.macd()
    df["MACD_signal"] = macd.macd_signal()
    df["MACD_diff"]   = macd.macd_diff()
    bb                = ta.volatility.BollingerBands(close, 20)
    df["BB_upper"]    = bb.bollinger_hband()
    df["BB_lower"]    = bb.bollinger_lband()
    df["BB_pct"]      = bb.bollinger_pband()
    df["EMA_9"]       = ta.trend.EMAIndicator(close, 9).ema_indicator()
    df["EMA_21"]      = ta.trend.EMAIndicator(close, 21).ema_indicator()
    df["EMA_50"]      = ta.trend.EMAIndicator(close, 50).ema_indicator()
    df["ATR"]         = ta.volatility.AverageTrueRange(
                            high, low, close, 14).average_true_range()
    df["OBV"]         = ta.volume.OnBalanceVolumeIndicator(
                            close, vol).on_balance_volume()
    df["Return"]      = close.pct_change()

    # ── Short-term features ───────────────────────────────────────
    # Gap %: overnight gap (open vs previous close)
    df["Gap_pct"]     = (open_ - close.shift(1)) / close.shift(1)

    # High-Low intraday range as % of close
    df["HL_range_pct"] = (high - low) / close

    # Volume surge: today's vol vs 10-day rolling avg
    vol_avg           = vol.rolling(10).mean()
    df["Vol_surge"]   = vol / vol_avg.replace(0, np.nan)

    # Price streak: count consecutive up or down closes (-5 to +5)
    direction         = np.sign(df["Return"])
    streak            = []
    count             = 0
    for d in direction:
        if np.isnan(d):
            streak.append(np.nan)
            count = 0
        elif d > 0:
            count = count + 1 if count > 0 else 1
            streak.append(min(count, 5))
        elif d < 0:
            count = count - 1 if count < 0 else -1
            streak.append(max(count, -5))
        else:
            count = 0
            streak.append(0)
    df["Price_streak"] = streak

    # Stochastic %K — 14-period
    df["Stoch_k"]     = ta.momentum.StochasticOscillator(
                            high, low, close, 14).stoch()

    # Williams %R — 14-period (range -100 to 0, where -100=oversold)
    df["Williams_R"]  = ta.momentum.WilliamsRIndicator(
                            high, low, close, 14).williams_r()

    # Multi-day cumulative returns
    df["Return_3d"]   = close.pct_change(3)
    df["Return_5d"]   = close.pct_change(5)

    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────────────────────────────
# 5. SEQUENCE BUILDER
# ─────────────────────────────────────────────────────────────────────

def build_sequences(features_scaled: np.ndarray,
                    close: np.ndarray,
                    seq_len: int,
                    horizon_days: int):
    """
    Sliding windows with horizon-specific seq_len.
    X: (n, seq_len, n_features)
    y: % return horizon_days ahead
    """
    X, y = [], []
    for i in range(seq_len, len(features_scaled) - horizon_days):
        X.append(features_scaled[i - seq_len : i])
        ret = (close[i + horizon_days] - close[i]) / close[i] * 100
        y.append(ret)
    return np.array(X), np.array(y)


# ─────────────────────────────────────────────────────────────────────
# 6. XGB FEATURE EXTRACTOR
#    Compress a window into flat summary stats for XGBoost
# ─────────────────────────────────────────────────────────────────────

def window_to_xgb_row(window: np.ndarray) -> np.ndarray:
    """
    Compress (seq_len, n_features) → 1D vector.
    Stats: mean | std | min | max | last | slope per feature.
    Min/max added vs v2 — important for short windows (15 days).
    """
    t     = np.arange(len(window))
    mean  = window.mean(axis=0)
    std   = window.std(axis=0)
    mn    = window.min(axis=0)
    mx    = window.max(axis=0)
    last  = window[-1]
    slope = np.array([np.polyfit(t, window[:, j], 1)[0]
                      for j in range(window.shape[1])])
    return np.concatenate([mean, std, mn, mx, last, slope])


def build_xgb_dataset(X_seq: np.ndarray, y: np.ndarray):
    X_flat = np.array([window_to_xgb_row(seq) for seq in X_seq])
    return X_flat, y


# ─────────────────────────────────────────────────────────────────────
# 7. LSTM MODEL BUILDER
# ─────────────────────────────────────────────────────────────────────

def build_lstm(seq_len: int, n_features: int) -> Model:
    """Single-output LSTM for one horizon."""
    inp = Input(shape=(seq_len, n_features))
    x   = LSTM(64, return_sequences=True)(inp)
    x   = Dropout(0.2)(x)
    x   = LSTM(32)(x)
    x   = Dropout(0.2)(x)
    out = Dense(1, activation="linear")(x)
    model = Model(inp, out)
    model.compile(optimizer="adam", loss="mse")
    return model


# ─────────────────────────────────────────────────────────────────────
# 8. XGB MODEL BUILDER
# ─────────────────────────────────────────────────────────────────────

def build_xgb(horizon_label: str) -> xgb.XGBRegressor:
    """
    XGBoost with horizon-tuned hyperparameters.
    1d uses more trees + shallower depth (avoids overfitting on short window).
    1m uses fewer trees (targets longer-term signal).
    """
    params = {
        "1d": dict(n_estimators=400, max_depth=3, learning_rate=0.03,
                   subsample=0.7, colsample_bytree=0.7,
                   min_child_weight=5, verbosity=0),
        "1w": dict(n_estimators=300, max_depth=4, learning_rate=0.04,
                   subsample=0.8, colsample_bytree=0.8,
                   min_child_weight=3, verbosity=0),
        "1m": dict(n_estimators=300, max_depth=4, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8,
                   min_child_weight=2, verbosity=0),
    }
    return xgb.XGBRegressor(**params[horizon_label])


# ─────────────────────────────────────────────────────────────────────
# 9. WALK-FORWARD VALIDATION  (per horizon, correct window size)
# ─────────────────────────────────────────────────────────────────────

def walk_forward_validate(df: pd.DataFrame,
                           label: str,
                           horizon_days: int,
                           n_folds: int = WF_FOLDS) -> dict:
    """
    Walk-forward validation using the horizon-specific seq_len and features.
    Model type (xgb-only vs blend) is read from MODEL_TYPE config.
    """
    seq_len      = SEQ_LEN[label]
    feat_cols    = FEATURES[label]
    close        = df["Close"].values
    features     = df[feat_cols].values
    scaler       = MinMaxScaler()
    f_scaled     = scaler.fit_transform(features)

    X_seq, y_arr = build_sequences(f_scaled, close, seq_len, horizon_days)
    n_samples    = len(X_seq)
    fold_size    = n_samples // (n_folds + 1)

    if fold_size < 5:
        raise ValueError(
            f"Too few samples ({n_samples}) for {n_folds} folds "
            f"at label={label}"
        )

    all_preds, all_true = [], []
    early_stop = EarlyStopping(
        monitor="val_loss", patience=4,
        restore_best_weights=True, verbose=0,
    )

    for fold in range(n_folds):
        tr_end = fold_size * (fold + 1)
        te_end = tr_end + fold_size
        if te_end > n_samples:
            break

        X_tr, X_te = X_seq[:tr_end], X_seq[tr_end:te_end]
        y_tr, y_te = y_arr[:tr_end], y_arr[tr_end:te_end]

        val_split    = int(len(X_tr) * 0.85)
        X_tv, y_tv   = X_tr[:val_split], y_tr[:val_split]
        X_vl, y_vl   = X_tr[val_split:], y_tr[val_split:]
        if len(X_vl) == 0:
            X_vl, y_vl = X_tv[-5:], y_tv[-5:]

        model_type = MODEL_TYPE[label]

        # ── XGBoost ───────────────────────────────────────────────
        X_flat_tr, _ = build_xgb_dataset(X_tr, y_tr)
        X_flat_te, _ = build_xgb_dataset(X_te, y_te)
        xgb_m        = build_xgb(label)
        xgb_m.fit(X_flat_tr, y_tr)
        xgb_preds    = xgb_m.predict(X_flat_te)

        if model_type == "xgb":
            # XGBoost only — no LSTM
            fold_preds = xgb_preds

        else:
            # LSTM + XGBoost blend
            lstm_m = build_lstm(seq_len, len(feat_cols))
            lstm_m.fit(
                X_tv, y_tv,
                validation_data=(X_vl, y_vl),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=[early_stop],
                verbose=0,
            )
            lstm_preds = lstm_m.predict(X_te, verbose=0).flatten()
            fold_preds = (lstm_preds + xgb_preds) / 2.0

        all_preds.extend(fold_preds.tolist())
        all_true.extend(y_te.tolist())

    if len(all_preds) < 2:
        raise ValueError("Walk-forward produced too few test predictions")

    y_true = np.array(all_true)
    y_pred = np.array(all_preds)

    return {
        "mae"    : round(float(mean_absolute_error(y_true, y_pred)), 4),
        "rmse"   : round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
        "r2"     : round(float(r2_score(y_true, y_pred)), 4),
        "dir_acc": round(float(np.mean(np.sign(y_pred) == np.sign(y_true)) * 100), 2),
    }


# ─────────────────────────────────────────────────────────────────────
# 10. FINAL TRAIN + PREDICT  (per horizon)
# ─────────────────────────────────────────────────────────────────────

def train_and_predict(df: pd.DataFrame,
                       label: str,
                       horizon_days: int) -> dict:
    """
    Train on full history using horizon-correct seq_len and features.
    Returns dict with prediction and individual model values.
    """
    seq_len   = SEQ_LEN[label]
    feat_cols = FEATURES[label]
    close     = df["Close"].values
    features  = df[feat_cols].values
    scaler    = MinMaxScaler()
    f_scaled  = scaler.fit_transform(features)

    X_seq, y_arr = build_sequences(f_scaled, close, seq_len, horizon_days)

    if len(X_seq) < 30:
        raise ValueError(f"Insufficient sequences: {len(X_seq)} for {label}")

    split      = int(len(X_seq) * 0.85)
    X_tr, X_vl = X_seq[:split], X_seq[split:]
    y_tr, y_vl = y_arr[:split], y_arr[split:]
    if len(X_vl) == 0:
        X_vl, y_vl = X_tr[-5:], y_tr[-5:]

    model_type = MODEL_TYPE[label]
    last_seq   = f_scaled[-seq_len:].reshape(1, seq_len, len(feat_cols))
    last_flat  = window_to_xgb_row(f_scaled[-seq_len:]).reshape(1, -1)

    # ── XGBoost ───────────────────────────────────────────────────
    X_flat_all, _ = build_xgb_dataset(X_seq, y_arr)
    xgb_m         = build_xgb(label)
    xgb_m.fit(X_flat_all, y_arr)
    xgb_pred      = float(xgb_m.predict(last_flat)[0])

    if model_type == "xgb":
        return {
            "pred"  : round(xgb_pred, 4),
            "xgb"   : round(xgb_pred, 4),
            "lstm"  : None,
        }

    # ── LSTM + XGBoost blend ──────────────────────────────────────
    early_stop = EarlyStopping(
        monitor="val_loss", patience=6,
        restore_best_weights=True, verbose=0,
    )
    lstm_m = build_lstm(seq_len, len(feat_cols))
    lstm_m.fit(
        X_tr, y_tr,
        validation_data=(X_vl, y_vl),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0,
    )
    lstm_pred = float(lstm_m.predict(last_seq, verbose=0)[0][0])
    blend     = (lstm_pred + xgb_pred) / 2.0

    return {
        "pred" : round(blend, 4),
        "xgb"  : round(xgb_pred, 4),
        "lstm" : round(lstm_pred, 4),
    }


# ─────────────────────────────────────────────────────────────────────
# 11. PER-STOCK PIPELINE
# ─────────────────────────────────────────────────────────────────────

def run_stock(ticker: str, index: int, total: int):
    print(f"\n[{index:>3}/{total}]  {ticker}")
    print(f"  {'─'*54}")

    try:
        raw = yf.download(
            ticker,
            start=START_DATE, end=END_DATE,
            progress=False, auto_adjust=True,
        )
        if raw.empty:
            raise ValueError("No data returned")
        if len(raw) < max(SEQ_LEN.values()) + max(HORIZONS.values()) + 50:
            raise ValueError(f"Insufficient rows: {len(raw)}")

        df = add_features(raw)
        print(f"  Data: {len(raw)} raw -> {len(df)} rows after features")

        result = {
            "Ticker": ticker,
            "Date"  : datetime.today().strftime("%Y-%m-%d"),
        }

        for label, horizon_days in HORIZONS.items():
            mtype = MODEL_TYPE[label]
            seq   = SEQ_LEN[label]
            print(
                f"  [{label}] "
                f"model={mtype:<5}  seq={seq}d  "
                "walk-forward ...",
                end="  ", flush=True,
            )

            # Walk-forward validation
            metrics = walk_forward_validate(df, label, horizon_days)
            print(
                f"MAE={metrics['mae']:.3f}  "
                f"RMSE={metrics['rmse']:.3f}  "
                f"R2={metrics['r2']:.3f}  "
                f"DirAcc={metrics['dir_acc']:.1f}%",
                end="  ", flush=True,
            )

            # Final prediction
            preds = train_and_predict(df, label, horizon_days)
            print(f"Pred={preds['pred']:+.2f}%")

            result[f"pred_{label}_pct"]    = preds["pred"]
            result[f"xgb_{label}_pct"]     = preds["xgb"]
            result[f"lstm_{label}_pct"]    = preds["lstm"]  # None for 1d
            result[f"mae_{label}"]         = metrics["mae"]
            result[f"rmse_{label}"]        = metrics["rmse"]
            result[f"r2_{label}"]          = metrics["r2"]
            result[f"dir_acc_{label}_pct"] = metrics["dir_acc"]

        return result

    except Exception as exc:
        logging.error("%s | %s", ticker, exc)
        print(f"  Skipped -> {exc}")
        return None


# ─────────────────────────────────────────────────────────────────────
# 12. MAIN
# ─────────────────────────────────────────────────────────────────────

CSV_COLUMNS = (
    ["Ticker", "Date"]
    + [f"pred_{l}_pct"    for l in HORIZONS]
    + [f"xgb_{l}_pct"     for l in HORIZONS]
    + [f"lstm_{l}_pct"    for l in HORIZONS]
    + [f"mae_{l}"         for l in HORIZONS]
    + [f"rmse_{l}"        for l in HORIZONS]
    + [f"r2_{l}"          for l in HORIZONS]
    + [f"dir_acc_{l}_pct" for l in HORIZONS]
)


def main():
    div = "=" * 64
    print(f"\n{div}")
    print("  Stock Prediction Pipeline v3  |  Nifty 100  |  NSE India")
    print(f"  Run date   : {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Data range : {START_DATE.strftime('%Y-%m-%d')} -> {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Stocks     : {len(NIFTY_100_TICKERS)}")
    print(f"  1d model   : XGBoost only  (seq=15d, short-term features)")
    print(f"  1w model   : LSTM + XGBoost blend  (seq=30d)")
    print(f"  1m model   : LSTM + XGBoost blend  (seq=60d)")
    print(f"  Validation : Walk-forward ({WF_FOLDS} folds, per-horizon window)")
    print(f"  Output     : {OUTPUT_CSV}")
    print(f"{div}\n")

    success_count = 0
    skipped_count = 0

    for idx, ticker in enumerate(NIFTY_100_TICKERS, start=1):
        result = run_stock(ticker, idx, len(NIFTY_100_TICKERS))

        if result:
            row_df = pd.DataFrame([result], columns=CSV_COLUMNS)
            if os.path.exists(OUTPUT_CSV):
                row_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
            else:
                row_df.to_csv(OUTPUT_CSV, index=False)
            success_count += 1
        else:
            skipped_count += 1

    # ── Summary ───────────────────────────────────────────────────
    print(f"\n{div}")
    print(f"  Pipeline complete")
    print(f"  Successful : {success_count} / {len(NIFTY_100_TICKERS)}")
    print(f"  Skipped    : {skipped_count}  (see {LOG_FILE})")
    print(f"  Saved to   : {os.path.abspath(OUTPUT_CSV)}")

    if success_count > 0 and os.path.exists(OUTPUT_CSV):
        final_df = pd.read_csv(OUTPUT_CSV)

        print(f"\n  Aggregate accuracy (mean over {success_count} stocks)")
        print(f"  {'─'*54}")
        print(f"  {'Horizon':<8} {'Model':<12} {'MAE':>6} {'RMSE':>6} {'R2':>7} {'DirAcc':>8}")
        print(f"  {'─'*54}")
        for label in HORIZONS:
            mtype = MODEL_TYPE[label]
            mae   = final_df[f"mae_{label}"].mean()
            rmse  = final_df[f"rmse_{label}"].mean()
            r2    = final_df[f"r2_{label}"].mean()
            da    = final_df[f"dir_acc_{label}_pct"].mean()
            print(
                f"  {label:<8} {mtype:<12} "
                f"{mae:>6.3f} {rmse:>6.3f} {r2:>7.3f} {da:>7.1f}%"
            )

        print(f"\n  Top 5 stocks by 1-day directional accuracy:")
        top5 = (
            final_df[["Ticker", "pred_1d_pct", "dir_acc_1d_pct",
                       "pred_1w_pct", "dir_acc_1w_pct",
                       "pred_1m_pct", "dir_acc_1m_pct"]]
            .sort_values("dir_acc_1d_pct", ascending=False)
            .head(5)
        )
        print(top5.to_string(index=False))
    print(f"\n{div}\n")


if __name__ == "__main__":
    main()

#final merger for v2


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║         Merge Script  —  LSTM Predictions + Sentiment Scores        ║
║                                                                      ║
║  Reads:                                                              ║
║    nifty100_predictions_v2.csv   (from lstm_nifty100_pipeline_v2.py)║
║    sentiment_scores.csv          (from sentiment_pipeline.py)        ║
║                                                                      ║
║  Formula:                                                            ║
║    sentiment_pct  = sentiment_score × |model_blend|                 ║
║    final_Xd       = 0.80 × pred_Xd_pct + 0.20 × sentiment_pct_Xd  ║
║                                                                      ║
║  Output:                                                             ║
║    final_predictions.csv                                            ║
╚══════════════════════════════════════════════════════════════════════╝

Run order:
    1. python lstm_nifty100_pipeline_v2.py   →  nifty100_predictions_v2.csv
    2. python sentiment_pipeline.py           →  sentiment_scores.csv
    3. python merge_final_predictions.py      →  final_predictions.csv
"""

import os
import pandas as pd
import numpy as np
from datetime import datetime

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────
LSTM_CSV       = "nifty100_predictions_v2.csv"
SENTIMENT_CSV  = "sentiment_scores.csv"
OUTPUT_CSV     = "final_predictions.csv"

MODEL_WEIGHT   = 0.80    # 80% weight to LSTM+XGBoost blend
SENTIMENT_WEIGHT = 0.20  # 20% weight to sentiment

HORIZONS       = ["1d", "1w", "1m"]

TODAY          = datetime.today().strftime("%Y-%m-%d")


# ─────────────────────────────────────────────────────────────────────
# 2. SENTIMENT SCALING
#
#   The LSTM/XGBoost blend produces % return predictions (e.g. +1.5%)
#   The sentiment score is a float between -1.0 and +1.0
#
#   To make them comparable, we scale sentiment by the magnitude
#   of the model prediction for that horizon:
#
#       sentiment_pct = sentiment_score × |model_blend|
#
#   Example:
#       model predicts +2.0%,  sentiment = +0.8 (strongly positive)
#       sentiment_pct = +0.8 × 2.0 = +1.6%
#       final = 0.8 × 2.0 + 0.2 × 1.6 = 1.6 + 0.32 = +1.92%
#
#   Example (conflicting signals):
#       model predicts -2.0%,  sentiment = +0.8 (positive news)
#       sentiment_pct = +0.8 × 2.0 = +1.6%
#       final = 0.8 × (-2.0) + 0.2 × 1.6 = -1.6 + 0.32 = -1.28%
#       → Sentiment partially offsets the bearish model prediction
#
#   If no sentiment data for a stock:
#       sentiment_score = 0.0  → final = model_blend (no change)
# ─────────────────────────────────────────────────────────────────────

def compute_final_predictions(lstm_row: pd.Series,
                               sentiment_score: float) -> dict:
    """
    Apply the blending formula for all three horizons.
    Returns dict of final_1d, final_1w, final_1m predictions.
    """
    results = {}

    for h in HORIZONS:
        model_pred   = lstm_row.get(f"pred_{h}_pct", 0.0)

        # Scale sentiment to the same magnitude as model prediction
        sentiment_pct = sentiment_score * abs(model_pred)

        # Weighted blend
        final = MODEL_WEIGHT * model_pred + SENTIMENT_WEIGHT * sentiment_pct

        results[f"final_{h}_pct"]     = round(float(final), 4)
        results[f"sentiment_pct_{h}"] = round(float(sentiment_pct), 4)

    return results


# ─────────────────────────────────────────────────────────────────────
# 3. DIRECTION SIGNAL HELPER
#    Summarises agreement/disagreement between model and sentiment
# ─────────────────────────────────────────────────────────────────────

def signal_label(model_pred: float, sentiment_score: float) -> str:
    """
    Returns a conviction label based on model + sentiment alignment.

    HIGH    — both model and sentiment agree on direction (strong signal)
    MEDIUM  — sentiment is neutral (model drives the prediction)
    LOW     — model and sentiment disagree (weak/conflicting signal)
    """
    if abs(sentiment_score) < 0.1:
        return "MEDIUM"           # sentiment essentially neutral
    model_dir     = 1 if model_pred > 0 else -1
    sentiment_dir = 1 if sentiment_score > 0 else -1
    return "HIGH" if model_dir == sentiment_dir else "LOW"


# ─────────────────────────────────────────────────────────────────────
# 4. MAIN MERGE LOGIC
# ─────────────────────────────────────────────────────────────────────

def main():
    div = "=" * 64
    print(f"\n{div}")
    print("  Merge Script  —  LSTM Predictions + Sentiment")
    print(f"  Run date : {TODAY}")
    print(f"  Formula  : 0.80 × model_blend + 0.20 × sentiment")
    print(f"{div}\n")

    # ── Load LSTM predictions ─────────────────────────────────────
    if not os.path.exists(LSTM_CSV):
        print(f"ERROR: {LSTM_CSV} not found.")
        print(f"Run lstm_nifty100_pipeline_v2.py first.")
        return

    lstm_df = pd.read_csv(LSTM_CSV)
    print(f"Loaded LSTM predictions : {len(lstm_df)} rows  ({LSTM_CSV})")

    # Filter to today's predictions only
    lstm_today = lstm_df[lstm_df["Date"] == TODAY].copy()

    if lstm_today.empty:
        print(f"WARNING: No LSTM predictions found for today ({TODAY}).")
        print(f"Using all rows in the file instead.")
        lstm_today = lstm_df.copy()

    print(f"LSTM rows for today     : {len(lstm_today)}")

    # ── Load sentiment scores ─────────────────────────────────────
    if not os.path.exists(SENTIMENT_CSV):
        print(f"\nWARNING: {SENTIMENT_CSV} not found.")
        print(f"Proceeding without sentiment — final predictions = model predictions.")
        sentiment_today = pd.DataFrame(columns=["Ticker", "sentiment_score"])
    else:
        sentiment_df    = pd.read_csv(SENTIMENT_CSV)
        sentiment_today = sentiment_df[sentiment_df["Date"] == TODAY].copy()

        if sentiment_today.empty:
            print(f"WARNING: No sentiment data for today ({TODAY}). Using all rows.")
            sentiment_today = sentiment_df.copy()

        print(f"Sentiment rows for today: {len(sentiment_today)}")

    # ── Build sentiment lookup dict  {ticker: score} ─────────────
    sentiment_map = {}
    if not sentiment_today.empty:
        for _, row in sentiment_today.iterrows():
            sentiment_map[row["Ticker"]] = float(row.get("sentiment_score", 0.0))

    # ── Apply blend formula per stock ─────────────────────────────
    print(f"\n{'─'*64}")
    print(f"  {'Ticker':<20} {'Sent':>6}  {'1d Model':>9} {'1d Final':>9}  {'Signal'}")
    print(f"{'─'*64}")

    output_rows = []

    for _, lstm_row in lstm_today.iterrows():
        ticker          = lstm_row["Ticker"]
        sentiment_score = sentiment_map.get(ticker, 0.0)  # 0.0 if no news

        # Compute blended final predictions
        blended = compute_final_predictions(lstm_row, sentiment_score)

        # Conviction signal for 1-day prediction
        signal = signal_label(lstm_row.get("pred_1d_pct", 0.0), sentiment_score)

        print(
            f"  {ticker:<20} "
            f"{sentiment_score:>+6.3f}  "
            f"{lstm_row.get('pred_1d_pct', 0.0):>+8.2f}% "
            f"{blended['final_1d_pct']:>+8.2f}%  "
            f"{signal}"
        )

        # Build the full output row
        output_row = {
            "Ticker"         : ticker,
            "Date"           : TODAY,
            # Original model blend (80% component)
            "model_1d_pct"   : lstm_row.get("pred_1d_pct",  0.0),
            "model_1w_pct"   : lstm_row.get("pred_1w_pct",  0.0),
            "model_1m_pct"   : lstm_row.get("pred_1m_pct",  0.0),
            # Sentiment info (20% component)
            "sentiment_score": sentiment_score,
            "num_articles"   : sentiment_today[
                sentiment_today["Ticker"] == ticker
            ]["num_articles"].values[0]
            if ticker in sentiment_map and not sentiment_today.empty
            else 0,
            # Scaled sentiment contributions
            "sentiment_pct_1d": blended["sentiment_pct_1d"],
            "sentiment_pct_1w": blended["sentiment_pct_1w"],
            "sentiment_pct_1m": blended["sentiment_pct_1m"],
            # Final blended predictions (the main output)
            "final_1d_pct"   : blended["final_1d_pct"],
            "final_1w_pct"   : blended["final_1w_pct"],
            "final_1m_pct"   : blended["final_1m_pct"],
            # Conviction signals
            "signal_1d"      : signal_label(lstm_row.get("pred_1d_pct", 0.0), sentiment_score),
            "signal_1w"      : signal_label(lstm_row.get("pred_1w_pct", 0.0), sentiment_score),
            "signal_1m"      : signal_label(lstm_row.get("pred_1m_pct", 0.0), sentiment_score),
            # Pass-through accuracy metrics from LSTM pipeline
            "dir_acc_1d_pct" : lstm_row.get("dir_acc_1d_pct", None),
            "dir_acc_1w_pct" : lstm_row.get("dir_acc_1w_pct", None),
            "dir_acc_1m_pct" : lstm_row.get("dir_acc_1m_pct", None),
            "r2_1d"          : lstm_row.get("r2_1d", None),
            "r2_1w"          : lstm_row.get("r2_1w", None),
            "r2_1m"          : lstm_row.get("r2_1m", None),
        }
        output_rows.append(output_row)

    # ── Save final predictions ────────────────────────────────────
    if not output_rows:
        print("\nNo output rows generated.")
        return

    final_df = pd.DataFrame(output_rows)

    if os.path.exists(OUTPUT_CSV):
        existing   = pd.read_csv(OUTPUT_CSV)
        final_df   = pd.concat([existing, final_df], ignore_index=True)
        print(f"\n  Appended to existing {OUTPUT_CSV}")
    else:
        print(f"\n  Created new {OUTPUT_CSV}")

    final_df.to_csv(OUTPUT_CSV, index=False)

    # ── Summary ───────────────────────────────────────────────────
    today_out = final_df[final_df["Date"] == TODAY]

    high_conviction = today_out[today_out["signal_1d"] == "HIGH"]
    conflicting     = today_out[today_out["signal_1d"] == "LOW"]

    print(f"\n{div}")
    print(f"  Merge complete  |  {len(output_rows)} stocks processed")
    print(f"  Saved to        : {os.path.abspath(OUTPUT_CSV)}")
    print(f"\n  Signal breakdown (1-day):")
    print(f"    HIGH (model + sentiment agree) : {len(high_conviction)} stocks")
    print(f"    MEDIUM (neutral sentiment)     : {len(today_out[today_out['signal_1d']=='MEDIUM'])} stocks")
    print(f"    LOW  (conflicting signals)     : {len(conflicting)} stocks")

    if not high_conviction.empty:
        print(f"\n  Top HIGH conviction picks (1-day):")
        top = high_conviction.nlargest(5, "final_1d_pct")[
            ["Ticker", "model_1d_pct", "sentiment_score", "final_1d_pct", "dir_acc_1d_pct"]
        ]
        print(top.to_string(index=False))

    print(f"\n  Output columns in {OUTPUT_CSV}:")
    print(f"    Ticker | Date | model_Xd_pct | sentiment_score | num_articles")
    print(f"    sentiment_pct_Xd | final_Xd_pct | signal_Xd | dir_acc_Xd | r2_Xd")
    print(f"{div}\n")


if __name__ == "__main__":
    main()

#Version 4

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║   Stock Prediction Pipeline v4  —  Nifty 100  (NSE India)          ║
║                                                                      ║
║  Architecture:                                                       ║
║    1d → XGBoost classifier   seq=15d  ±0.5% threshold              ║
║         output: UP / FLAT / DOWN  + probability %                   ║
║    1w → LSTM + XGB classifier seq=30d  ±1.0% threshold             ║
║         output: UP / FLAT / DOWN  + probability %                   ║
║    1m → Multi-output LSTM + XGB regressor  seq=60d                  ║
║         output: % return  (restored from v2 — was best performer)   ║
║                                                                      ║
║  Key design decisions:                                               ║
║    • 1d/1w optimise for direction directly (classification loss)     ║
║    • 1m keeps regression — magnitude matters at monthly horizon      ║
║    • v2 multi-output LSTM backbone restored for 1m                  ║
║    • Walk-forward validation: F1 + per-class acc for classifiers     ║
╚══════════════════════════════════════════════════════════════════════╝

Requirements:
    pip install yfinance ta scikit-learn xgboost tensorflow pandas numpy
"""

import os
import warnings
import logging
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    f1_score, accuracy_score,
    mean_absolute_error, mean_squared_error, r2_score,
)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
import ta
import gc
from tensorflow.keras.backend import clear_session
import tensorflow as tf


warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Prevent TensorFlow from instantly grabbing 100% of GPU memory
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────

SEQ_LEN = {"1d": 15, "1w": 30, "1m": 60}

# Classification thresholds — return outside ±thresh → UP or DOWN
# return inside ±thresh → FLAT
CLF_THRESH = {"1d": 0.5, "1w": 1.0}   # percent

HORIZONS   = {"1d": 1, "1w": 5, "1m": 21}

EPOCHS     = 60
BATCH_SIZE = 32
WF_FOLDS   = 5

OUTPUT_CSV = "nifty100_predictions_v4.csv"
LOG_FILE   = "pipeline_v4_errors.log"

END_DATE   = datetime.today()
START_DATE = END_DATE - timedelta(days=365 * 3)   # 3 years

# Class labels — fixed order kept consistent everywhere
CLASSES    = ["DOWN", "FLAT", "UP"]   # indices 0, 1, 2

# ─────────────────────────────────────────────────────────────────────
# 2. NIFTY 100 TICKERS
# ─────────────────────────────────────────────────────────────────────
_NIFTY100_SYMBOLS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "SBIN", "BAJFINANCE", "BHARTIARTL", "KOTAKBANK",
    "LT", "HCLTECH", "ASIANPAINT", "AXISBANK", "MARUTI",
    "SUNPHARMA", "TITAN", "ULTRACEMCO", "WIPRO", "NESTLEIND",
    "POWERGRID", "NTPC", "TECHM", "INDUSINDBK", "M&M",
    "ONGC", "TATAMOTORS", "BAJAJFINSV", "ADANIPORTS", "COALINDIA",
    "JSWSTEEL", "TATASTEEL", "HINDALCO", "DRREDDY", "CIPLA",
    "DIVISLAB", "APOLLOHOSP", "EICHERMOT", "BPCL", "GRASIM",
    "BRITANNIA", "SBILIFE", "HDFCLIFE", "BAJAJ-AUTO", "SHREECEM",
    "HEROMOTOCO", "UPL", "TATACONSUM", "ADANIENT", "VEDL",
    "CHOLAFIN", "DLF", "SIEMENS", "ABB", "PIDILITIND",
    "HAVELLS", "MUTHOOTFIN", "BANKBARODA", "PNB", "CANBK",
    "UNIONBANK", "IDFCFIRSTB", "FEDERALBNK", "RBLBANK", "BANDHANBNK",
    "BERGEPAINT", "GODREJCP", "COLPAL", "DABUR", "EMAMILTD",
    "MARICO", "TRENT", "NYKAA", "ZOMATO", "DMART",
    "IRCTC", "POLYCAB", "AMBUJACEM", "ACC", "INDIGO",
    "ADANIGREEN", "TATAPOWER", "TORNTPOWER", "NHPC", "SJVN",
    "RECLTD", "PFC", "IRFC", "SAIL", "NMDC",
    "NATIONALUM", "HINDCOPPER", "MPHASIS", "LTIM", "PERSISTENT",
    "COFORGE", "OFSS", "KPITTECH", "GMRINFRA", "CUMMINSIND",
]
NIFTY_100_TICKERS = [f"{s}.NS" for s in _NIFTY100_SYMBOLS]

# ─────────────────────────────────────────────────────────────────────
# 3. LOGGING
# ─────────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE, level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ─────────────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────

BASE_FEATURES = [
    "Close", "Volume",
    "RSI", "MACD", "MACD_signal", "MACD_diff",
    "BB_upper", "BB_lower", "BB_pct",
    "EMA_9", "EMA_21", "EMA_50",
    "ATR", "OBV", "Return",
]

SHORT_TERM_FEATURES = [
    "Gap_pct", "HL_range_pct", "Vol_surge",
    "Price_streak", "Stoch_k", "Williams_R",
    "Return_3d", "Return_5d",
]

FEATURES = {
    "1d": BASE_FEATURES + SHORT_TERM_FEATURES,
    "1w": BASE_FEATURES + SHORT_TERM_FEATURES,
    "1m": BASE_FEATURES,
}


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close = df["Close"]
    high  = df["High"]
    low   = df["Low"]
    vol   = df["Volume"]
    open_ = df["Open"]

    # Base
    df["RSI"]         = ta.momentum.RSIIndicator(close, 14).rsi()
    macd              = ta.trend.MACD(close)
    df["MACD"]        = macd.macd()
    df["MACD_signal"] = macd.macd_signal()
    df["MACD_diff"]   = macd.macd_diff()
    bb                = ta.volatility.BollingerBands(close, 20)
    df["BB_upper"]    = bb.bollinger_hband()
    df["BB_lower"]    = bb.bollinger_lband()
    df["BB_pct"]      = bb.bollinger_pband()
    df["EMA_9"]       = ta.trend.EMAIndicator(close, 9).ema_indicator()
    df["EMA_21"]      = ta.trend.EMAIndicator(close, 21).ema_indicator()
    df["EMA_50"]      = ta.trend.EMAIndicator(close, 50).ema_indicator()
    df["ATR"]         = ta.volatility.AverageTrueRange(
                            high, low, close, 14).average_true_range()
    df["OBV"]         = ta.volume.OnBalanceVolumeIndicator(
                            close, vol).on_balance_volume()
    df["Return"]      = close.pct_change()

    # Short-term
    df["Gap_pct"]      = (open_ - close.shift(1)) / close.shift(1)
    df["HL_range_pct"] = (high - low) / close
    vol_avg            = vol.rolling(10).mean()
    df["Vol_surge"]    = vol / vol_avg.replace(0, np.nan)

    direction = np.sign(df["Return"])
    streak, count = [], 0
    for d in direction:
        if np.isnan(d):
            streak.append(np.nan); count = 0
        elif d > 0:
            count = count + 1 if count > 0 else 1
            streak.append(min(count, 5))
        elif d < 0:
            count = count - 1 if count < 0 else -1
            streak.append(max(count, -5))
        else:
            count = 0; streak.append(0)
    df["Price_streak"] = streak

    df["Stoch_k"]   = ta.momentum.StochasticOscillator(
                          high, low, close, 14).stoch()
    df["Williams_R"] = ta.momentum.WilliamsRIndicator(
                          high, low, close, 14).williams_r()
    df["Return_3d"] = close.pct_change(3)
    df["Return_5d"] = close.pct_change(5)

    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────────────────────────────
# 5. LABEL MAKERS
# ─────────────────────────────────────────────────────────────────────

def make_clf_labels(returns: np.ndarray, thresh: float) -> np.ndarray:
    """
    Convert % return array to class indices.
    0=DOWN  1=FLAT  2=UP
    """
    labels = np.ones(len(returns), dtype=int)   # default FLAT=1
    labels[returns >  thresh] = 2               # UP=2
    labels[returns < -thresh] = 0               # DOWN=0
    return labels


def make_reg_targets(close: np.ndarray,
                     features_scaled: np.ndarray,
                     seq_len: int,
                     horizon: int):
    """Build regression sequences and % return targets."""
    X, y = [], []
    for i in range(seq_len, len(features_scaled) - horizon):
        X.append(features_scaled[i - seq_len : i])
        ret = (close[i + horizon] - close[i]) / close[i] * 100
        y.append(ret)
    return np.array(X), np.array(y)


def make_clf_sequences(close: np.ndarray,
                       features_scaled: np.ndarray,
                       seq_len: int,
                       horizon: int,
                       thresh: float):
    """Build classification sequences and class label targets."""
    X, returns = [], []
    for i in range(seq_len, len(features_scaled) - horizon):
        X.append(features_scaled[i - seq_len : i])
        ret = (close[i + horizon] - close[i]) / close[i] * 100
        returns.append(ret)
    X_arr  = np.array(X)
    y_int  = make_clf_labels(np.array(returns), thresh)
    return X_arr, y_int, np.array(returns)


# ─────────────────────────────────────────────────────────────────────
# 6. XGB FEATURE EXTRACTOR
# ─────────────────────────────────────────────────────────────────────

def window_to_xgb_row(window: np.ndarray) -> np.ndarray:
    t     = np.arange(len(window))
    mean  = window.mean(axis=0)
    std   = window.std(axis=0)
    mn    = window.min(axis=0)
    mx    = window.max(axis=0)
    last  = window[-1]
    slope = np.array([np.polyfit(t, window[:, j], 1)[0]
                      for j in range(window.shape[1])])
    return np.concatenate([mean, std, mn, mx, last, slope])


def build_xgb_dataset(X_seq: np.ndarray, y: np.ndarray):
    return np.array([window_to_xgb_row(s) for s in X_seq]), y


# ─────────────────────────────────────────────────────────────────────
# 7. MODEL BUILDERS
# ─────────────────────────────────────────────────────────────────────

def build_lstm_clf(seq_len: int, n_features: int) -> Model:
    """LSTM classifier → softmax over 3 classes."""
    inp = Input(shape=(seq_len, n_features))
    x   = LSTM(64, return_sequences=True)(inp)
    x   = Dropout(0.2)(x)
    x   = LSTM(32)(x)
    x   = Dropout(0.2)(x)
    out = Dense(3, activation="softmax")(x)   # 3 classes
    model = Model(inp, out)
    model.compile(optimizer="adam",
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model


def build_lstm_reg(seq_len: int, n_features: int) -> Model:
    """
    Multi-output LSTM regressor — shared backbone, 3 heads.
    Restored from v2 — was the best performing 1m architecture.
    """
    inp    = Input(shape=(seq_len, n_features))
    x      = LSTM(128, return_sequences=True)(inp)
    x      = Dropout(0.2)(x)
    x      = LSTM(64)(x)
    x      = Dropout(0.2)(x)
    shared = Dense(32, activation="relu")(x)
    # Three output heads: 1d_reg (unused), 1w_reg (unused), 1m_reg
    # We only train the 1m head here — keeping the backbone identical to v2
    out = Dense(1, activation="linear", name="out_1m")(shared)
    model = Model(inp, out)
    model.compile(optimizer="adam", loss="mse")
    return model


def build_xgb_clf(label: str) -> xgb.XGBClassifier:
    params = {
        "1d": dict(n_estimators=400, max_depth=3, learning_rate=0.03,
                   subsample=0.7, colsample_bytree=0.7,
                   min_child_weight=5, verbosity=0,
                   objective="multi:softprob", num_class=3,
                   eval_metric="mlogloss",
                  tree_method="hist", device="cuda"),
        "1w": dict(n_estimators=300, max_depth=4, learning_rate=0.04,
                   subsample=0.8, colsample_bytree=0.8,
                   min_child_weight=3, verbosity=0,
                   objective="multi:softprob", num_class=3,
                   eval_metric="mlogloss",
                  tree_method="hist", device="cuda"),
    }
    return xgb.XGBClassifier(**params[label])


def build_xgb_reg(label: str) -> xgb.XGBRegressor:
    return xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=2, verbosity=0,
        tree_method="hist", device="cuda"
    )


# ─────────────────────────────────────────────────────────────────────
# 8. CLASSIFICATION METRICS HELPER
# ─────────────────────────────────────────────────────────────────────

def clf_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Compute classification accuracy metrics.
    dir_acc = accuracy on UP vs DOWN only (ignores FLAT rows) —
    this is the most practically meaningful metric.
    """
    f1  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    # Directional accuracy — only on rows where actual is UP or DOWN
    mask = y_true != 1   # exclude FLAT
    if mask.sum() > 0:
        dir_acc = accuracy_score(y_true[mask], y_pred[mask])
    else:
        dir_acc = 0.0

    # Per-class counts
    up_correct   = int(((y_true == 2) & (y_pred == 2)).sum())
    down_correct = int(((y_true == 0) & (y_pred == 0)).sum())
    flat_correct = int(((y_true == 1) & (y_pred == 1)).sum())
    total_up     = int((y_true == 2).sum())
    total_down   = int((y_true == 0).sum())
    total_flat   = int((y_true == 1).sum())

    return {
        "f1"        : round(float(f1), 4),
        "acc"       : round(float(acc), 4),
        "dir_acc"   : round(float(dir_acc), 4),
        "up_acc"    : round(up_correct   / total_up   if total_up   else 0, 4),
        "down_acc"  : round(down_correct / total_down if total_down else 0, 4),
        "flat_acc"  : round(flat_correct / total_flat if total_flat else 0, 4),
        "class_dist": f"UP={total_up} DOWN={total_down} FLAT={total_flat}",
    }


# ─────────────────────────────────────────────────────────────────────
# 9. WALK-FORWARD — CLASSIFIER (1d and 1w)
# ─────────────────────────────────────────────────────────────────────

def wf_validate_clf(df: pd.DataFrame, label: str,
                     horizon_days: int,
                     thresh: float,
                     n_folds: int = WF_FOLDS) -> dict:
    """
    Walk-forward validation for classification horizons (1d, 1w).
    Returns aggregated classification metrics across all OOS folds.
    """
    seq_len   = SEQ_LEN[label]
    feat_cols = FEATURES[label]
    close     = df["Close"].values
    features  = df[feat_cols].values
    scaler    = MinMaxScaler()
    f_scaled  = scaler.fit_transform(features)

    X_seq, y_int, _ = make_clf_sequences(
        close, f_scaled, seq_len, horizon_days, thresh
    )
    n_samples = len(X_seq)
    fold_size = n_samples // (n_folds + 1)

    if fold_size < 10:
        raise ValueError(
            f"Too few samples ({n_samples}) for {n_folds} folds at {label}"
        )

    all_true, all_pred = [], []
    early_stop = EarlyStopping(
        monitor="val_loss", patience=4,
        restore_best_weights=True, verbose=0,
    )

    for fold in range(n_folds):
        tr_end = fold_size * (fold + 1)
        te_end = tr_end + fold_size
        if te_end > n_samples:
            break

        X_tr, X_te = X_seq[:tr_end], X_seq[tr_end:te_end]
        y_tr, y_te = y_int[:tr_end], y_int[tr_end:te_end]

        val_split       = int(len(X_tr) * 0.85)
        X_tv, y_tv      = X_tr[:val_split], y_tr[:val_split]
        X_vl, y_vl      = X_tr[val_split:], y_tr[val_split:]
        if len(X_vl) == 0:
            X_vl, y_vl = X_tv[-5:], y_tv[-5:]

        # ── XGBoost classifier ────────────────────────────────────
        X_flat_tr, _ = build_xgb_dataset(X_tr, y_tr)
        X_flat_te, _ = build_xgb_dataset(X_te, y_te)
        xgb_m        = build_xgb_clf(label)
        xgb_m.fit(X_flat_tr, y_tr)
        xgb_proba    = xgb_m.predict_proba(X_flat_te)   # (n, 3)

        if label == "1d":
            # XGBoost only for 1d
            fold_pred = xgb_proba.argmax(axis=1)

        else:
            # LSTM + XGB blend for 1w
            y_tv_cat = to_categorical(y_tv, num_classes=3)
            y_vl_cat = to_categorical(y_vl, num_classes=3)

            lstm_m = build_lstm_clf(seq_len, len(feat_cols))
            lstm_m.fit(
                X_tv, y_tv_cat,
                validation_data=(X_vl, y_vl_cat),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=[early_stop],
                verbose=0,
            )
            lstm_proba = lstm_m.predict(X_te, verbose=0)  # (n, 3)
            # Average probability arrays → argmax
            blend_proba = (lstm_proba + xgb_proba) / 2.0
            fold_pred   = blend_proba.argmax(axis=1)

        all_true.extend(y_te.tolist())
        all_pred.extend(fold_pred.tolist())

    if len(all_true) < 5:
        raise ValueError("Walk-forward produced too few predictions")

    return clf_metrics(np.array(all_true), np.array(all_pred))


# ─────────────────────────────────────────────────────────────────────
# 10. WALK-FORWARD — REGRESSOR (1m)
# ─────────────────────────────────────────────────────────────────────

def wf_validate_reg(df: pd.DataFrame,
                     n_folds: int = WF_FOLDS) -> dict:
    """
    Walk-forward validation for 1m regression.
    Uses multi-output LSTM backbone (v2 architecture).
    """
    label     = "1m"
    seq_len   = SEQ_LEN[label]
    feat_cols = FEATURES[label]
    horizon   = HORIZONS[label]
    close     = df["Close"].values
    features  = df[feat_cols].values
    scaler    = MinMaxScaler()
    f_scaled  = scaler.fit_transform(features)

    X_seq, y_arr = make_reg_targets(close, f_scaled, seq_len, horizon)
    n_samples    = len(X_seq)
    fold_size    = n_samples // (n_folds + 1)

    if fold_size < 5:
        raise ValueError(f"Too few samples ({n_samples}) for regression folds")

    all_true, all_pred = [], []
    early_stop = EarlyStopping(
        monitor="val_loss", patience=4,
        restore_best_weights=True, verbose=0,
    )

    for fold in range(n_folds):
        tr_end = fold_size * (fold + 1)
        te_end = tr_end + fold_size
        if te_end > n_samples:
            break

        X_tr, X_te = X_seq[:tr_end], X_seq[tr_end:te_end]
        y_tr, y_te = y_arr[:tr_end], y_arr[tr_end:te_end]

        val_split    = int(len(X_tr) * 0.85)
        X_tv, y_tv   = X_tr[:val_split], y_tr[:val_split]
        X_vl, y_vl   = X_tr[val_split:], y_tr[val_split:]
        if len(X_vl) == 0:
            X_vl, y_vl = X_tv[-5:], y_tv[-5:]

        # XGBoost regressor
        X_flat_tr, _ = build_xgb_dataset(X_tr, y_tr)
        X_flat_te, _ = build_xgb_dataset(X_te, y_te)
        xgb_m        = build_xgb_reg(label)
        xgb_m.fit(X_flat_tr, y_tr)
        xgb_preds    = xgb_m.predict(X_flat_te)

        # Multi-output LSTM (v2 backbone)
        lstm_m = build_lstm_reg(seq_len, len(feat_cols))
        lstm_m.fit(
            X_tv, y_tv,
            validation_data=(X_vl, y_vl),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop],
            verbose=0,
        )
        lstm_preds = lstm_m.predict(X_te, verbose=0).flatten()
        blend      = (lstm_preds + xgb_preds) / 2.0

        all_true.extend(y_te.tolist())
        all_pred.extend(blend.tolist())

    y_true = np.array(all_true)
    y_pred = np.array(all_pred)

    return {
        "mae"    : round(float(mean_absolute_error(y_true, y_pred)), 4),
        "rmse"   : round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
        "r2"     : round(float(r2_score(y_true, y_pred)), 4),
        "dir_acc": round(float(
            np.mean(np.sign(y_pred) == np.sign(y_true)) * 100), 2),
    }


# ─────────────────────────────────────────────────────────────────────
# 11. FINAL TRAIN + PREDICT  — CLASSIFIER (1d, 1w)
# ─────────────────────────────────────────────────────────────────────

def train_predict_clf(df: pd.DataFrame, label: str,
                       horizon_days: int,
                       thresh: float) -> dict:
    """
    Train on full history, return final direction prediction + probability.
    """
    seq_len   = SEQ_LEN[label]
    feat_cols = FEATURES[label]
    close     = df["Close"].values
    features  = df[feat_cols].values
    scaler    = MinMaxScaler()
    f_scaled  = scaler.fit_transform(features)

    X_seq, y_int, _ = make_clf_sequences(
        close, f_scaled, seq_len, horizon_days, thresh
    )

    if len(X_seq) < 30:
        raise ValueError(f"Not enough sequences: {len(X_seq)}")

    split      = int(len(X_seq) * 0.85)
    X_tr, X_vl = X_seq[:split], X_seq[split:]
    y_tr, y_vl = y_int[:split], y_int[split:]
    if len(X_vl) == 0:
        X_vl, y_vl = X_tr[-5:], y_tr[-5:]

    last_seq  = f_scaled[-seq_len:].reshape(1, seq_len, len(feat_cols))
    last_flat = window_to_xgb_row(f_scaled[-seq_len:]).reshape(1, -1)
    X_flat, _ = build_xgb_dataset(X_seq, y_int)

    # XGBoost
    xgb_m     = build_xgb_clf(label)
    xgb_m.fit(X_flat, y_int)
    xgb_proba = xgb_m.predict_proba(last_flat)[0]   # (3,)

    if label == "1d":
        final_proba = xgb_proba
    else:
        # LSTM + XGB blend
        y_tr_cat = to_categorical(y_tr, 3)
        y_vl_cat = to_categorical(y_vl, 3)
        early_stop = EarlyStopping(
            monitor="val_loss", patience=6,
            restore_best_weights=True, verbose=0,
        )
        lstm_m = build_lstm_clf(seq_len, len(feat_cols))
        lstm_m.fit(
            X_tr, y_tr_cat,
            validation_data=(X_vl, y_vl_cat),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[early_stop], verbose=0,
        )
        lstm_proba  = lstm_m.predict(last_seq, verbose=0)[0]   # (3,)
        final_proba = (lstm_proba + xgb_proba) / 2.0

    # Normalise so probabilities sum to 1.0
    final_proba = final_proba / final_proba.sum()

    pred_idx   = int(final_proba.argmax())
    direction  = CLASSES[pred_idx]
    confidence = round(float(final_proba[pred_idx]) * 100, 1)

    return {
        "direction"   : direction,
        "confidence"  : confidence,
        "prob_up"     : round(float(final_proba[2]) * 100, 1),
        "prob_flat"   : round(float(final_proba[1]) * 100, 1),
        "prob_down"   : round(float(final_proba[0]) * 100, 1),
    }


# ─────────────────────────────────────────────────────────────────────
# 12. FINAL TRAIN + PREDICT  — REGRESSOR (1m)
# ─────────────────────────────────────────────────────────────────────

def train_predict_reg(df: pd.DataFrame) -> dict:
    """Train multi-output LSTM + XGB on full history, predict 1m return."""
    label     = "1m"
    seq_len   = SEQ_LEN[label]
    feat_cols = FEATURES[label]
    horizon   = HORIZONS[label]
    close     = df["Close"].values
    features  = df[feat_cols].values
    scaler    = MinMaxScaler()
    f_scaled  = scaler.fit_transform(features)

    X_seq, y_arr = make_reg_targets(close, f_scaled, seq_len, horizon)

    if len(X_seq) < 30:
        raise ValueError(f"Not enough sequences: {len(X_seq)}")

    split      = int(len(X_seq) * 0.85)
    X_tr, X_vl = X_seq[:split], X_seq[split:]
    y_tr, y_vl = y_arr[:split], y_arr[split:]
    if len(X_vl) == 0:
        X_vl, y_vl = X_tr[-5:], y_tr[-5:]

    last_seq  = f_scaled[-seq_len:].reshape(1, seq_len, len(feat_cols))
    last_flat = window_to_xgb_row(f_scaled[-seq_len:]).reshape(1, -1)

    # XGBoost
    X_flat, _ = build_xgb_dataset(X_seq, y_arr)
    xgb_m     = build_xgb_reg(label)
    xgb_m.fit(X_flat, y_arr)
    xgb_pred  = float(xgb_m.predict(last_flat)[0])

    # Multi-output LSTM (v2 backbone)
    early_stop = EarlyStopping(
        monitor="val_loss", patience=6,
        restore_best_weights=True, verbose=0,
    )
    lstm_m = build_lstm_reg(seq_len, len(feat_cols))
    lstm_m.fit(
        X_tr, y_tr,
        validation_data=(X_vl, y_vl),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[early_stop], verbose=0,
    )
    lstm_pred = float(lstm_m.predict(last_seq, verbose=0)[0][0])
    blend     = (lstm_pred + xgb_pred) / 2.0

    return {
        "pred_1m_pct" : round(blend, 4),
        "lstm_1m_pct" : round(lstm_pred, 4),
        "xgb_1m_pct"  : round(xgb_pred, 4),
    }


# ─────────────────────────────────────────────────────────────────────
# 13. PER-STOCK PIPELINE
# ─────────────────────────────────────────────────────────────────────

def run_stock(ticker: str, index: int, total: int):
    print(f"\n[{index:>3}/{total}]  {ticker}")
    print(f"  {'─'*56}")

    try:
        raw = yf.download(
            ticker, start=START_DATE, end=END_DATE,
            progress=False, auto_adjust=True,
        )
        if raw.empty:
            raise ValueError("No data returned")
        if len(raw) < SEQ_LEN["1m"] + HORIZONS["1m"] + 50:
            raise ValueError(f"Insufficient rows: {len(raw)}")

        df = add_features(raw)
        print(f"  Data: {len(raw)} raw -> {len(df)} rows")

        result = {
            "Ticker": ticker,
            "Date"  : datetime.today().strftime("%Y-%m-%d"),
        }

        # ── 1d (XGBoost classifier) ───────────────────────────────
        print(f"  [1d] XGBoost-clf  seq=15d  thresh=±0.5%  validating...",
              end="  ", flush=True)
        m1d = wf_validate_clf(df, "1d", HORIZONS["1d"],
                               CLF_THRESH["1d"])
        print(
            f"F1={m1d['f1']:.3f}  "
            f"DirAcc={m1d['dir_acc']*100:.1f}%  "
            f"UP={m1d['up_acc']*100:.0f}% "
            f"DN={m1d['down_acc']*100:.0f}%  "
            f"{m1d['class_dist']}",
            end="  ", flush=True,
        )
        p1d = train_predict_clf(df, "1d", HORIZONS["1d"], CLF_THRESH["1d"])
        print(f"→ {p1d['direction']} {p1d['confidence']}%")

        result.update({
            "pred_1d_dir"     : p1d["direction"],
            "pred_1d_conf"    : p1d["confidence"],
            "prob_1d_up"      : p1d["prob_up"],
            "prob_1d_flat"    : p1d["prob_flat"],
            "prob_1d_down"    : p1d["prob_down"],
            "f1_1d"           : m1d["f1"],
            "acc_1d"          : m1d["acc"],
            "dir_acc_1d"      : round(m1d["dir_acc"] * 100, 2),
            "up_acc_1d"       : round(m1d["up_acc"] * 100, 2),
            "down_acc_1d"     : round(m1d["down_acc"] * 100, 2),
        })

        # ── 1w (LSTM + XGB classifier) ────────────────────────────
        print(f"  [1w] LSTM+XGB-clf seq=30d  thresh=±1.0%  validating...",
              end="  ", flush=True)
        m1w = wf_validate_clf(df, "1w", HORIZONS["1w"],
                               CLF_THRESH["1w"])
        print(
            f"F1={m1w['f1']:.3f}  "
            f"DirAcc={m1w['dir_acc']*100:.1f}%  "
            f"UP={m1w['up_acc']*100:.0f}% "
            f"DN={m1w['down_acc']*100:.0f}%  "
            f"{m1w['class_dist']}",
            end="  ", flush=True,
        )
        p1w = train_predict_clf(df, "1w", HORIZONS["1w"], CLF_THRESH["1w"])
        print(f"→ {p1w['direction']} {p1w['confidence']}%")

        result.update({
            "pred_1w_dir"     : p1w["direction"],
            "pred_1w_conf"    : p1w["confidence"],
            "prob_1w_up"      : p1w["prob_up"],
            "prob_1w_flat"    : p1w["prob_flat"],
            "prob_1w_down"    : p1w["prob_down"],
            "f1_1w"           : m1w["f1"],
            "acc_1w"          : m1w["acc"],
            "dir_acc_1w"      : round(m1w["dir_acc"] * 100, 2),
            "up_acc_1w"       : round(m1w["up_acc"] * 100, 2),
            "down_acc_1w"     : round(m1w["down_acc"] * 100, 2),
        })

        # ── 1m (multi-output LSTM + XGB regressor) ────────────────
        print(f"  [1m] LSTM+XGB-reg  seq=60d  validating...",
              end="  ", flush=True)
        m1m = wf_validate_reg(df)
        print(
            f"MAE={m1m['mae']:.3f}  "
            f"RMSE={m1m['rmse']:.3f}  "
            f"R2={m1m['r2']:.3f}  "
            f"DirAcc={m1m['dir_acc']:.1f}%",
            end="  ", flush=True,
        )
        p1m = train_predict_reg(df)
        print(f"→ {p1m['pred_1m_pct']:+.2f}%")

        result.update({
            "pred_1m_pct"     : p1m["pred_1m_pct"],
            "lstm_1m_pct"     : p1m["lstm_1m_pct"],
            "xgb_1m_pct"      : p1m["xgb_1m_pct"],
            "mae_1m"          : m1m["mae"],
            "rmse_1m"         : m1m["rmse"],
            "r2_1m"           : m1m["r2"],
            "dir_acc_1m"      : m1m["dir_acc"],
        })

        return result

    except Exception as exc:
        logging.error("%s | %s", ticker, exc)
        print(f"  Skipped -> {exc}")
        return None


# ─────────────────────────────────────────────────────────────────────
# 14. MAIN
# ─────────────────────────────────────────────────────────────────────

CSV_COLUMNS = [
    "Ticker", "Date",
    # 1d classifier output
    "pred_1d_dir", "pred_1d_conf",
    "prob_1d_up", "prob_1d_flat", "prob_1d_down",
    "f1_1d", "acc_1d", "dir_acc_1d", "up_acc_1d", "down_acc_1d",
    # 1w classifier output
    "pred_1w_dir", "pred_1w_conf",
    "prob_1w_up", "prob_1w_flat", "prob_1w_down",
    "f1_1w", "acc_1w", "dir_acc_1w", "up_acc_1w", "down_acc_1w",
    # 1m regressor output
    "pred_1m_pct", "lstm_1m_pct", "xgb_1m_pct",
    "mae_1m", "rmse_1m", "r2_1m", "dir_acc_1m",
]


def main():
    div = "=" * 66
    print(f"\n{div}")
    print("  Stock Prediction Pipeline v4  |  Nifty 100  |  NSE India")
    print(f"  Run date : {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Data     : {START_DATE.strftime('%Y-%m-%d')} -> {END_DATE.strftime('%Y-%m-%d')}  (3 years)")
    print(f"  1d       : XGBoost classifier  seq=15d  thresh=±0.5%")
    print(f"  1w       : LSTM+XGB classifier seq=30d  thresh=±1.0%")
    print(f"  1m       : LSTM+XGB regressor  seq=60d  (v2 backbone)")
    print(f"  Output   : {OUTPUT_CSV}")
    print(f"{div}\n")

    success_count = 0
    skipped_count = 0

    for idx, ticker in enumerate(NIFTY_100_TICKERS, start=1):
        result = run_stock(ticker, idx, len(NIFTY_100_TICKERS))

        if result:
            row_df = pd.DataFrame([result], columns=CSV_COLUMNS)
            if os.path.exists(OUTPUT_CSV):
                row_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
            else:
                row_df.to_csv(OUTPUT_CSV, index=False)
            success_count += 1
        else:
            skipped_count += 1
        # ── THE MEMORY FLUSH ────────────────────────────────────
        # Destroy the TensorFlow graph for the previous stock
        clear_session()
        # Force Python to immediately dump unreferenced variables
        gc.collect()


    # ── Final summary ─────────────────────────────────────────────
    print(f"\n{div}")
    print(f"  Complete  |  {success_count} stocks  |  {skipped_count} skipped")

    if success_count > 0 and os.path.exists(OUTPUT_CSV):
        final_df = pd.read_csv(OUTPUT_CSV)
        today    = final_df[final_df["Date"] == END_DATE.strftime("%Y-%m-%d")]

        print(f"\n  1d classifier summary:")
        print(f"    Mean F1       : {today['f1_1d'].mean():.3f}")
        print(f"    Mean DirAcc   : {today['dir_acc_1d'].mean():.1f}%")
        up1d   = (today["pred_1d_dir"] == "UP").sum()
        dn1d   = (today["pred_1d_dir"] == "DOWN").sum()
        fl1d   = (today["pred_1d_dir"] == "FLAT").sum()
        print(f"    Predictions   : UP={up1d}  DOWN={dn1d}  FLAT={fl1d}")

        print(f"\n  1w classifier summary:")
        print(f"    Mean F1       : {today['f1_1w'].mean():.3f}")
        print(f"    Mean DirAcc   : {today['dir_acc_1w'].mean():.1f}%")
        up1w   = (today["pred_1w_dir"] == "UP").sum()
        dn1w   = (today["pred_1w_dir"] == "DOWN").sum()
        fl1w   = (today["pred_1w_dir"] == "FLAT").sum()
        print(f"    Predictions   : UP={up1w}  DOWN={dn1w}  FLAT={fl1w}")

        print(f"\n  1m regressor summary:")
        print(f"    Mean DirAcc   : {today['dir_acc_1m'].mean():.1f}%")
        print(f"    Mean R²       : {today['r2_1m'].mean():.3f}")
        print(f"    Mean MAE      : {today['mae_1m'].mean():.3f}")

        print(f"\n  Top 5 highest-confidence 1d UP calls:")
        top = (today[today["pred_1d_dir"] == "UP"]
               .nlargest(5, "pred_1d_conf")
               [["Ticker", "pred_1d_dir", "pred_1d_conf",
                 "prob_1d_up", "dir_acc_1d"]])
        if not top.empty:
            print(top.to_string(index=False))

        print(f"\n  Saved to : {os.path.abspath(OUTPUT_CSV)}")
    print(f"\n{div}\n")


if __name__ == "__main__":
    main()

- MERge final

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║     Merge Script v2  —  v4 Predictions + Sentiment Scores           ║
║                                                                      ║
║  Reads:                                                              ║
║    nifty100_predictions_v4.csv   (from lstm_nifty100_pipeline_v4.py)║
║    sentiment_scores.csv          (from sentiment_pipeline.py)        ║
║                                                                      ║
║  Blending logic:                                                     ║
║    1d/1w (classification):                                           ║
║      sentiment → probability vector                                  ║
║      final_proba = 0.80 × model_proba + 0.20 × sentiment_proba      ║
║      → renormalise → argmax → final direction + confidence           ║
║                                                                      ║
║    1m (regression):                                                  ║
║      sentiment_pct  = sentiment_score × |pred_1m_pct|               ║
║      final_1m       = 0.80 × pred_1m_pct + 0.20 × sentiment_pct    ║
║                                                                      ║
║  Output:                                                             ║
║    final_predictions.csv                                            ║
╚══════════════════════════════════════════════════════════════════════╝

Run order:
    1. python lstm_nifty100_pipeline_v4.py  →  nifty100_predictions_v4.csv
    2. python sentiment_pipeline.py          →  sentiment_scores.csv
    3. python merge_final_predictions.py     →  final_predictions.csv
"""

import os
import numpy as np
import pandas as pd
from datetime import datetime

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────
V4_CSV         = "nifty100_predictions_v4.csv"
SENTIMENT_CSV  = "sentiment_scores.csv"
OUTPUT_CSV     = "final_predictions.csv"

MODEL_WEIGHT   = 0.80
SENT_WEIGHT    = 0.20

TODAY          = datetime.today().strftime("%Y-%m-%d")

# ─────────────────────────────────────────────────────────────────────
# 2. SENTIMENT → PROBABILITY VECTOR
#
#   Converts a scalar sentiment score (-1 to +1) into a 3-class
#   probability vector [P(DOWN), P(FLAT), P(UP)] that can be
#   blended with the model's output probability array.
#
#   Design:
#     score = +1.0  →  [0.05, 0.10, 0.85]  very bullish
#     score = +0.5  →  [0.15, 0.20, 0.65]  mildly bullish
#     score =  0.0  →  [0.33, 0.34, 0.33]  neutral (uniform)
#     score = -0.5  →  [0.65, 0.20, 0.15]  mildly bearish
#     score = -1.0  →  [0.85, 0.10, 0.05]  very bearish
#
#   Formula:
#     base     = [0.33, 0.34, 0.33]
#     shift    = score × max_shift       (max_shift = 0.52)
#     P(UP)    = base[2] + shift
#     P(DOWN)  = base[0] - shift
#     P(FLAT)  = 1 - P(UP) - P(DOWN)    (remainder)
#     clip and renormalise to [0,1] summing to 1.0
# ─────────────────────────────────────────────────────────────────────

MAX_SHIFT = 0.52   # how far sentiment can push probabilities from base


def sentiment_to_proba(score: float) -> np.ndarray:
    """
    Convert sentiment score → [P(DOWN), P(FLAT), P(UP)] probability vector.
    """
    score    = float(np.clip(score, -1.0, 1.0))
    shift    = score * MAX_SHIFT

    p_up     = 0.33 + shift
    p_down   = 0.33 - shift
    p_flat   = 1.0 - p_up - p_down

    proba    = np.array([p_down, p_flat, p_up])
    proba    = np.clip(proba, 0.01, 1.0)       # no zero probabilities
    proba    = proba / proba.sum()              # renormalise to sum=1
    return proba


# ─────────────────────────────────────────────────────────────────────
# 3. SIGNAL CONVICTION
# ─────────────────────────────────────────────────────────────────────

CLASSES = ["DOWN", "FLAT", "UP"]


def conviction_signal(model_dir: str,
                       final_dir: str,
                       sentiment_score: float,
                       final_conf: float) -> str:
    """
    BOOST  — sentiment strongly amplifies model direction (|score| > 0.6, same dir)
    HIGH   — model + sentiment agree on direction
    MEDIUM — sentiment neutral (|score| < 0.1) or FLAT prediction
    LOW    — model and sentiment conflict
    """
    if abs(sentiment_score) < 0.1:
        return "MEDIUM"
    if final_dir == "FLAT":
        return "MEDIUM"

    sent_dir = "UP" if sentiment_score > 0 else "DOWN"

    if sent_dir == model_dir:
        return "BOOST" if abs(sentiment_score) > 0.6 else "HIGH"
    else:
        return "LOW"


# ─────────────────────────────────────────────────────────────────────
# 4. BLEND CLASSIFICATION ROW  (1d or 1w)
# ─────────────────────────────────────────────────────────────────────

def blend_clf(model_row: pd.Series,
               horizon: str,
               sentiment_score: float) -> dict:
    """
    Blend model probability array with sentiment probability vector.
    Returns final direction, confidence, and blended probabilities.

    horizon: "1d" or "1w"
    """
    # Extract model probabilities
    model_proba = np.array([
        float(model_row.get(f"prob_{horizon}_down", 33.3)) / 100,
        float(model_row.get(f"prob_{horizon}_flat",  33.4)) / 100,
        float(model_row.get(f"prob_{horizon}_up",   33.3)) / 100,
    ])
    model_proba = model_proba / model_proba.sum()   # ensure sums to 1

    # Sentiment probability vector
    sent_proba  = sentiment_to_proba(sentiment_score)

    # Weighted blend
    final_proba = MODEL_WEIGHT * model_proba + SENT_WEIGHT * sent_proba
    final_proba = final_proba / final_proba.sum()

    # Final prediction
    pred_idx   = int(final_proba.argmax())
    direction  = CLASSES[pred_idx]
    confidence = round(float(final_proba[pred_idx]) * 100, 1)

    # Original model direction (before sentiment)
    model_dir  = str(model_row.get(f"pred_{horizon}_dir", "FLAT"))

    # Conviction
    signal = conviction_signal(
        model_dir, direction, sentiment_score, confidence
    )

    return {
        f"final_{horizon}_dir"    : direction,
        f"final_{horizon}_conf"   : confidence,
        f"final_{horizon}_prob_up"  : round(float(final_proba[2]) * 100, 1),
        f"final_{horizon}_prob_flat": round(float(final_proba[1]) * 100, 1),
        f"final_{horizon}_prob_down": round(float(final_proba[0]) * 100, 1),
        f"model_{horizon}_dir"    : model_dir,
        f"signal_{horizon}"       : signal,
        # How much sentiment shifted the prediction
        f"sent_shift_{horizon}"   : round(
            float(final_proba[2] - model_proba[2]) * 100, 1
        ),
    }


# ─────────────────────────────────────────────────────────────────────
# 5. BLEND REGRESSION ROW  (1m)
# ─────────────────────────────────────────────────────────────────────

def blend_reg(model_row: pd.Series,
               sentiment_score: float) -> dict:
    """
    Blend 1m regression prediction with sentiment.
    Same formula as original merge script.
    """
    pred        = float(model_row.get("pred_1m_pct", 0.0))
    sent_pct    = sentiment_score * abs(pred)
    final       = MODEL_WEIGHT * pred + SENT_WEIGHT * sent_pct
    model_dir   = "UP" if pred > 0 else "DOWN"

    signal = conviction_signal(
        model_dir,
        "UP" if final > 0 else "DOWN",
        sentiment_score,
        abs(final),
    )

    return {
        "final_1m_pct"     : round(final, 4),
        "model_1m_pct"     : round(pred, 4),
        "sentiment_pct_1m" : round(sent_pct, 4),
        "signal_1m"        : signal,
    }


# ─────────────────────────────────────────────────────────────────────
# 6. MAIN MERGE LOGIC
# ─────────────────────────────────────────────────────────────────────

def main():
    div = "=" * 66
    print(f"\n{div}")
    print("  Merge Script v2  —  v4 Predictions + Sentiment")
    print(f"  Run date : {TODAY}")
    print(f"  Blend    : 80% model  +  20% sentiment")
    print(f"  1d/1w    : probability-level blending → direction + confidence")
    print(f"  1m       : weighted % return blending")
    print(f"{div}\n")

    # ── Load v4 predictions ───────────────────────────────────────
    if not os.path.exists(V4_CSV):
        print(f"ERROR: {V4_CSV} not found. Run lstm_nifty100_pipeline_v4.py first.")
        return

    v4_df    = pd.read_csv(V4_CSV)
    v4_today = v4_df[v4_df["Date"] == TODAY].copy()
    if v4_today.empty:
        print(f"No v4 rows for {TODAY} — using latest available rows.")
        v4_today = v4_df.copy()
    print(f"v4 predictions loaded  : {len(v4_today)} stocks")

    # ── Load sentiment scores ─────────────────────────────────────
    sent_map = {}
    art_map  = {}
    if os.path.exists(SENTIMENT_CSV):
        sent_df    = pd.read_csv(SENTIMENT_CSV)
        sent_today = sent_df[sent_df["Date"] == TODAY]
        if sent_today.empty:
            sent_today = sent_df.copy()
        for _, r in sent_today.iterrows():
            sent_map[r["Ticker"]] = float(r.get("sentiment_score", 0.0))
            art_map[r["Ticker"]]  = int(r.get("num_articles", 0))
        print(f"Sentiment loaded       : {len(sent_map)} stocks")
        stocks_with_news = sum(1 for v in art_map.values() if v > 0)
        print(f"Stocks with news       : {stocks_with_news}/{len(sent_map)}")
    else:
        print(f"WARNING: {SENTIMENT_CSV} not found — sentiment = 0 for all stocks.")

    print(f"\n{'─'*66}")
    print(f"  {'Ticker':<20} {'Sent':>6}  "
          f"{'1d Model':>9} {'1d Final':>9}  "
          f"{'1w Model':>9} {'1w Final':>9}  "
          f"{'1m Final':>9}  Signal")
    print(f"{'─'*66}")

    output_rows = []

    for _, row in v4_today.iterrows():
        ticker  = row["Ticker"]
        score   = sent_map.get(ticker, 0.0)
        n_arts  = art_map.get(ticker, 0)

        # Apply minimum article threshold —
        # single-article sentiment scores are unreliable (FinBERT overconfident)
        if n_arts < 3:
            score = 0.0

        # Blend 1d
        b1d = blend_clf(row, "1d", score)

        # Blend 1w
        b1w = blend_clf(row, "1w", score)

        # Blend 1m
        b1m = blend_reg(row, score)

        # Combine into output row
        out = {
            "Ticker"          : ticker,
            "Date"            : TODAY,
            "sentiment_score" : score,
            "num_articles"    : n_arts,
            **b1d, **b1w, **b1m,
            # Pass through model accuracy metrics
            "f1_1d"           : row.get("f1_1d"),
            "dir_acc_1d"      : row.get("dir_acc_1d"),
            "up_acc_1d"       : row.get("up_acc_1d"),
            "down_acc_1d"     : row.get("down_acc_1d"),
            "f1_1w"           : row.get("f1_1w"),
            "dir_acc_1w"      : row.get("dir_acc_1w"),
            "up_acc_1w"       : row.get("up_acc_1w"),
            "down_acc_1w"     : row.get("down_acc_1w"),
            "r2_1m"           : row.get("r2_1m"),
            "dir_acc_1m"      : row.get("dir_acc_1m"),
            "mae_1m"          : row.get("mae_1m"),
        }
        output_rows.append(out)

        print(
            f"  {ticker:<20} "
            f"{score:>+6.3f}  "
            f"{str(row.get('pred_1d_dir','?')):>9} "
            f"{b1d['final_1d_dir']:>9}  "
            f"{str(row.get('pred_1w_dir','?')):>9} "
            f"{b1w['final_1w_dir']:>9}  "
            f"{b1m['final_1m_pct']:>+8.2f}%  "
            f"{b1d['signal_1d']}"
        )

    # ── Save ──────────────────────────────────────────────────────
    if not output_rows:
        print("\nNo output rows generated.")
        return

    out_df = pd.DataFrame(output_rows)
    if os.path.exists(OUTPUT_CSV):
        existing = pd.read_csv(OUTPUT_CSV)
        out_df   = pd.concat([existing, out_df], ignore_index=True)
        print(f"\n  Appended to {OUTPUT_CSV}")
    else:
        print(f"\n  Created {OUTPUT_CSV}")

    out_df.to_csv(OUTPUT_CSV, index=False)

    # ── Summary ───────────────────────────────────────────────────
    today_out = out_df[out_df["Date"] == TODAY]

    boost  = (today_out["signal_1d"] == "BOOST").sum()
    high   = (today_out["signal_1d"] == "HIGH").sum()
    medium = (today_out["signal_1d"] == "MEDIUM").sum()
    low    = (today_out["signal_1d"] == "LOW").sum()

    up1d   = (today_out["final_1d_dir"] == "UP").sum()
    dn1d   = (today_out["final_1d_dir"] == "DOWN").sum()
    fl1d   = (today_out["final_1d_dir"] == "FLAT").sum()
    up1w   = (today_out["final_1w_dir"] == "UP").sum()
    dn1w   = (today_out["final_1w_dir"] == "DOWN").sum()
    fl1w   = (today_out["final_1w_dir"] == "FLAT").sum()

    print(f"\n{div}")
    print(f"  Merge complete  |  {len(output_rows)} stocks")
    print(f"  Saved to        : {os.path.abspath(OUTPUT_CSV)}")

    print(f"\n  Final predictions breakdown:")
    print(f"    1d  →  UP={up1d}  DOWN={dn1d}  FLAT={fl1d}")
    print(f"    1w  →  UP={up1w}  DOWN={dn1w}  FLAT={fl1w}")

    print(f"\n  Signal conviction (1-day):")
    print(f"    BOOST  (sentiment strongly amplifies model) : {boost}")
    print(f"    HIGH   (model + sentiment agree)            : {high}")
    print(f"    MEDIUM (neutral sentiment)                  : {medium}")
    print(f"    LOW    (conflicting signals)                : {low}")

    # Best picks — BOOST or HIGH, UP direction, good model accuracy
    best = today_out[
        (today_out["signal_1d"].isin(["BOOST", "HIGH"])) &
        (today_out["final_1d_dir"] == "UP")
    ].copy()

    if not best.empty:
        print(f"\n  Top BOOST/HIGH conviction UP picks (1-day):")
        best_sorted = best.sort_values(
            ["signal_1d", "final_1d_conf"], ascending=[True, False]
        )[["Ticker", "sentiment_score", "final_1d_dir",
           "final_1d_conf", "signal_1d", "dir_acc_1d"]].head(10)
        print(best_sorted.to_string(index=False))

    print(f"\n  How to read the output CSV:")
    print(f"    final_1d_dir    — UP / FLAT / DOWN  (after sentiment blend)")
    print(f"    final_1d_conf   — confidence %  (e.g. 71.2%)")
    print(f"    signal_1d       — BOOST / HIGH / MEDIUM / LOW")
    print(f"    sent_shift_1d   — how many % points sentiment shifted UP prob")
    print(f"    final_1m_pct    — % return forecast for 1 month")
    print(f"{div}\n")


if __name__ == "__main__":
    main()

# FINAL VERSION - V5

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║   Stock Prediction Pipeline v5  —  Nifty 100  (NSE India)          ║
║                                                                      ║
║  Architecture  (v2 backbone, 1m only):                              ║
║    • Multi-output LSTM  (shared backbone, single 1m head)           ║
║    • XGBoost regressor  (window stats → 1m return)                  ║
║    • Blend = (LSTM + XGBoost) / 2                                   ║
║    • Walk-forward validation  (5 expanding folds)                   ║
║                                                                      ║
║  RAM optimisations:                                                  ║
║    • float32 arrays throughout  (halves sequence memory)            ║
║    • clear_session() after every stock  (frees TF graph memory)     ║
║    • explicit del + gc.collect() on large arrays per fold           ║
║    • TF memory growth capped at startup                             ║
║    • XGBoost hist tree_method  (lower RAM than exact)               ║
║    • intermediate arrays deleted immediately after use              ║
╚══════════════════════════════════════════════════════════════════════╝

Requirements:
    pip install yfinance ta scikit-learn xgboost tensorflow pandas numpy
"""

import os
import gc
import warnings
import logging
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── TensorFlow memory config — must happen BEFORE any tf import ──────
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf

# Prevent TF from grabbing all available RAM at startup
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Limit CPU-side TF memory as well
try:
    tf.config.set_logical_device_configuration(
        tf.config.list_physical_devices("CPU")[0],
        [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]  # 2 GB cap
    )
except Exception:
    pass  # non-critical — continue without limit if it fails

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K
import ta

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────
SEQ_LEN    = 60
HORIZON    = 21          # 1 month = 21 trading days
EPOCHS     = 60
BATCH_SIZE = 32
WF_FOLDS   = 5

OUTPUT_CSV = "nifty100_predictions_v5.csv"
LOG_FILE   = "pipeline_v5_errors.log"

END_DATE   = datetime.today()
START_DATE = END_DATE - timedelta(days=365 * 3)

CONF_SCALE = 2.0

# ─────────────────────────────────────────────────────────────────────
# 2. NIFTY 100 TICKERS
# ─────────────────────────────────────────────────────────────────────
_NIFTY100_SYMBOLS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "SBIN", "BAJFINANCE", "BHARTIARTL", "KOTAKBANK",
    "LT", "HCLTECH", "ASIANPAINT", "AXISBANK", "MARUTI",
    "SUNPHARMA", "TITAN", "ULTRACEMCO", "WIPRO", "NESTLEIND",
    "POWERGRID", "NTPC", "TECHM", "INDUSINDBK", "M&M",
    "ONGC", "TATAMOTORS", "BAJAJFINSV", "ADANIPORTS", "COALINDIA",
    "JSWSTEEL", "TATASTEEL", "HINDALCO", "DRREDDY", "CIPLA",
    "DIVISLAB", "APOLLOHOSP", "EICHERMOT", "BPCL", "GRASIM",
    "BRITANNIA", "SBILIFE", "HDFCLIFE", "BAJAJ-AUTO", "SHREECEM",
    "HEROMOTOCO", "UPL", "TATACONSUM", "ADANIENT", "VEDL",
    "CHOLAFIN", "DLF", "SIEMENS", "ABB", "PIDILITIND",
    "HAVELLS", "MUTHOOTFIN", "BANKBARODA", "PNB", "CANBK",
    "UNIONBANK", "IDFCFIRSTB", "FEDERALBNK", "RBLBANK", "BANDHANBNK",
    "BERGEPAINT", "GODREJCP", "COLPAL", "DABUR", "EMAMILTD",
    "MARICO", "TRENT", "NYKAA", "ZOMATO", "DMART",
    "IRCTC", "POLYCAB", "AMBUJACEM", "ACC", "INDIGO",
    "ADANIGREEN", "TATAPOWER", "TORNTPOWER", "NHPC", "SJVN",
    "RECLTD", "PFC", "IRFC", "SAIL", "NMDC",
    "NATIONALUM", "HINDCOPPER", "MPHASIS", "LTIM", "PERSISTENT",
    "COFORGE", "OFSS", "KPITTECH", "GMRINFRA", "CUMMINSIND",
]
NIFTY_100_TICKERS = [f"{s}.NS" for s in _NIFTY100_SYMBOLS]

# ─────────────────────────────────────────────────────────────────────
# 3. LOGGING
# ─────────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE, level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ─────────────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "Close", "Volume",
    "RSI", "MACD", "MACD_signal", "MACD_diff",
    "BB_upper", "BB_lower", "BB_pct",
    "EMA_9", "EMA_21", "EMA_50",
    "ATR", "OBV", "Return",
]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close = df["Close"]
    high  = df["High"]
    low   = df["Low"]
    vol   = df["Volume"]

    df["RSI"]         = ta.momentum.RSIIndicator(close, 14).rsi()
    macd              = ta.trend.MACD(close)
    df["MACD"]        = macd.macd()
    df["MACD_signal"] = macd.macd_signal()
    df["MACD_diff"]   = macd.macd_diff()
    bb                = ta.volatility.BollingerBands(close, 20)
    df["BB_upper"]    = bb.bollinger_hband()
    df["BB_lower"]    = bb.bollinger_lband()
    df["BB_pct"]      = bb.bollinger_pband()
    df["EMA_9"]       = ta.trend.EMAIndicator(close, 9).ema_indicator()
    df["EMA_21"]      = ta.trend.EMAIndicator(close, 21).ema_indicator()
    df["EMA_50"]      = ta.trend.EMAIndicator(close, 50).ema_indicator()
    df["ATR"]         = ta.volatility.AverageTrueRange(
                            high, low, close, 14).average_true_range()
    df["OBV"]         = ta.volume.OnBalanceVolumeIndicator(
                            close, vol).on_balance_volume()
    df["Return"]      = close.pct_change()

    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────────────────────────────
# 5. SEQUENCE BUILDER
#    Uses float32 throughout — halves memory vs default float64
# ─────────────────────────────────────────────────────────────────────

def build_sequences(features_scaled: np.ndarray,
                    close: np.ndarray) -> tuple:
    """
    Returns float32 arrays — halves RAM vs float64.
    X shape: (n, SEQ_LEN, n_features)  dtype=float32
    y shape: (n,)                       dtype=float32
    """
    X, y = [], []
    for i in range(SEQ_LEN, len(features_scaled) - HORIZON):
        X.append(features_scaled[i - SEQ_LEN : i])
        ret = (close[i + HORIZON] - close[i]) / close[i] * 100
        y.append(ret)

    # Cast to float32 immediately — do not create float64 intermediates
    X_arr = np.array(X, dtype=np.float32)
    y_arr = np.array(y, dtype=np.float32)
    return X_arr, y_arr


# ─────────────────────────────────────────────────────────────────────
# 6. XGB FEATURE EXTRACTOR
# ─────────────────────────────────────────────────────────────────────

def window_to_xgb_row(window: np.ndarray) -> np.ndarray:
    """Compress (SEQ_LEN, n_features) → 1D float32 vector."""
    t     = np.arange(len(window), dtype=np.float32)
    mean  = window.mean(axis=0)
    std   = window.std(axis=0)
    mn    = window.min(axis=0)
    mx    = window.max(axis=0)
    last  = window[-1]
    slope = np.array([
        np.polyfit(t, window[:, j], 1)[0]
        for j in range(window.shape[1])
    ], dtype=np.float32)
    return np.concatenate([mean, std, mn, mx, last, slope]).astype(np.float32)


def build_xgb_matrix(X_seq: np.ndarray) -> np.ndarray:
    """Convert sequence array to flat XGBoost feature matrix (float32)."""
    return np.array(
        [window_to_xgb_row(s) for s in X_seq],
        dtype=np.float32,
    )


# ─────────────────────────────────────────────────────────────────────
# 7. MODEL BUILDERS
# ─────────────────────────────────────────────────────────────────────

def build_lstm(seq_len: int, n_features: int) -> Model:
    """v2 multi-output backbone — single 1m head."""
    inp    = Input(shape=(seq_len, n_features))
    x      = LSTM(128, return_sequences=True)(inp)
    x      = Dropout(0.2)(x)
    x      = LSTM(64)(x)
    x      = Dropout(0.2)(x)
    shared = Dense(32, activation="relu")(x)
    out    = Dense(1, activation="linear", name="pred_1m")(shared)
    model  = Model(inp, out)
    model.compile(optimizer="adam", loss="mse")
    return model


def build_xgb() -> xgb.XGBRegressor:
    return xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        tree_method="hist",   # ← lower RAM than default "exact"
        verbosity=0,
    )


# ─────────────────────────────────────────────────────────────────────
# 8. MEMORY CLEANUP HELPER
# ─────────────────────────────────────────────────────────────────────

def free_memory(*arrays) -> None:
    """
    Delete all passed arrays and models, clear TF session, run GC.
    Called after every fold and after every stock.
    """
    for obj in arrays:
        try:
            del obj
        except Exception:
            pass
    K.clear_session()   # frees TF graph, layer weights, optimizer state
    gc.collect()


# ─────────────────────────────────────────────────────────────────────
# 9. DIRECTIONAL CONFIDENCE
# ─────────────────────────────────────────────────────────────────────

def direction_confidence(pred_pct: float, close: np.ndarray) -> tuple:
    direction = "UP" if pred_pct >= 0 else "DOWN"
    monthly_returns = [
        (close[i + HORIZON] - close[i]) / close[i] * 100
        for i in range(len(close) - HORIZON)
    ]
    recent_std = float(np.std(monthly_returns)) if monthly_returns else 3.0
    if recent_std == 0:
        recent_std = 3.0
    raw_conf   = abs(pred_pct) / (CONF_SCALE * recent_std)
    confidence = round(float(np.clip(raw_conf, 0.0, 1.0) * 100), 1)
    return direction, confidence


# ─────────────────────────────────────────────────────────────────────
# 10. WALK-FORWARD VALIDATION
#     Memory-efficient: arrays deleted after every fold
# ─────────────────────────────────────────────────────────────────────

def walk_forward_validate(df: pd.DataFrame) -> dict:
    close    = df["Close"].values.astype(np.float32)
    features = df[FEATURE_COLS].values.astype(np.float32)
    scaler   = MinMaxScaler()
    f_scaled = scaler.fit_transform(features).astype(np.float32)

    X_seq, y_arr = build_sequences(f_scaled, close)
    del f_scaled, features   # free scaled features — no longer needed
    gc.collect()

    n_samples = len(X_seq)
    fold_size = n_samples // (WF_FOLDS + 1)

    if fold_size < 5:
        raise ValueError(f"Too few samples ({n_samples}) for {WF_FOLDS} folds")

    all_true, all_pred = [], []
    early_stop = EarlyStopping(
        monitor="val_loss", patience=4,
        restore_best_weights=True, verbose=0,
    )

    for fold in range(WF_FOLDS):
        tr_end = fold_size * (fold + 1)
        te_end = tr_end + fold_size
        if te_end > n_samples:
            break

        # Slice — views not copies where possible
        X_tr, X_te = X_seq[:tr_end], X_seq[tr_end:te_end]
        y_tr, y_te = y_arr[:tr_end], y_arr[tr_end:te_end]

        val_split    = int(len(X_tr) * 0.85)
        X_tv, y_tv   = X_tr[:val_split], y_tr[:val_split]
        X_vl, y_vl   = X_tr[val_split:], y_tr[val_split:]
        if len(X_vl) == 0:
            X_vl, y_vl = X_tv[-5:], y_tv[-5:]

        # ── XGBoost ───────────────────────────────────────────────
        X_flat_tr = build_xgb_matrix(X_tr)
        X_flat_te = build_xgb_matrix(X_te)
        xgb_m     = build_xgb()
        xgb_m.fit(X_flat_tr, y_tr)
        xgb_preds = xgb_m.predict(X_flat_te)

        del X_flat_tr        # free immediately after use
        gc.collect()

        # ── LSTM ──────────────────────────────────────────────────
        lstm_m = build_lstm(SEQ_LEN, len(FEATURE_COLS))
        lstm_m.fit(
            X_tv, y_tv,
            validation_data=(X_vl, y_vl),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[early_stop], verbose=0,
        )
        lstm_preds = lstm_m.predict(X_te, verbose=0).flatten()

        blend = (lstm_preds + xgb_preds) / 2.0
        all_true.extend(y_te.tolist())
        all_pred.extend(blend.tolist())

        # ── Free fold memory ──────────────────────────────────────
        free_memory(lstm_m, xgb_m, X_flat_te,
                    lstm_preds, xgb_preds, blend)

    del X_seq, y_arr   # free full sequence arrays
    gc.collect()

    y_true = np.array(all_true, dtype=np.float32)
    y_pred = np.array(all_pred, dtype=np.float32)

    metrics = {
        "mae"    : round(float(mean_absolute_error(y_true, y_pred)), 4),
        "rmse"   : round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
        "r2"     : round(float(r2_score(y_true, y_pred)), 4),
        "dir_acc": round(float(
            np.mean(np.sign(y_pred) == np.sign(y_true)) * 100), 2),
    }

    del y_true, y_pred
    gc.collect()
    return metrics


# ─────────────────────────────────────────────────────────────────────
# 11. FINAL TRAIN + PREDICT
#     Same memory-efficient pattern
# ─────────────────────────────────────────────────────────────────────

def train_and_predict(df: pd.DataFrame) -> dict:
    close    = df["Close"].values.astype(np.float32)
    features = df[FEATURE_COLS].values.astype(np.float32)
    scaler   = MinMaxScaler()
    f_scaled = scaler.fit_transform(features).astype(np.float32)

    del features
    gc.collect()

    X_seq, y_arr = build_sequences(f_scaled, close)

    if len(X_seq) < 30:
        raise ValueError(f"Insufficient sequences: {len(X_seq)}")

    split      = int(len(X_seq) * 0.85)
    X_tr, X_vl = X_seq[:split],  X_seq[split:]
    y_tr, y_vl = y_arr[:split],  y_arr[split:]
    if len(X_vl) == 0:
        X_vl, y_vl = X_tr[-5:], y_tr[-5:]

    last_seq  = f_scaled[-SEQ_LEN:].reshape(
        1, SEQ_LEN, len(FEATURE_COLS)
    ).astype(np.float32)
    last_flat = window_to_xgb_row(
        f_scaled[-SEQ_LEN:]
    ).reshape(1, -1)

    del f_scaled
    gc.collect()

    # ── XGBoost ───────────────────────────────────────────────────
    X_flat = build_xgb_matrix(X_seq)
    xgb_m  = build_xgb()
    xgb_m.fit(X_flat, y_arr)
    xgb_pred = float(xgb_m.predict(last_flat)[0])

    del X_flat
    gc.collect()

    # ── LSTM ──────────────────────────────────────────────────────
    early_stop = EarlyStopping(
        monitor="val_loss", patience=6,
        restore_best_weights=True, verbose=0,
    )
    lstm_m = build_lstm(SEQ_LEN, len(FEATURE_COLS))
    lstm_m.fit(
        X_tr, y_tr,
        validation_data=(X_vl, y_vl),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[early_stop], verbose=0,
    )
    lstm_pred = float(lstm_m.predict(last_seq, verbose=0)[0][0])
    blend     = (lstm_pred + xgb_pred) / 2.0

    direction, confidence = direction_confidence(blend, close)

    result = {
        "pred_1m_pct" : round(blend, 4),
        "lstm_1m_pct" : round(lstm_pred, 4),
        "xgb_1m_pct"  : round(xgb_pred, 4),
        "direction"   : direction,
        "confidence"  : confidence,
    }

    # Free everything before returning
    free_memory(lstm_m, xgb_m, X_seq, y_arr,
                X_tr, X_vl, last_seq, last_flat)

    return result


# ─────────────────────────────────────────────────────────────────────
# 12. PER-STOCK PIPELINE
# ─────────────────────────────────────────────────────────────────────

def run_stock(ticker: str, index: int, total: int):
    print(f"\n[{index:>3}/{total}]  {ticker}")
    print(f"  {'─'*54}")

    try:
        raw = yf.download(
            ticker, start=START_DATE, end=END_DATE,
            progress=False, auto_adjust=True,
        )
        if raw.empty:
            raise ValueError("No data returned")
        if len(raw) < SEQ_LEN + HORIZON + 50:
            raise ValueError(f"Insufficient rows: {len(raw)}")

        df = add_features(raw)
        del raw   # free raw OHLCV — no longer needed
        gc.collect()

        print(f"  Data : {len(df)} usable rows")

        # Walk-forward validation
        print(f"  Validating ...", end="  ", flush=True)
        metrics = walk_forward_validate(df)
        print(
            f"MAE={metrics['mae']:.3f}  "
            f"RMSE={metrics['rmse']:.3f}  "
            f"R²={metrics['r2']:.3f}  "
            f"DirAcc={metrics['dir_acc']:.1f}%"
        )

        # Final prediction
        print(f"  Predicting ...", end="  ", flush=True)
        preds = train_and_predict(df)
        del df   # free feature df — done with this stock
        gc.collect()

        arrow = "↑" if preds["direction"] == "UP" else "↓"
        print(
            f"Pred={preds['pred_1m_pct']:+.2f}%  "
            f"{arrow} {preds['direction']}  "
            f"Conf={preds['confidence']:.1f}%"
        )

        return {
            "Ticker"      : ticker,
            "Date"        : datetime.today().strftime("%Y-%m-%d"),
            "pred_1m_pct" : preds["pred_1m_pct"],
            "direction"   : preds["direction"],
            "confidence"  : preds["confidence"],
            "lstm_1m_pct" : preds["lstm_1m_pct"],
            "xgb_1m_pct"  : preds["xgb_1m_pct"],
            "mae"         : metrics["mae"],
            "rmse"        : metrics["rmse"],
            "r2"          : metrics["r2"],
            "dir_acc_pct" : metrics["dir_acc"],
        }

    except Exception as exc:
        logging.error("%s | %s", ticker, exc)
        print(f"  Skipped → {exc}")
        # Still clean up on failure
        K.clear_session()
        gc.collect()
        return None


# ─────────────────────────────────────────────────────────────────────
# 13. MAIN
# ─────────────────────────────────────────────────────────────────────

CSV_COLUMNS = [
    "Ticker", "Date",
    "pred_1m_pct", "direction", "confidence",
    "lstm_1m_pct", "xgb_1m_pct",
    "mae", "rmse", "r2", "dir_acc_pct",
]


def main():
    div = "=" * 62
    print(f"\n{div}")
    print("  Stock Prediction Pipeline v5  |  Nifty 100  |  NSE India")
    print(f"  Run date  : {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Data      : {START_DATE.strftime('%Y-%m-%d')} → {END_DATE.strftime('%Y-%m-%d')}")
    print(f"  Horizon   : 1 month  (21 trading days)")
    print(f"  Model     : Multi-output LSTM + XGBoost  (v2 backbone)")
    print(f"  RAM saves : float32 arrays | clear_session per stock |")
    print(f"              hist tree_method | del arrays per fold")
    print(f"  Output    : pred_1m_pct | direction | confidence")
    print(f"  CSV       : {OUTPUT_CSV}")
    print(f"{div}\n")

    success_count = 0
    skipped_count = 0

    for idx, ticker in enumerate(NIFTY_100_TICKERS, start=1):
        result = run_stock(ticker, idx, len(NIFTY_100_TICKERS))

        if result:
            row_df = pd.DataFrame([result], columns=CSV_COLUMNS)
            if os.path.exists(OUTPUT_CSV):
                row_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
            else:
                row_df.to_csv(OUTPUT_CSV, index=False)
            success_count += 1
        else:
            skipped_count += 1

    # ── Final summary ─────────────────────────────────────────────
    print(f"\n{div}")
    print(f"  Complete   : {success_count} / {len(NIFTY_100_TICKERS)} stocks")
    print(f"  Skipped    : {skipped_count}  (see {LOG_FILE})")
    print(f"  Saved to   : {os.path.abspath(OUTPUT_CSV)}")

    if success_count > 0 and os.path.exists(OUTPUT_CSV):
        final_df = pd.read_csv(OUTPUT_CSV)
        today    = final_df[
            final_df["Date"] == END_DATE.strftime("%Y-%m-%d")
        ]

        if not today.empty:
            up_count = (today["direction"] == "UP").sum()
            dn_count = (today["direction"] == "DOWN").sum()
            print(f"\n  Direction split  : ↑ UP={up_count}  ↓ DOWN={dn_count}")
            print(f"  Avg confidence   : {today['confidence'].mean():.1f}%")
            print(f"  Avg dir. accuracy: {today['dir_acc_pct'].mean():.1f}%")
            print(f"  Avg R²           : {today['r2'].mean():.3f}")
            print(f"  R² > 0           : {(today['r2']>0).sum()} stocks")
            print(f"  DirAcc > 60%     : {(today['dir_acc_pct']>60).sum()} stocks")
            print(f"  DirAcc > 65%     : {(today['dir_acc_pct']>65).sum()} stocks")

            print(f"\n  Top 5 UP picks  (by dir. accuracy):")
            top_up = (
                today[today["direction"] == "UP"]
                .nlargest(5, "dir_acc_pct")
                [["Ticker","pred_1m_pct","direction","confidence",
                  "dir_acc_pct","r2"]]
            )
            if not top_up.empty:
                print(top_up.to_string(index=False))

            print(f"\n  Top 5 DOWN picks  (by dir. accuracy):")
            top_dn = (
                today[today["direction"] == "DOWN"]
                .nlargest(5, "dir_acc_pct")
                [["Ticker","pred_1m_pct","direction","confidence",
                  "dir_acc_pct","r2"]]
            )
            if not top_dn.empty:
                print(top_dn.to_string(index=False))

    print(f"\n{div}\n")


if __name__ == "__main__":
    main()

# FINAL SENTIMENTAL PIPELINE


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║         Sentiment Analysis Pipeline  —  Nifty 100 (NSE India)       ║
║                                                                      ║
║  News sources (in priority order):                                   ║
║    1. Google News RSS  — free, no key, best Indian coverage          ║
║    2. NewsAPI          — optional fallback, needs API key            ║
║                                                                      ║
║  Flow per stock:                                                     ║
║    Google News RSS → FinBERT scoring → weighted sentiment score      ║
║    → Append to sentiment_scores.csv                                  ║
╚══════════════════════════════════════════════════════════════════════╝

Requirements:
    pip install feedparser requests transformers torch pandas
"""

import os
import time
import logging
import warnings
from datetime import datetime, timedelta
from urllib.parse import quote_plus

import feedparser
import requests
import pandas as pd
from transformers import pipeline as hf_pipeline

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────
# NewsAPI is now OPTIONAL — Google News RSS is the primary source.
# If you have a NewsAPI key, paste it below for extra articles.
# If not, leave it as None — the pipeline works without it.
NEWSAPI_KEY     = "6b4428da9830434a99f5680356d001b4"        # e.g. "abc123..." or None

OUTPUT_CSV      = "sentiment_scores.csv"
LOG_FILE        = "sentiment_errors.log"
MAX_ARTICLES    = 15                    # max articles to score per stock
SLEEP_BETWEEN   = 0.5                   # seconds between stocks

TODAY           = datetime.today()
YESTERDAY       = TODAY - timedelta(days=3)

# ─────────────────────────────────────────────────────────────────────
# 2. TICKER → COMPANY NAME MAPPING
# ─────────────────────────────────────────────────────────────────────
COMPANY_NAMES = {
    "RELIANCE.NS"   : "Reliance Industries",
    "TCS.NS"        : "Tata Consultancy Services TCS",
    "HDFCBANK.NS"   : "HDFC Bank",
    "INFY.NS"       : "Infosys",
    "ICICIBANK.NS"  : "ICICI Bank",
    "HINDUNILVR.NS" : "Hindustan Unilever HUL",
    "SBIN.NS"       : "State Bank of India SBI",
    "BAJFINANCE.NS" : "Bajaj Finance",
    "BHARTIARTL.NS" : "Bharti Airtel",
    "KOTAKBANK.NS"  : "Kotak Mahindra Bank",
    "LT.NS"         : "Larsen Toubro L&T",
    "HCLTECH.NS"    : "HCL Technologies",
    "ASIANPAINT.NS" : "Asian Paints",
    "AXISBANK.NS"   : "Axis Bank",
    "MARUTI.NS"     : "Maruti Suzuki",
    "SUNPHARMA.NS"  : "Sun Pharmaceutical",
    "TITAN.NS"      : "Titan Company",
    "ULTRACEMCO.NS" : "UltraTech Cement",
    "WIPRO.NS"      : "Wipro",
    "NESTLEIND.NS"  : "Nestle India",
    "POWERGRID.NS"  : "Power Grid Corporation",
    "NTPC.NS"       : "NTPC Limited",
    "TECHM.NS"      : "Tech Mahindra",
    "INDUSINDBK.NS" : "IndusInd Bank",
    "M&M.NS"        : "Mahindra Mahindra",
    "ONGC.NS"       : "ONGC Oil Natural Gas",
    "TATAMOTORS.NS" : "Tata Motors",
    "BAJAJFINSV.NS" : "Bajaj Finserv",
    "ADANIPORTS.NS" : "Adani Ports",
    "COALINDIA.NS"  : "Coal India",
    "JSWSTEEL.NS"   : "JSW Steel",
    "TATASTEEL.NS"  : "Tata Steel",
    "HINDALCO.NS"   : "Hindalco Industries",
    "DRREDDY.NS"    : "Dr Reddys Laboratories",
    "CIPLA.NS"      : "Cipla",
    "DIVISLAB.NS"   : "Divis Laboratories",
    "APOLLOHOSP.NS" : "Apollo Hospitals",
    "EICHERMOT.NS"  : "Eicher Motors Royal Enfield",
    "BPCL.NS"       : "Bharat Petroleum BPCL",
    "GRASIM.NS"     : "Grasim Industries",
    "BRITANNIA.NS"  : "Britannia Industries",
    "SBILIFE.NS"    : "SBI Life Insurance",
    "HDFCLIFE.NS"   : "HDFC Life Insurance",
    "BAJAJ-AUTO.NS" : "Bajaj Auto",
    "SHREECEM.NS"   : "Shree Cement",
    "HEROMOTOCO.NS" : "Hero MotoCorp",
    "UPL.NS"        : "UPL Limited",
    "TATACONSUM.NS" : "Tata Consumer Products",
    "ADANIENT.NS"   : "Adani Enterprises",
    "VEDL.NS"       : "Vedanta Limited",
    "CHOLAFIN.NS"   : "Cholamandalam Finance",
    "DLF.NS"        : "DLF Limited",
    "SIEMENS.NS"    : "Siemens India",
    "ABB.NS"        : "ABB India",
    "PIDILITIND.NS" : "Pidilite Industries",
    "HAVELLS.NS"    : "Havells India",
    "MUTHOOTFIN.NS" : "Muthoot Finance",
    "BANKBARODA.NS" : "Bank of Baroda",
    "PNB.NS"        : "Punjab National Bank PNB",
    "CANBK.NS"      : "Canara Bank",
    "UNIONBANK.NS"  : "Union Bank of India",
    "IDFCFIRSTB.NS" : "IDFC First Bank",
    "FEDERALBNK.NS" : "Federal Bank",
    "RBLBANK.NS"    : "RBL Bank",
    "BANDHANBNK.NS" : "Bandhan Bank",
    "BERGEPAINT.NS" : "Berger Paints India",
    "GODREJCP.NS"   : "Godrej Consumer Products",
    "COLPAL.NS"     : "Colgate Palmolive India",
    "DABUR.NS"      : "Dabur India",
    "EMAMILTD.NS"   : "Emami Limited",
    "MARICO.NS"     : "Marico",
    "TRENT.NS"      : "Trent Westside Tata",
    "NYKAA.NS"      : "Nykaa FSN E-Commerce",
    "ZOMATO.NS"     : "Zomato",
    "DMART.NS"      : "DMart Avenue Supermarts",
    "IRCTC.NS"      : "IRCTC Indian Railway",
    "POLYCAB.NS"    : "Polycab India",
    "AMBUJACEM.NS"  : "Ambuja Cements",
    "ACC.NS"        : "ACC Limited cement",
    "INDIGO.NS"     : "IndiGo airlines InterGlobe",
    "ADANIGREEN.NS" : "Adani Green Energy",
    "TATAPOWER.NS"  : "Tata Power",
    "TORNTPOWER.NS" : "Torrent Power",
    "NHPC.NS"       : "NHPC hydro power",
    "SJVN.NS"       : "SJVN Limited",
    "RECLTD.NS"     : "REC Limited power finance",
    "PFC.NS"        : "Power Finance Corporation PFC",
    "IRFC.NS"       : "IRFC Indian Railway Finance",
    "SAIL.NS"       : "SAIL Steel Authority India",
    "NMDC.NS"       : "NMDC iron ore",
    "NATIONALUM.NS" : "NALCO National Aluminium",
    "HINDCOPPER.NS" : "Hindustan Copper",
    "MPHASIS.NS"    : "Mphasis IT",
    "LTIM.NS"       : "LTIMindtree",
    "PERSISTENT.NS" : "Persistent Systems",
    "COFORGE.NS"    : "Coforge IT",
    "OFSS.NS"       : "Oracle Financial Services OFSS",
    "KPITTECH.NS"   : "KPIT Technologies",
    "GMRINFRA.NS"   : "GMR Airports Infrastructure",
    "CUMMINSIND.NS" : "Cummins India",
}

NIFTY_100_TICKERS = list(COMPANY_NAMES.keys())

# ─────────────────────────────────────────────────────────────────────
# 3. LOGGING
# ─────────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE, level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ─────────────────────────────────────────────────────────────────────
# 4. FINBERT LOADER
# ─────────────────────────────────────────────────────────────────────
print("Loading FinBERT model ...")
finbert = hf_pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    top_k=None,
    truncation=True,
    max_length=512,
)
print("FinBERT ready.\n")

# ─────────────────────────────────────────────────────────────────────
# 5. NEWS FETCHERS
# ─────────────────────────────────────────────────────────────────────

def fetch_google_news_rss(ticker: str) -> list[dict]:
    """
    Fetch articles from Google News RSS — Indian edition.
    URL targets en-IN locale so Indian sources are prioritised.

    Two queries are tried per stock:
        1. Company name + "NSE" → most specific
        2. Company name alone  → broader fallback
    """
    company = COMPANY_NAMES.get(ticker, ticker.replace(".NS", ""))

    # Try two progressively broader queries
    queries = [
        f"{company} NSE stock shares",
        f"{company} India",
    ]

    articles = []
    seen_titles = set()

    for query in queries:
        if len(articles) >= MAX_ARTICLES:
            break

        url = (
            "https://news.google.com/rss/search?"
            f"q={quote_plus(query)}"
            "&hl=en-IN"   # Indian English
            "&gl=IN"      # India geo
            "&ceid=IN:en" # India, English
        )

        try:
            feed = feedparser.parse(url)

            for entry in feed.entries:
                if len(articles) >= MAX_ARTICLES:
                    break

                title = entry.get("title", "").strip()
                if not title or title in seen_titles:
                    continue

                # Only keep articles from last 24 hours
                published = entry.get("published_parsed")
                if published:
                    pub_dt = datetime(*published[:6])
                    if (TODAY - pub_dt).total_seconds() > 86400:
                        continue

                summary = entry.get("summary", "") or ""
                # Google News RSS wraps summaries in HTML — strip tags
                summary = summary.replace("<b>", "").replace("</b>", "")

                seen_titles.add(title)
                articles.append({
                    "title"      : title,
                    "description": summary[:300],
                    "source"     : entry.get("source", {}).get("title", "Google News"),
                    "publishedAt": entry.get("published", ""),
                })

        except Exception as e:
            logging.warning("Google RSS error for %s: %s", ticker, e)
            continue

    return articles


def fetch_newsapi(ticker: str) -> list[dict]:
    """
    Optional fallback: NewsAPI /v2/everything.
    Only called if NEWSAPI_KEY is set AND Google News found fewer than 3 articles.

    Fixes from original version:
        - Simpler query without complex boolean operators
        - Uses 'language=en' without quote-enclosed phrases
        - Shorter time window to avoid free-tier limitations
    """
    if not NEWSAPI_KEY:
        return []

    company = COMPANY_NAMES.get(ticker, ticker.replace(".NS", ""))
    # Simplified query — no quotes, no AND — NewsAPI free tier handles simple queries better
    query = f"{company} stock India"

    params = {
        "q"       : query,
        "from"    : YESTERDAY.strftime("%Y-%m-%d"),
        "language": "en",
        "sortBy"  : "publishedAt",
        "pageSize": 5,
        "apiKey"  : NEWSAPI_KEY,
    }

    try:
        resp = requests.get(
            "https://newsapi.org/v2/everything",
            params=params,
            timeout=10,
        )
        data = resp.json()
        if data.get("status") != "ok":
            return []

        return [
            {
                "title"      : a.get("title", "") or "",
                "description": a.get("description", "") or "",
                "source"     : a.get("source", {}).get("name", "NewsAPI"),
                "publishedAt": a.get("publishedAt", ""),
            }
            for a in data.get("articles", [])
            if a.get("title")
        ]
    except Exception:
        return []


def fetch_news(ticker: str) -> list[dict]:
    """
    Master fetcher — tries Google News RSS first, then NewsAPI fallback.
    """
    articles = fetch_google_news_rss(ticker)

    # If Google News found fewer than 3, top-up with NewsAPI
    if len(articles) < 3 and NEWSAPI_KEY:
        newsapi_articles = fetch_newsapi(ticker)
        # Deduplicate by title
        existing_titles = {a["title"] for a in articles}
        for a in newsapi_articles:
            if a["title"] not in existing_titles:
                articles.append(a)
                existing_titles.add(a["title"])

    return articles[:MAX_ARTICLES]


# ─────────────────────────────────────────────────────────────────────
# 6. SENTIMENT SCORER
# ─────────────────────────────────────────────────────────────────────
LABEL_MAP = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}


def score_text(text: str) -> tuple[float, float]:
    """
    Run text through FinBERT.
    Returns (sentiment_value, confidence).
      +1.0 = positive, 0.0 = neutral, -1.0 = negative
    """
    text = text.strip()
    if not text:
        return 0.0, 0.0

    try:
        results = finbert(text[:512])[0]
        best    = max(results, key=lambda x: x["score"])
        return LABEL_MAP.get(best["label"].lower(), 0.0), best["score"]
    except Exception:
        return 0.0, 0.0


def aggregate_sentiment(articles: list[dict]) -> dict:
    """
    Score all articles and compute confidence-weighted aggregate score.

    Each article weight = confidence score (0–1).
    Final = sum(value × confidence) / sum(confidence)
    Range: -1.0 (very bearish) to +1.0 (very bullish)
    """
    if not articles:
        return {
            "sentiment_score" : 0.0,
            "num_articles"    : 0,
            "positive_count"  : 0,
            "negative_count"  : 0,
            "neutral_count"   : 0,
            "avg_confidence"  : 0.0,
            "headlines_sample": "",
        }

    weighted_scores, weights   = [], []
    label_counts               = {"positive": 0, "neutral": 0, "negative": 0}
    confidences, headlines     = [], []

    for article in articles:
        text  = f"{article['title']}. {article['description']}".strip(". ")
        score, conf = score_text(text)

        weighted_scores.append(score * conf)
        weights.append(conf)
        confidences.append(conf)
        headlines.append(article["title"])

        if score > 0:
            label_counts["positive"] += 1
        elif score < 0:
            label_counts["negative"] += 1
        else:
            label_counts["neutral"] += 1

    total_w     = sum(weights)
    final_score = sum(weighted_scores) / total_w if total_w > 0 else 0.0

    return {
        "sentiment_score" : round(final_score, 4),
        "num_articles"    : len(articles),
        "positive_count"  : label_counts["positive"],
        "negative_count"  : label_counts["negative"],
        "neutral_count"   : label_counts["neutral"],
        "avg_confidence"  : round(sum(confidences) / len(confidences), 4),
        "headlines_sample": " | ".join(headlines[:3]),
    }


# ─────────────────────────────────────────────────────────────────────
# 7. PER-STOCK PIPELINE
# ─────────────────────────────────────────────────────────────────────

def run_stock_sentiment(ticker: str, index: int, total: int):
    print(f"[{index:>3}/{total}]  {ticker:<20}", end="  ", flush=True)

    try:
        articles = fetch_news(ticker)

        if not articles:
            print("No articles (24h) — score=0.000 (neutral)")
            return {
                "Ticker"          : ticker,
                "Date"            : TODAY.strftime("%Y-%m-%d"),
                "sentiment_score" : 0.0,
                "num_articles"    : 0,
                "positive_count"  : 0,
                "negative_count"  : 0,
                "neutral_count"   : 0,
                "avg_confidence"  : 0.0,
                "headlines_sample": "No articles found",
            }

        result = aggregate_sentiment(articles)
        score  = result["sentiment_score"]
        marker = "↑" if score > 0.1 else ("↓" if score < -0.1 else "→")

        print(
            f"n={result['num_articles']:>2}  "
            f"score={score:+.3f} {marker}  "
            f"[+{result['positive_count']} "
            f"-{result['negative_count']} "
            f"~{result['neutral_count']}]  "
            f"conf={result['avg_confidence']:.2f}"
        )

        return {"Ticker": ticker, "Date": TODAY.strftime("%Y-%m-%d"), **result}

    except Exception as exc:
        logging.error("%s | %s", ticker, exc)
        print(f"Error -> {exc}")
        return None


# ─────────────────────────────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────────────────────────────

CSV_COLUMNS = [
    "Ticker", "Date",
    "sentiment_score",
    "num_articles",
    "positive_count", "negative_count", "neutral_count",
    "avg_confidence",
    "headlines_sample",
]


def main():
    div = "=" * 66
    source_info = "Google News RSS (India)"
    if NEWSAPI_KEY:
        source_info += " + NewsAPI (fallback)"

    print(f"\n{div}")
    print("  Sentiment Pipeline  |  Nifty 100  |  NSE India  [FIXED]")
    print(f"  Run date  : {TODAY.strftime('%Y-%m-%d')}")
    print(f"  News from : last 24 hours")
    print(f"  Sources   : {source_info}")
    print(f"  Model     : FinBERT (ProsusAI/finbert)")
    print(f"  Stocks    : {len(NIFTY_100_TICKERS)}")
    print(f"  Output    : {OUTPUT_CSV}")
    print(f"{div}\n")

    success_count = 0
    skipped_count = 0

    for idx, ticker in enumerate(NIFTY_100_TICKERS, start=1):
        result = run_stock_sentiment(ticker, idx, len(NIFTY_100_TICKERS))

        if result:
            row_df = pd.DataFrame([result], columns=CSV_COLUMNS)
            if os.path.exists(OUTPUT_CSV):
                row_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
            else:
                row_df.to_csv(OUTPUT_CSV, index=False)
            success_count += 1
        else:
            skipped_count += 1

        time.sleep(SLEEP_BETWEEN)

    # ── Summary ───────────────────────────────────────────────────
    print(f"\n{div}")
    print(f"  Complete  |  {success_count} stocks saved  |  {skipped_count} skipped")
    print(f"  Saved to  : {os.path.abspath(OUTPUT_CSV)}")

    if success_count > 0 and os.path.exists(OUTPUT_CSV):
        df       = pd.read_csv(OUTPUT_CSV)
        today_df = df[df["Date"] == TODAY.strftime("%Y-%m-%d")]

        if not today_df.empty:
            has_news    = today_df[today_df["num_articles"] > 0]
            avg_score   = today_df["sentiment_score"].mean()
            coverage    = len(has_news) / len(today_df) * 100

            print(f"\n  Coverage  : {len(has_news)}/{len(today_df)} stocks had articles ({coverage:.0f}%)")
            print(f"  Avg score : {avg_score:+.3f}  ", end="")
            print("(Bullish)" if avg_score > 0.1 else ("(Bearish)" if avg_score < -0.1 else "(Neutral)"))

            if not has_news.empty:
                print(f"\n  Most bullish  (top 5):")
                top5 = has_news.nlargest(5, "sentiment_score")[
                    ["Ticker", "sentiment_score", "num_articles", "positive_count"]
                ]
                print(top5.to_string(index=False))

                print(f"\n  Most bearish  (bottom 5):")
                bot5 = has_news.nsmallest(5, "sentiment_score")[
                    ["Ticker", "sentiment_score", "num_articles", "negative_count"]
                ]
                print(bot5.to_string(index=False))

    print(f"\n{div}\n")


if __name__ == "__main__":
    main()


# FINAL MERGE V5

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║     Final Merge Script  —  v5 Predictions + Sentiment Scores        ║
║                                                                      ║
║  Reads:                                                              ║
║    nifty100_predictions_v5.csv   (from lstm_nifty100_pipeline_v5.py)║
║    sentiment_scores.csv          (from sentiment_pipeline.py)        ║
║                                                                      ║
║  Steps:                                                              ║
║    1. Deduplicate v5 — keep best dir_acc_pct per ticker             ║
║    2. Merge sentiment by ticker + date                               ║
║    3. Apply minimum article threshold  (< 3 articles → score = 0)   ║
║    4. Convert sentiment score → probability shift                    ║
║    5. Blend: final_pred = 0.80 × model + 0.20 × sentiment_adj      ║
║    6. Recompute direction + conviction signal                        ║
║    7. Save final_predictions.csv                                     ║
║                                                                      ║
║  Conviction signals:                                                 ║
║    BOOST  — sentiment strongly amplifies model  (|score| > 0.6)     ║
║    HIGH   — sentiment agrees with model direction                    ║
║    MEDIUM — sentiment neutral or no news                             ║
║    LOW    — sentiment conflicts with model                           ║
╚══════════════════════════════════════════════════════════════════════╝

Run order:
    1. python lstm_nifty100_pipeline_v5.py  →  nifty100_predictions_v5.csv
    2. python sentiment_pipeline.py          →  sentiment_scores.csv
    3. python merge_final_predictions.py     →  final_predictions.csv
"""

import os
import numpy as np
import pandas as pd
from datetime import datetime

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────
V5_CSV         = "nifty100_predictions_v5.csv"
SENTIMENT_CSV  = "sentiment_scores.csv"
OUTPUT_CSV     = "final_predictions.csv"

MODEL_WEIGHT   = 0.80
SENT_WEIGHT    = 0.20
MIN_ARTICLES   = 3       # stocks with fewer articles → sentiment = 0.0
MAX_SHIFT      = 0.52    # max probability shift from sentiment
TODAY          = datetime.today().strftime("%Y-%m-%d")

# ─────────────────────────────────────────────────────────────────────
# 2. SENTIMENT → PREDICTION ADJUSTMENT
#
#   The model outputs a % return (e.g. +2.4%).
#   Sentiment is a float -1.0 to +1.0.
#
#   Adjustment formula:
#     sentiment_pct  = sentiment_score × |pred_1m_pct|
#     final_pred     = 0.80 × pred_1m_pct + 0.20 × sentiment_pct
#
#   Example — model and sentiment agree:
#     pred = +3.0%,  sentiment = +0.7
#     sent_pct  = +0.7 × 3.0 = +2.1%
#     final     = 0.8 × 3.0 + 0.2 × 2.1 = 2.4 + 0.42 = +2.82%
#
#   Example — model and sentiment conflict:
#     pred = +3.0%,  sentiment = -0.6
#     sent_pct  = -0.6 × 3.0 = -1.8%
#     final     = 0.8 × 3.0 + 0.2 × (-1.8) = 2.4 - 0.36 = +2.04%
#     → Sentiment partially offsets the bullish prediction
#
#   Example — no news:
#     pred = +3.0%,  sentiment = 0.0 (< 3 articles)
#     final = 0.8 × 3.0 + 0.2 × 0.0 = +2.40%  (unchanged)
# ─────────────────────────────────────────────────────────────────────

def blend_prediction(pred_pct: float,
                     sentiment_score: float) -> float:
    """Apply sentiment blend to % return prediction."""
    sent_pct = sentiment_score * abs(pred_pct)
    return round(MODEL_WEIGHT * pred_pct + SENT_WEIGHT * sent_pct, 4)


def conviction_signal(model_pred: float,
                       final_pred: float,
                       sentiment_score: float) -> str:
    """
    BOOST  — strong sentiment amplification same direction (|score|>0.6)
    HIGH   — sentiment agrees with model direction
    MEDIUM — neutral sentiment or no news (|score|<0.1)
    LOW    — sentiment conflicts with model
    """
    if abs(sentiment_score) < 0.1:
        return "MEDIUM"

    model_dir = "UP"  if model_pred >= 0 else "DOWN"
    sent_dir  = "UP"  if sentiment_score > 0 else "DOWN"

    if model_dir == sent_dir:
        return "BOOST" if abs(sentiment_score) > 0.6 else "HIGH"
    return "LOW"


# ─────────────────────────────────────────────────────────────────────
# 3. CONFIDENCE RECALCULATION
#    After blending, confidence is recomputed from the final prediction
#    magnitude relative to the original model prediction magnitude.
# ─────────────────────────────────────────────────────────────────────

def recalc_confidence(final_pred: float,
                       original_conf: float,
                       original_pred: float) -> float:
    """
    Scale confidence proportionally to how much sentiment moved the prediction.
    If sentiment reinforced the model → confidence goes up slightly.
    If sentiment conflicted  → confidence goes down slightly.
    """
    if abs(original_pred) < 0.001:
        return original_conf
    ratio = abs(final_pred) / abs(original_pred)
    new_conf = original_conf * ratio
    return round(float(np.clip(new_conf, 0.0, 100.0)), 1)


# ─────────────────────────────────────────────────────────────────────
# 4. MAIN
# ─────────────────────────────────────────────────────────────────────

def main():
    div = "=" * 66
    print(f"\n{div}")
    print("  Final Merge  —  v5 Predictions + Sentiment  |  NSE India")
    print(f"  Run date  : {TODAY}")
    print(f"  Weights   : {int(MODEL_WEIGHT*100)}% model  +  {int(SENT_WEIGHT*100)}% sentiment")
    print(f"  Min arts  : {MIN_ARTICLES} articles required for sentiment to apply")
    print(f"{div}\n")

    # ── Load v5 predictions ───────────────────────────────────────
    if not os.path.exists(V5_CSV):
        print(f"ERROR: {V5_CSV} not found.")
        print("Run lstm_nifty100_pipeline_v5.py first.")
        return

    v5_raw = pd.read_csv(V5_CSV)
    print(f"v5 raw rows loaded : {len(v5_raw)}")

    # ── Deduplicate — keep best dir_acc_pct per ticker ────────────
    # If a stock ran twice, keep the run with higher directional accuracy
    v5_df = (
        v5_raw
        .sort_values("dir_acc_pct", ascending=False)
        .drop_duplicates(subset="Ticker", keep="first")
        .reset_index(drop=True)
    )
    dupes_removed = len(v5_raw) - len(v5_df)
    print(f"After dedup        : {len(v5_df)} unique stocks  "
          f"({dupes_removed} duplicates removed — kept best dir_acc)")

    # ── Load sentiment ────────────────────────────────────────────
    sent_map = {}   # {ticker: sentiment_score}
    art_map  = {}   # {ticker: num_articles}

    if os.path.exists(SENTIMENT_CSV):
        sent_raw   = pd.read_csv(SENTIMENT_CSV)
        sent_today = sent_raw[sent_raw["Date"] == TODAY]
        if sent_today.empty:
            print(f"No sentiment for {TODAY} — using latest available.")
            sent_today = sent_raw.copy()

        for _, r in sent_today.iterrows():
            sent_map[r["Ticker"]] = float(r.get("sentiment_score", 0.0))
            art_map[r["Ticker"]]  = int(r.get("num_articles", 0))

        covered = sum(1 for v in art_map.values() if v >= MIN_ARTICLES)
        print(f"Sentiment loaded   : {len(sent_map)} stocks  "
              f"({covered} meet {MIN_ARTICLES}-article threshold)")
    else:
        print(f"WARNING: {SENTIMENT_CSV} not found — "
              f"sentiment = 0.0 for all stocks.")

    # ── Merge and blend ───────────────────────────────────────────
    print(f"\n{'─'*66}")
    print(f"  {'Ticker':<20} {'Score':>6}  "
          f"{'Model':>8} {'Final':>8}  "
          f"{'Dir':>5} {'Conf':>6}  Signal")
    print(f"{'─'*66}")

    output_rows = []

    for _, row in v5_df.iterrows():
        ticker     = row["Ticker"]
        model_pred = float(row["pred_1m_pct"])
        model_dir  = str(row["direction"])
        model_conf = float(row["confidence"])

        # Sentiment — zero if below article threshold
        raw_score  = sent_map.get(ticker, 0.0)
        n_arts     = art_map.get(ticker, 0)
        score      = raw_score if n_arts >= MIN_ARTICLES else 0.0

        # Blend
        final_pred  = blend_prediction(model_pred, score)
        final_dir   = "UP" if final_pred >= 0 else "DOWN"
        final_conf  = recalc_confidence(final_pred, model_conf, model_pred)
        signal      = conviction_signal(model_pred, final_pred, score)

        arrow = "↑" if final_dir == "UP" else "↓"
        print(
            f"  {ticker:<20} {score:>+6.3f}  "
            f"{model_pred:>+7.2f}% {final_pred:>+7.2f}%  "
            f"{arrow}{final_dir:<5} {final_conf:>5.1f}%  {signal}"
        )

        output_rows.append({
            # Identity
            "Ticker"           : ticker,
            "Date"             : TODAY,
            # Final blended output
            "final_pred_pct"   : final_pred,
            "final_direction"  : final_dir,
            "final_confidence" : final_conf,
            "signal"           : signal,
            # Model component
            "model_pred_pct"   : model_pred,
            "model_direction"  : model_dir,
            "model_confidence" : model_conf,
            "lstm_pred_pct"    : float(row.get("lstm_1m_pct", 0.0)),
            "xgb_pred_pct"     : float(row.get("xgb_1m_pct", 0.0)),
            # Sentiment component
            "sentiment_score"  : score,
            "sentiment_raw"    : raw_score,
            "num_articles"     : n_arts,
            "sent_contribution": round(score * abs(model_pred), 4),
            # Accuracy metrics (from walk-forward)
            "dir_acc_pct"      : float(row.get("dir_acc_pct", 0.0)),
            "r2"               : float(row.get("r2", 0.0)),
            "mae"              : float(row.get("mae", 0.0)),
            "rmse"             : float(row.get("rmse", 0.0)),
        })

    # ── Save ──────────────────────────────────────────────────────
    if not output_rows:
        print("\nNo output rows generated.")
        return

    out_df = pd.DataFrame(output_rows)

    if os.path.exists(OUTPUT_CSV):
        existing = pd.read_csv(OUTPUT_CSV)
        out_df   = pd.concat([existing, out_df], ignore_index=True)
        print(f"\n  Appended to existing {OUTPUT_CSV}")
    else:
        print(f"\n  Created {OUTPUT_CSV}")

    out_df.to_csv(OUTPUT_CSV, index=False)

    # ── Summary ───────────────────────────────────────────────────
    today_out = out_df[out_df["Date"] == TODAY]

    up     = (today_out["final_direction"] == "UP").sum()
    dn     = (today_out["final_direction"] == "DOWN").sum()
    boost  = (today_out["signal"] == "BOOST").sum()
    high   = (today_out["signal"] == "HIGH").sum()
    medium = (today_out["signal"] == "MEDIUM").sum()
    low    = (today_out["signal"] == "LOW").sum()

    # High conviction = BOOST or HIGH + dir_acc > 60 + r2 > 0
    hc = today_out[
        (today_out["signal"].isin(["BOOST", "HIGH"])) &
        (today_out["dir_acc_pct"] > 60) &
        (today_out["r2"] > 0)
    ]

    print(f"\n{div}")
    print(f"  Merge complete  |  {len(output_rows)} stocks")
    print(f"  Saved to        : {os.path.abspath(OUTPUT_CSV)}")

    print(f"\n  Final direction split:")
    print(f"    ↑ UP   : {up} stocks")
    print(f"    ↓ DOWN : {dn} stocks")

    print(f"\n  Conviction signals:")
    print(f"    BOOST  (sentiment strongly amplifies) : {boost}")
    print(f"    HIGH   (model + sentiment agree)      : {high}")
    print(f"    MEDIUM (neutral / no news)             : {medium}")
    print(f"    LOW    (conflicting signals)           : {low}")

    print(f"\n  Model accuracy summary:")
    print(f"    Mean dir. accuracy  : {today_out['dir_acc_pct'].mean():.1f}%")
    print(f"    DirAcc > 60%        : {(today_out['dir_acc_pct']>60).sum()} stocks")
    print(f"    DirAcc > 70%        : {(today_out['dir_acc_pct']>70).sum()} stocks")
    print(f"    R² positive         : {(today_out['r2']>0).sum()} stocks")
    print(f"    Mean R²             : {today_out['r2'].mean():.3f}")

    print(f"\n  High conviction picks  "
          f"(BOOST/HIGH + dir_acc>60 + R²>0)  →  {len(hc)} stocks")
    if not hc.empty:
        hc_sorted = hc.sort_values(
            ["signal", "dir_acc_pct"], ascending=[True, False]
        )[["Ticker", "final_pred_pct", "final_direction",
           "final_confidence", "signal",
           "sentiment_score", "dir_acc_pct", "r2"]]
        print(hc_sorted.to_string(index=False))

    print(f"\n  Output CSV columns:")
    print(f"    final_pred_pct   — blended % return  (model + sentiment)")
    print(f"    final_direction  — UP / DOWN")
    print(f"    final_confidence — 0–100% volatility-normalised conviction")
    print(f"    signal           — BOOST / HIGH / MEDIUM / LOW")
    print(f"    model_pred_pct   — model-only prediction (before sentiment)")
    print(f"    sentiment_score  — FinBERT score  (-1 to +1)")
    print(f"    dir_acc_pct      — walk-forward directional accuracy")
    print(f"    r2               — walk-forward R² score")
    print(f"{div}\n")


if __name__ == "__main__":
    main()

# Final Merge with prediction.json

In [ ]:
import os
import numpy as np
import pandas as pd
from datetime import datetime
import json

# ─────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────
V5_CSV         = "nifty100_predictions_v5.csv"
SENTIMENT_CSV  = "sentiment_scores.csv"
OUTPUT_CSV     = "final_predictions.csv"
OUTPUT_JSON    = "predictions.json"  # <-- ADDED JSON OUTPUT

MODEL_WEIGHT   = 0.80
SENT_WEIGHT    = 0.20
MIN_ARTICLES   = 3       # stocks with fewer articles → sentiment = 0.0
MAX_SHIFT      = 0.52    # max probability shift from sentiment
TODAY          = datetime.today().strftime("%Y-%m-%d")

# ─────────────────────────────────────────────────────────────────────
# 2. SENTIMENT → PREDICTION ADJUSTMENT
# ─────────────────────────────────────────────────────────────────────
def blend_prediction(pred_pct: float, sentiment_score: float) -> float:
    """Apply sentiment blend to % return prediction."""
    sent_pct = sentiment_score * abs(pred_pct)
    return round(MODEL_WEIGHT * pred_pct + SENT_WEIGHT * sent_pct, 4)

def conviction_signal(model_pred: float, final_pred: float, sentiment_score: float) -> str:
    """Determine conviction signal based on model and sentiment agreement."""
    if abs(sentiment_score) < 0.1:
        return "MEDIUM"

    model_dir = "UP"  if model_pred >= 0 else "DOWN"
    sent_dir  = "UP"  if sentiment_score > 0 else "DOWN"

    if model_dir == sent_dir:
        return "BOOST" if abs(sentiment_score) > 0.6 else "HIGH"
    return "LOW"

# ─────────────────────────────────────────────────────────────────────
# 3. CONFIDENCE RECALCULATION
# ─────────────────────────────────────────────────────────────────────
def recalc_confidence(final_pred: float, original_conf: float, original_pred: float) -> float:
    """Scale confidence proportionally to how much sentiment moved the prediction."""
    if abs(original_pred) < 0.001:
        return original_conf
    ratio = abs(final_pred) / abs(original_pred)
    new_conf = original_conf * ratio
    return round(float(np.clip(new_conf, 0.0, 100.0)), 1)

# ─────────────────────────────────────────────────────────────────────
# 4. MAIN
# ─────────────────────────────────────────────────────────────────────
def main():
    div = "=" * 66
    print(f"\n{div}")
    print("  Final Merge  —  v5 Predictions + Sentiment  |  NSE India")
    print(f"  Run date  : {TODAY}")
    print(f"  Weights   : {int(MODEL_WEIGHT*100)}% model  +  {int(SENT_WEIGHT*100)}% sentiment")
    print(f"  Min arts  : {MIN_ARTICLES} articles required for sentiment")
    print(f"{div}\n")

    if not os.path.exists(V5_CSV):
        print(f"ERROR: {V5_CSV} not found.")
        return

    v5_raw = pd.read_csv(V5_CSV)
    v5_df = (
        v5_raw
        .sort_values("dir_acc_pct", ascending=False)
        .drop_duplicates(subset="Ticker", keep="first")
        .reset_index(drop=True)
    )

    sent_map = {}
    art_map  = {}

    if os.path.exists(SENTIMENT_CSV):
        sent_raw   = pd.read_csv(SENTIMENT_CSV)
        sent_today = sent_raw[sent_raw["Date"] == TODAY]
        if sent_today.empty:
            sent_today = sent_raw.copy()

        for _, r in sent_today.iterrows():
            sent_map[r["Ticker"]] = float(r.get("sentiment_score", 0.0))
            art_map[r["Ticker"]]  = int(r.get("num_articles", 0))

    output_rows = []

    for _, row in v5_df.iterrows():
        ticker     = row["Ticker"]
        model_pred = float(row["pred_1m_pct"])
        model_dir  = str(row["direction"])
        model_conf = float(row["confidence"])

        raw_score  = sent_map.get(ticker, 0.0)
        n_arts     = art_map.get(ticker, 0)
        score      = raw_score if n_arts >= MIN_ARTICLES else 0.0

        final_pred  = blend_prediction(model_pred, score)
        final_dir   = "UP" if final_pred >= 0 else "DOWN"
        final_conf  = recalc_confidence(final_pred, model_conf, model_pred)
        signal      = conviction_signal(model_pred, final_pred, score)

        output_rows.append({
            "Ticker"           : ticker,
            "Date"             : TODAY,
            "final_pred_pct"   : final_pred,
            "final_direction"  : final_dir,
            "final_confidence" : final_conf,
            "signal"           : signal,
            "model_pred_pct"   : model_pred,
            "model_direction"  : model_dir,
            "model_confidence" : model_conf,
            "lstm_pred_pct"    : float(row.get("lstm_1m_pct", 0.0)),
            "xgb_pred_pct"     : float(row.get("xgb_1m_pct", 0.0)),
            "sentiment_score"  : score,
            "sentiment_raw"    : raw_score,
            "num_articles"     : n_arts,
            "sent_contribution": round(score * abs(model_pred), 4),
            "dir_acc_pct"      : float(row.get("dir_acc_pct", 0.0)),
            "r2"               : float(row.get("r2", 0.0)),
            "mae"              : float(row.get("mae", 0.0)),
            "rmse"             : float(row.get("rmse", 0.0)),
        })

    if not output_rows:
        return

    out_df = pd.DataFrame(output_rows)

    # 1. Save to CSV (Historical record)
    if os.path.exists(OUTPUT_CSV):
        existing = pd.read_csv(OUTPUT_CSV)
        out_df   = pd.concat([existing, out_df], ignore_index=True)
    else:
        out_df.to_csv(OUTPUT_CSV, index=False)

    out_df.to_csv(OUTPUT_CSV, index=False)

    # 2. Save to JSON (For the API endpoint - Only today's data)
    today_out = out_df[out_df["Date"] == TODAY]
    today_out.to_json(OUTPUT_JSON, orient="records", indent=4)
    print(f"\n  Saved API payload to : {os.path.abspath(OUTPUT_JSON)}")
    print(f"{div}\n")

if __name__ == "__main__":
    main()

# Automate GITHUB push

In [ ]:
import requests
import base64
import json

# Your GitHub setup
GITHUB_TOKEN = "your_github_personal_access_token"
REPO = "yourusername/nifty-api"
FILE_PATH = "predictions.json"

# Read the JSON file you just generated
with open("predictions.json", "rb") as f:
    content = f.read()
encoded_content = base64.b64encode(content).decode("utf-8")

# Get the SHA of the existing file (required by GitHub to update it)
url = f"https://api.github.com/repos/{REPO}/contents/{FILE_PATH}"
headers = {"Authorization": f"token {GITHUB_TOKEN}"}
response = requests.get(url, headers=headers)
sha = response.json().get("sha", "") if response.status_code == 200 else ""

# Push the new JSON file
payload = {
    "message": "Daily prediction update",
    "content": encoded_content,
    "sha": sha
}
requests.put(url, headers=headers, json=payload)
print("Successfully pushed new predictions to API!")

**Step 3: Fetch it on your Frontend**
Now, you have a completely free, infinitely scalable API. Whenever a user loads your website, your frontend JavaScript just makes a simple GET request to your GitHub Pages URL:

```javascript
async function getPredictions() {
  try {
    // This is your new Static API endpoint!
    const response = await fetch('https://yourusername.github.io/nifty-api/predictions.json');
    const data = await response.json();

    console.log("Loaded Nifty 100 Predictions:", data);

    // Example: Filter high conviction stocks
    const topPicks = data.filter(stock => stock.signal === "BOOST" || stock.signal === "HIGH");

    // Render to your dashboard here...

  } catch (error) {
    console.error('Error fetching API:', error);
  }
}

getPredictions();

Let me know if you want to go deeper into generating a Personal Access Token on GitHub, or if you want to start mocking up the frontend UI that will consume this data.